# Model Development

This Notebook contains all information pertaining to the development of the models and was developed by Marc C. Hennig (mhennig@hm.edu). Requires the preprocessed datasets from the first file.

# Environment

## Dependency installation

### A. PIP Dependencies

In [ ]:
!pip install ipdb keras_nlp keras-tuner tensorboard-plugin-profile==2.19.9 umap-learn
!pip freeze > requirements.txt

## Dependency Imports

In [ ]:
# Python dependencies
import os
import sys
import re
import math
import datetime
import random
import json
import time
import shutil
import warnings
from pathlib import Path
from typing import List, Tuple, Union, Optional, Literal, Callable, Dict, Any
from collections import namedtuple
import itertools
import functools
from enum import Enum

# Debugging
import ipdb
from tqdm.auto import tqdm

# Colab dependencies
from google.colab import files, drive

# Basic dependencies
import numpy as np
import pandas as pd
import scipy as sp
import statsmodels as sm

# Plotting dependencies
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns

# Machine learning depenencies
import sklearn as sl
import sklearn.metrics

import umap

# Neural network dependencies
%load_ext tensorboard

import tensorflow as tf
from tensorflow.data import Dataset

import keras
import keras_nlp
import keras_tuner

from keras import backend as K

## Variables & Global Settings

In [ ]:
# Assign a random seed for reproduceability
RANDOM_STATE = 1337

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
keras.utils.set_random_seed(RANDOM_STATE)

# Keras backend
os.environ["KERAS_BACKEND"] = "tensorflow"

# Show all Pandas columns
pd.set_option("display.max_columns", None)

# Set Matplotlib and Seaborn color scheme
plt.rcParams["image.cmap"] = "Blues"
sns.set_palette("Blues")

# Masking is handled manually
warnings.filterwarnings("ignore", message=r"(absl:)?Layer '\w+' \(of type \w+\) was passed an input with a mask attached to it\. However, this layer does not support masking and will therefore destroy the mask information\. Downstream layers will not see the mask\.")
warnings.filterwarnings("ignore", message=r"(absl:)?You are explicitly setting `\w+` while the `\w+` have built-in mask, so the built-in mask is ignored\.")

In [ ]:
# Google Drive folders
GDRIVE_INPUT_DIR = "/content/drive/My Drive/Colab Notebooks/Eventlogs/Preprocessed Event Logs"
GDRIVE_OUTPUT_DIR = "/content/drive/My Drive/Colab Notebooks/Results"

# Local Colab folders
UTIL_DIR = os.path.join(".", "Util")
DATA_DIR = os.path.join(".", "Data")
INPUT_DATA_DIR = os.path.join(DATA_DIR, "Input")
INPUT_DATA_BPIC2013_DIR = os.path.join(INPUT_DATA_DIR, "BPIC 2013")
INPUT_DATA_BPIC2014_DIR = os.path.join(INPUT_DATA_DIR, "BPIC 2014")
INTERIM_DATA_DIR = os.path.join(DATA_DIR, "Interim")
OUTPUT_DATA_DIR = os.path.join(DATA_DIR, "Output")
OUTPUT_LOG_DATA_DIR = os.path.join(OUTPUT_DATA_DIR, "Logs")

GRAPHIC_DIR = os.path.join(".", "Graphics")
MODEL_DIR = os.path.join(".", "Models")
MODEL_CHECKPOINT_DIR = os.path.join(MODEL_DIR, "Checkpoints")
MODEL_BACKUP_DIR = os.path.join(MODEL_DIR, "Backups")

Path(DATA_DIR).mkdir(exist_ok=True)
Path(INTERIM_DATA_DIR).mkdir(exist_ok=True)
Path(OUTPUT_DATA_DIR).mkdir(exist_ok=True)
Path(OUTPUT_LOG_DATA_DIR).mkdir(exist_ok=True)
Path(GRAPHIC_DIR).mkdir(exist_ok=True)
Path(MODEL_DIR).mkdir(exist_ok=True)
Path(MODEL_BACKUP_DIR).mkdir(exist_ok=True)
Path(MODEL_CHECKPOINT_DIR).mkdir(exist_ok=True)

In [ ]:
EVENTLOG_CASE = "case:concept:name"
EVENTLOG_ACTIVITY = "concept:name"
EVENTLOG_TIMESTAMP = "time:timestamp"
EVENTLOG_GROUP = "org:group"
EVENTLOG_RESOURCE = "org:resource"
EVENTLOG_ROLE = "org:role"
EVENTLOG_CASE_PREFIX = "case:"
EVENTLOG_LABEL_PREFIX = "label:"

EVENTLOG_LABEL_REM_TIME = f"{EVENTLOG_LABEL_PREFIX}time:timestamp:last"
EVENTLOG_LABEL_NEXT_ACT = f"{EVENTLOG_LABEL_PREFIX}concept:name:next"
EVENTLOG_LABEL_NEXT_TIME = f"{EVENTLOG_LABEL_PREFIX}time:timestamp:next"

TOKEN_PAD = "[PAD]"
TOKEN_PAD_NUM = -1.0
TOKEN_OOV = "[OOV]"
TOKEN_NA = "[NA]"
TOKEN_EOC = "[EOC]"

DEFAULT_LEARNING_RATE = 0.00003
DEFAULT_MIN_LEARNING_RATE = 1e-6

DEFAULT_WARMUP_EPOCHS = 10
DEFAULT_EPOCHS = 200
DEFAULT_BATCH_SIZE = 256

DEFAULT_REMAINING_TIME_OUTPUT = "remaining_time"
DEFAULT_NEXT_ACTIVITY_OUTPUT = "next_activity"
DEFAULT_NEXT_TIME_OUTPUT = "next_time"

## Data Import

### A: Import from Google Drive

In [ ]:
drive.mount("/content/drive")

!cp -r "$GDRIVE_INPUT_DIR" "$INPUT_DATA_DIR"

drive.flush_and_unmount()

### B: Upload from Local Machine

In [ ]:
#uploaded = files.upload()

#for filename in uploaded.keys():
#  target = os.path.join(INPUT_DATA_DIR, filename)
#  !mv "$filename" "$target"

#del uploaded

## Common Functions

In [ ]:
def df_write_files(df: pd.DataFrame, filename: str, index: bool = False) -> None:
  df.to_csv(f"{filename}.csv", index=index)
  df.to_pickle(f"{filename}.pkl.gz")
  try:
    df.reset_index().to_feather(f"{filename}.feather")
  except Exception as e:
    print(f"Skipping feather: {e}")
  try:
    df.to_parquet(f"{filename}.parquet", index=index)
  except Exception as e:
    print(f"Skipping parquet: {e}")

### Dataset Functions

In [ ]:
def ds_write_files(ds: tf.data.Dataset, filename: str, compression: Optional[str] = 'zip') -> None:
  ds.save(filename, compression='GZIP')
  shutil.make_archive(
      filename,
      format=compression,
      root_dir=filename
  )

def ds_find_static_attrs(ds: tf.data.Dataset) -> List[str]:
  input, _ = ds.element_spec

  stat_attrs = []
  for key, value in input.items():
    if 1 == value.shape.num_elements():
      stat_attrs.append(key)

  return stat_attrs

def ds_find_dynamic_attrs(ds: tf.data.Dataset) -> List[str]:
  input, _ = ds.element_spec

  dyn_attrs = []
  for key, value in input.items():
    if 1 < value.shape.num_elements():
      dyn_attrs.append(key)

  return dyn_attrs

def ds_find_max_seq_len(ds: tf.data.Dataset) -> int:
  input, _ = ds.element_spec

  dyn_attrs = ds_find_dynamic_attrs(ds)
  dyn_attr_lens = set()
  for dyn_attr in dyn_attrs:
    dyn_attr_lens.add(input[dyn_attr].shape[0])

  if len(dyn_attr_lens) > 1:
    raise ValueError(f"Different sequence lengths for different attributes {dyn_attr_lens}.")

  return dyn_attr_lens.pop()

def ds_calculate_class_weights(ds: tf.data.Dataset, y_key: str = DEFAULT_NEXT_ACTIVITY_OUTPUT) -> dict:
  y_train = np.concatenate([y[y_key] for x, y in ds], axis=0).flatten()

  class_weights = sl.utils.class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
  )
  return dict(enumerate(class_weights))

def ds_remove_attrs(ds: tf.data.Dataset, attrs: Union[List[str], str], keep: bool = False) -> tf.data.Dataset:
  if isinstance(attrs, str):
    attrs = [attrs]

  def ds_remove_attrs_map_helper(inputs, outputs):
    for attr in attrs:
      inputs.pop(attr)

    return inputs, outputs
  return ds.map(ds_remove_attrs_map_helper, num_parallel_calls=tf.data.AUTOTUNE)

def ds_rename_attr(ds: tf.data.Dataset, old_attr: str, new_attr: str, copy: bool = False) -> tf.data.Dataset:

  def ds_rename_attr_map_helper(inputs, outputs):
    inputs[new_attr] = inputs[old_attr]
    if not copy:
      inputs.pop(old_attr)

    return inputs, outputs

  return ds.map(ds_rename_attr_map_helper, num_parallel_calls=tf.data.AUTOTUNE)

def ds_desequentialize_dynamic_attrs(ds: tf.data.Dataset, dynamic_attrs: Optional[Union[List[str], str]] = None, strategy: Literal['last', 'first', 'mean', 'median'] = 'last', activity_attr: str = "activity", pad_token: str = TOKEN_PAD) -> tf.data.Dataset:
  if dynamic_attrs is None:
    dynamic_attrs = ds_find_dynamic_attrs(ds)
  elif isinstance(dynamic_attrs, str):
    dynamic_attrs = [dynamic_attrs]

  def ds_desequentialize_map_helper(inputs, outputs):
    # Find the current sequence length by looking for the padding token in the 'activity' attribute
    mask = tf.equal(inputs[activity_attr], pad_token)
    seq_len = tf.where(tf.reduce_any(mask), tf.argmax(mask, axis=-1), tf.cast(max_seq_len, tf.int64))

    for attr in dynamic_attrs:
      if 'last' == strategy:
        inputs[attr] = [inputs[attr][seq_len - 1]]
      elif 'first' == strategy:
        inputs[attr] = [inputs[attr][0]]
      elif 'mean' == strategy:
        inputs[attr] = [tf.reduce_mean(inputs[attr][:seq_len], axis=0)]
      elif 'median' == strategy:
        inputs[attr] = [tfp.stats.percentile(inputs[attr][:seq_len], 50)]
      else:
        raise ValueError(f"Unknown strategy {strategy}")

    return inputs, outputs

  return ds.map(ds_desequentialize_map_helper, num_parallel_calls=tf.data.AUTOTUNE)

def ds_remove_outputs(ds: tf.data.Dataset) -> tf.data.Dataset:
  return ds.map(lambda x, y: x, num_parallel_calls=tf.data.AUTOTUNE)

def ds_sequentialize_static_attrs(ds: tf.data.Dataset, static_attrs: Optional[Union[List[str], str]] = None, activity_attr: str = "activity", pad_token: str = TOKEN_PAD, pad_token_num = int(TOKEN_PAD_NUM)) -> tf.data.Dataset:
  if static_attrs is None:
    static_attrs = ds_find_static_attrs(ds)
  elif isinstance(static_attrs, str):
    static_attrs = [static_attrs]

  max_seq_len = ds_find_max_seq_len(ds)

  def ds_sequentialize_map_helper(inputs, outputs):
    # Find the current sequence length by looking for the padding token in the 'activity' attribute
    # Assumes 'activity' is a dynamic attribute holding the sequence
    mask = tf.equal(inputs[activity_attr], pad_token)
    seq_len = tf.where(tf.reduce_any(mask), tf.argmax(mask, axis=-1), tf.cast(max_seq_len, tf.int64))

    if tf.is_symbolic_tensor(seq_len):
      seq_len = max_seq_len

    for attr in static_attrs:
      # Repeat static attribute for the sequence length and pad to the max sequence length
      attr_val_repeated = tf.repeat(inputs[attr], repeats=seq_len, axis=0)
      paddings = [[0, max_seq_len - seq_len]]
      pad_const = pad_token_num if attr_val_repeated.dtype.is_numeric else pad_token
      attr_val_padded = tf.pad(attr_val_repeated, paddings, 'CONSTANT', constant_values=pad_const)
      inputs[attr] = attr_val_padded
      inputs[attr].set_shape((seq_len,))

    return inputs, outputs

  return ds.map(ds_sequentialize_map_helper, num_parallel_calls=tf.data.AUTOTUNE)

def ds_sparse_to_onehot(ds: tf.data.Dataset, target_attr: str, num_classes: Optional[int] = None) -> tf.data.Dataset:
  if num_classes is None:
    classes = ds_find_target_classes(ds, target_attr)
    num_classes = max(len(classes), classes.max() + 1)

  def ds_sparse_to_onehot_map_helper(inputs, outputs):
    labels = keras.ops.squeeze(outputs[target_attr])
    labels = keras.utils.to_categorical(labels, num_classes=num_classes)

    outputs[target_attr] = labels
    return inputs, outputs

  return ds.map(ds_sparse_to_onehot_map_helper, num_parallel_calls=tf.data.AUTOTUNE)

def ds_window_dynamic_attrs(ds: tf.data.Dataset, window: int, activity_attr: str = "activity", pad_token: str = TOKEN_PAD, pad_token_num = int(TOKEN_PAD_NUM)) -> tf.data.Dataset:
  max_seq_len = ds_find_max_seq_len(ds)
  dyn_attrs = ds_find_dynamic_attrs(ds)

  if max_seq_len <= window:
    return ds

  def ds_window_map_helper(inputs, outputs):
    # Find the current sequence length by looking for the padding token in the 'activity' attribute. Assumes 'activity' is a dynamic attribute holding the sequence and all dynamic attributes are of equal length
    mask = tf.equal(inputs[activity_attr], pad_token)
    seq_len = tf.where(tf.reduce_any(mask), tf.argmax(mask, axis=-1), tf.cast(max_seq_len, tf.int64))
    begin = tf.math.maximum(tf.zeros((), tf.int64), seq_len - window)
    end = begin + window

    for attr in dyn_attrs:
      inputs[attr] = inputs[attr][begin:end]
      shape = tuple([window] + inputs[attr].shape.as_list()[1:])
      inputs[attr].set_shape(shape)

    return inputs, outputs

  return ds.map(ds_window_map_helper, num_parallel_calls=tf.data.AUTOTUNE)

def ds_to_single_target(ds: tf.data.Dataset, target_attr: str) -> tf.data.Dataset:
  def single_target_map_helper(inputs, outputs):
    return inputs, outputs[target_attr]

  return ds.map(single_target_map_helper, num_parallel_calls=tf.data.AUTOTUNE)

def ds_find_target_classes(ds: tf.data.Dataset, target_attr: str) -> List[str]:
  return np.unique(np.concatenate([y[target_attr] for x, y in ds], axis=0).flatten())

def ds_sparse_target_to_onehot(ds: tf.data.Dataset, target_attr: str) -> tf.data.Dataset:
  num_classes = len(ds_find_target_classes(ds, target_attr))
  return ds.map(lambda inputs, outputs: (inputs, keras.utils.to_categorical(outputs[target_attr], num_classes=num_classes)), num_parallel_calls=tf.data.AUTOTUNE)

def ds_cache_and_batch(ds: tf.data.Dataset, batch_size: int = DEFAULT_BATCH_SIZE, cache_file: Optional[str] = None, shuffle: bool = False) -> tf.data.Dataset:
  ds = ds.batch(batch_size)

  if cache_file is not None:
    ds = ds.cache(filename=cache_file)
  else:
    ds = ds.cache()

  # Build the cache
  _ = list(ds.as_numpy_iterator())

  if shuffle:
    ds = ds.shuffle(len(ds), reshuffle_each_iteration=True)

  return ds.prefetch(tf.data.AUTOTUNE)

def ds_filter_length(ds: tf.data.Dataset, min: int = 0, max: int = float('inf'), activity_attr: str = "activity", pad_token: str = TOKEN_PAD) -> tf.data.Dataset:
  if min > max:
    raise ValueError(f"min {min} must be smaller than the max {max}")
  if max < 0:
    raise ValueError(f"max {max} must be a positive number")

  def ds_filter_length_helper(inputs, outputs):
    seq_len = keras.ops.argmax(keras.ops.equal(inputs[activity_attr], pad_token), axis=-1)
    gte = keras.ops.greater_equal(seq_len, min)
    lt = keras.ops.less(seq_len, max)
    return keras.ops.logical_and(gte, lt)

  ds = ds.filter(ds_filter_length_helper)

  ds_len = 0
  for _ in ds:
    ds_len += 1

  ds = ds.apply(tf.data.experimental.assert_cardinality(ds_len))

  return ds

def df_embedding_to_ds(df: pd.DataFrame, embeddings: np.ndarray, target_cols: str, case_col: Union[str, List[str]] = EVENTLOG_CASE) -> tf.data.Dataset:
  if isinstance(target_cols, str):
    target_cols = [target_cols]

  targets = {}
  for target_col in set(target_cols):
    target_name = target_col
    if target_name == EVENTLOG_LABEL_NEXT_ACT:
      target_name = DEFAULT_NEXT_ACTIVITY_OUTPUT
      target_dtype = tf.int16
    elif target_name == EVENTLOG_LABEL_NEXT_TIME:
      target_name = DEFAULT_NEXT_TIME_OUTPUT
      target_dtype = tf.float32
    elif target_name == EVENTLOG_LABEL_REM_TIME:
      target_name = DEFAULT_REMAINING_TIME_OUTPUT
      target_dtype = tf.float32

    target = np.expand_dims(df[target_col].astype('float64').to_numpy(), axis=-1)
    #df_emb = pd.DataFrame(embeddings).astype('float32').join(df.reset_index(drop=True))
    #embedding_cols = list(range(embeddings.shape[-1]))

    if target.shape[0] != embeddings.shape[0]:
      raise ValueError(f"Targets {targets.shape[0]} and embeddings {embeddings.shape[0]} do not have the same length")
    targets[target_name] = tf.constant(target, dtype=target_dtype)

  case_ids = df[case_col].to_numpy()
  case_changes = np.concatenate([[0], np.where(case_ids[:-1] != case_ids[1:])[0] + 1, [len(case_ids)]])

  # Build flat values and row splits
  flat_values = []
  row_splits = [0]

  for start, end in zip(case_changes[:-1], case_changes[1:]):
    case_embeddings = embeddings[start:end].astype('float32')
    for i in range(1, end - start + 1):
      flat_values.append(case_embeddings[:i])
      row_splits.append(row_splits[-1] + i)

  # Concatenate all embeddings into one flat array
  flat_values = np.concatenate(flat_values, axis=0)
  row_splits = np.array(row_splits, dtype=np.int64)

  # Create ragged tensor directly from flat values and row splits
  ragged_embeddings = tf.RaggedTensor.from_row_splits(
    values=tf.constant(flat_values, dtype=tf.float32),
    row_splits=tf.constant(row_splits, dtype=tf.int64)
  )

  ds = tf.data.Dataset.from_tensor_slices(
    (ragged_embeddings, targets)
  )

  return ds

### Model Functions

In [ ]:
class LayerNameMixin:
  """A mixin class that provides functionality to generate unique layer names.

    This class is designed to be used in conjunction with Keras models.

    Author:
        Marc C. Hennig (mhennig@hm.edu)
    """
  def _generate_layer_name(self, prefix: str, suffix: Optional[str] = None) -> str:
    """Generates a unique layer name based on the given prefix and suffix.

      The method ensures that the generated name is unique among the existing layer names
      in the `self.layers` collection. If the combination of prefix and suffix already
      exists, it appends a numerical suffix to make the name unique.

      Args:
          prefix (str): The base prefix for the layer name.
          suffix (Optional[str]): An optional suffix to append to the prefix. If None, only the prefix is used.

      Returns:
          str: A unique layer name that does not conflict with existing layer names.

      Example:
          If `prefix` is "conv" and `suffix` is "1", and "conv_1" already exists,
          the method will return "conv_1_1" (or the next available unique name).
    """
    layer_names = set(layer.name for layer in self.layers)

    name = prefix if suffix is None else f"{prefix}_{suffix}"
    name = name.lower()

    if name in layer_names:
      i = 1
      while f"{name}_{i}" in layer_names:
        i += 1
      name = f"{name}_{i}"

    return name

class ConfigSerializerMixin:
  def _serialize_config(self, config: dict) -> dict:

    def _serialize_config_helper(sub_config: dict) -> dict:
      serializable_config = {}
      for k, v in sub_config.items():
        if isinstance(v, dict):
          serializable_config[k] = _serialize_config_helper(v)
        elif isinstance(v, np.ndarray):
          serializable_config[k] = v.tolist()
        else:
          serializable_config[k] = v

      return serializable_config

    return _serialize_config_helper(config)

In [ ]:
class BaseModelMixin:
  """
    A mixin class that provides default configurations for machine learning models,
    including optimizers, loss functions, metrics, and callbacks.

    This class is designed to be extended by specific model types (e.g., regression,
    binary classification, multiclass classification) to provide default behaviors tailored to those tasks.

    Author:
        Marc C. Hennig (mhennig@hm.edu)
  """
  @property
  def default_optimizer(self) -> keras.Optimizer:
    #return keras.optimizers.AdamW(learning_rate=self.default_learning_rate, weight_decay=0.0003, clipnorm=1.0)
    return keras.optimizers.AdamW(learning_rate=self.default_learning_rate, weight_decay=0.01, clipnorm=3.0)

  def is_regression(self, activation: str) -> bool:
    return activation in ['linear', 'relu', 'softplus']

  def is_binary_classification(self, activation: str) -> bool:
    return activation in ['sigmoid']

  def is_multiclass_classification(self, activation: str) -> bool:
    return activation in ['softmax']

  @property
  def default_monitor_metric(self):
    return 'val_loss'

  @property
  def default_learning_rate(self) -> Union[float, keras.optimizers.schedules.LearningRateSchedule]:
    return keras.optimizers.schedules.CosineDecay(
      initial_learning_rate=3e-5,
      decay_steps=TRAIN_STEPS_EPOCH*10
    )

  def find_default_loss(self, activation: str) -> keras.losses.Loss:
    if self.is_regression(activation):
      return self.default_regression_loss
    elif self.is_binary_classification(activation):
      return self.default_binary_classification_loss
    elif self.is_multiclass_classification(activation):
      return self.default_multiclass_classification_loss
    else:
      raise ValueError(f"Unknown activation {activation}")

  def find_default_metrics(self, activation: str) -> List[keras.metrics.Metric]:
    if self.is_regression(activation):
      return self.default_regression_metrics
    elif self.is_binary_classification(activation):
      return self.default_binary_classification_metrics
    elif self.is_multiclass_classification(activation):
      return self.default_multiclass_classification_metrics
    else:
      raise ValueError(f"Unknown activation {activation}")

  @property
  def default_regression_loss(self) -> keras.losses.Loss:
    return keras.losses.LogCosh()

  @property
  def default_binary_classification_loss(self) -> keras.losses.Loss:
    return keras.losses.BinaryCrossentropy()

  @property
  def default_multiclass_classification_loss(self) -> keras.losses.Loss:
    return keras.losses.SparseCategoricalCrossentropy()

  @property
  def default_regression_metrics(self) -> List[keras.metrics.Metric]:
    return [
        keras.metrics.MeanAbsoluteError(),
        keras.metrics.MeanSquaredError(),
        keras.metrics.RootMeanSquaredError(),
        keras.metrics.MeanSquaredLogarithmicError(),
        keras.metrics.MeanAbsolutePercentageError(),
        keras.metrics.LogCoshError(),
    ]

  @property
  def default_binary_classification_metrics(self) -> List[keras.metrics.Metric]:
    return [
        keras.metrics.Accuracy(),
        keras.metrics.SparseCategoricalAccuracy(),
        keras.metrics.F1Score(name="f1_score"),
        keras.metrics.AUC(curve="ROC", name='roc_auc'),
        keras.metrics.AUC(curve="PR", name='pr_auc'),
        keras.metrics.BinaryCrossentropy(),
    ]

  @property
  def default_multiclass_classification_metrics(self) -> List[keras.metrics.Metric]:
    return [
        keras.metrics.SparseCategoricalAccuracy(),
        keras.metrics.F1Score(average='macro', name="f1_score_macro"),
        keras.metrics.F1Score(average='micro', name="f1_score_micro"),
        keras.metrics.F1Score(average='weighted', name="f1_score_weighted"),
        keras.metrics.SparseCategoricalCrossentropy(),
    ]

  def _build_activation_layer(self, activation: str, **kwargs) -> keras.Layer:
    """Builds an activation layer based on the specified activation function.

    Args:
        activation (str): Name of the activation function
        **kwargs: Additional keyword arguments passed to the activation
    Returns:
        keras.Layer: Activation layer
    """
    activation = activation.lower()
    if 'prelu' == activation:
      layer = keras.layers.PReLU(**kwargs)
    elif 'elu' == activation:
      layer = keras.layers.ELU(**kwargs)
    elif 'relu' == activation:
      layer = keras.layers.ReLU(**kwargs)
    elif 'leaky_relu' == activation:
      layer = keras.layers.LeakyReLU(**kwargs)
    else:
      layer = keras.layers.Activation(activation, **kwargs)

    return layer

  def _build_norm_layer(self, norm_type: str, **kwargs) -> keras.Layer:
    """Builds a normalization layer based on the specified normalization type.

    Args:
        **kwargs: Additional keyword arguments passed to the normalization layer

    Returns:
        keras.Layer: Normalization layer (BatchNorm, LayerNorm, or Identity)

    Raises:
        NotImplementedError: If norm_type is not one of [None, "batch", "layer"]
    """
    if norm_type is None:
      layer = keras.layers.Identity()
    elif norm_type == 'batch':
      layer = keras.layers.BatchNormalization(**kwargs)
    elif norm_type == 'layer':
      layer = keras.layers.LayerNormalization(**kwargs)
    else:
      raise NotImplementedError(f"Unknown normalization type {norm_type}")

    return layer

  def freeze_layers(self, layer_names: List[str]):
    """Freeze specific layers by name"""
    for name in layer_names:
      layer = getattr(self, name, None)
      if layer is not None:
        layer.trainable = False

  def unfreeze_layers(self, layer_names: List[str]):
    """Unfreeze specific layers by name"""
    for name in layer_names:
      layer = getattr(self, name, None)
      if layer is not None:
        layer.trainable = True

  def default_callbacks(
    self,
    log_file: Optional[str] = None,
    tensorboard_dir: Optional[str] = None,
    backup_dir: Optional[str] = None,
    checkpoint_file: Optional[str] = None,
    **kwargs
  ) -> List[keras.callbacks.Callback]:

    callbacks = []
    callbacks.append(keras.callbacks.ReduceLROnPlateau(patience=5, min_lr=DEFAULT_MIN_LEARNING_RATE, factor=0.5, monitor=self.default_monitor_metric, verbose=1))

    if log_file is not None:
      callbacks.append(keras.callbacks.CSVLogger(
        filename=log_file if log_file.endswith(".csv") else f"{log_file}.csv",
        separator=";",
        append=backup_dir is not None,
      ))

    if checkpoint_file is not None:
      callbacks.append(keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_file,
        monitor=self.default_monitor_metric,
        save_best_only=True,
        save_weights_only=False,
        mode='auto',
        save_freq='epoch',
        verbose=1,
      ))

    if tensorboard_dir is not None:
      callbacks.append(keras.callbacks.TensorBoard(
        log_dir=tensorboard_dir,
        update_freq='epoch',
        histogram_freq=5,
        embeddings_freq=5,
        write_graph=True,
        write_images=True,
        #profile_batch='10,30',
      ))

    if backup_dir is not None:
      callbacks.append(keras.callbacks.BackupAndRestore(
        backup_dir,
        save_freq='epoch',
        delete_checkpoint=False,
      ))

    callbacks = callbacks + [
      keras.callbacks.TerminateOnNaN(),
      keras.callbacks.EarlyStopping(
        monitor=self.default_monitor_metric,
        mode='auto',
        patience=20,
        verbose=1,
        restore_best_weights=True,
      ),
    ]
    return callbacks

  __author__ = "Marc C. Hennig"

class RegressionModelMixin(BaseModelMixin):
  """A mixin class for regression models. Extends `BaseModelMixin` to provide
    default loss and metrics for regression tasks.

    Author:
      Marc C. Hennig (mhennig@hm.edu)
   """
  @property
  def default_loss(self) -> keras.losses.Loss:
    return self.default_regression_loss

  @property
  def default_metrics(self) -> List[keras.metrics.Metric]:
    return self.default_regression_metrics

class MulticlassModelMixin(BaseModelMixin):
  """A mixin class for multiclass classification models. Extends `BaseModelMixin` to provide
    default loss and metrics for multiclass classification tasks.

    Author:
      Marc C. Hennig (mhennig@hm.edu)
  """
  @property
  def default_loss(self) -> keras.losses.Loss:
    return self.default_multiclass_classification_loss

  @property
  def default_metrics(self) -> List[keras.metrics.Metric]:
    return self.default_multiclass_classification_metrics

class BinaryclassModelMixin(BaseModelMixin):
  """A mixin class for binary classification models. Extends `BaseModelMixin` to provide
    default loss and metrics for multiclass classification tasks.

    Author:
      Marc C. Hennig (mhennig@hm.edu)
  """
  @property
  def default_loss(self) -> keras.losses.Loss:
    return self.default_binary_classification_loss

  @property
  def default_metrics(self) -> List[keras.metrics.Metric]:
    return self.default_binary_classification_metrics

def get_default_callbacks(
    log_file: Optional[str] = None,
    tensorboard_dir: Optional[str] = None,
    backup_dir: Optional[str] = None,
    checkpoint_file: Optional[str] = None,
  ) -> List[keras.callbacks.Callback]:

    def learning_rate_scheduler(epoch: int, learning_rate: float):
      if learning_rate_warmup is not None and epoch <= learning_rate_warmup.warmup_epochs:
        return learning_rate_warmup(epoch, learning_rate)
      elif learning_rate_decay is not None:
        return learning_rate_decay(epoch, learning_rate)
      else:
        return learning_rate

    callbacks = []
    if log_file is not None:
      callbacks.append(keras.callbacks.CSVLogger(
        filename=log_file if log_file.endswith(".csv") else f"{log_file}.csv",
        separator=";",
        append=backup_dir is not None,
      ))

    if checkpoint_file is not None:
      callbacks.append(keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_file,
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=False,
        mode='auto',
        save_freq='epoch',
        verbose=1,
      ))

    if tensorboard_dir is not None:
      callbacks.append(keras.callbacks.TensorBoard(
        log_dir=tensorboard_dir,
        update_freq='epoch',
        histogram_freq=5,
        embeddings_freq=5,
        write_graph=True,
        write_images=True,
        #profile_batch='10,30',
      ))

    if backup_dir is not None:
      callbacks.append(keras.callbacks.BackupAndRestore(
        backup_dir,
        save_freq='epoch',
        delete_checkpoint=False,
      ))
    callbacks = callbacks + [
      keras.callbacks.TerminateOnNaN(),
      keras.callbacks.LearningRateScheduler(learning_rate_scheduler, verbose=1),
      keras.callbacks.EarlyStopping(
        monitor='val_loss',
        mode='auto',
        patience=20,
        verbose=1,
        restore_best_weights=True,
        start_from_epoch=20,
      ),
    ]
    return callbacks


In [ ]:
def model_visualize_history(model: keras.Model, history = None):
  if history is not None:
    history = history.history
  else:
    history = model.history.history

  for metric in [key for key in history.keys() if not key.startswith('val_')]:
    if metric.startswith(DEFAULT_NEXT_ACTIVITY_OUTPUT):
      title = DEFAULT_NEXT_ACTIVITY_OUTPUT.replace("_", " ").title()
      y_label = metric.replace(DEFAULT_NEXT_ACTIVITY_OUTPUT, '').replace("_", " ").title()
    elif metric.startswith(DEFAULT_NEXT_TIME_OUTPUT):
      title = DEFAULT_NEXT_TIME_OUTPUT.replace("_", " ").title()
      y_label = metric.replace(DEFAULT_NEXT_TIME_OUTPUT, '').replace("_", " ").title()
    elif metric.startswith(DEFAULT_REMAINING_TIME_OUTPUT):
      title = DEFAULT_REMAINING_TIME_OUTPUT.replace("_", " ").title()
      y_label = metric.replace(DEFAULT_REMAINING_TIME_OUTPUT, '').replace("_", " ").title()
    else:
      title = ""
      y_label = metric.replace("_", " ").title()

    plt.plot(history[metric])
    if f"val_{metric}" in history:
      plt.plot(history[f"val_{metric}"])

    plt.title(title)
    plt.ylabel(y_label)
    plt.xlabel("Epoch")
    if f"val_{metric}" in history:
      plt.legend(["train", "test"], loc='upper left')
    plt.tight_layout()
    plt.savefig(
        os.path.join(GRAPHIC_DIR, f"{model.name}_{metric}.svg"),
        bbox_inches='tight'
    )
    plt.show()

def model_regression_report(model: keras.Model, ds: tf.data.Dataset, target: str, archive: bool = False):
  if not isinstance(next(iter(ds))[1], dict):
    y_true = np.concatenate([y for x, y in ds], axis=0).flatten()
  else:
    y_true = np.concatenate([y[target] for x, y in ds], axis=0).flatten()

  y_pred = model.predict(ds)

  if isinstance(y_pred, dict):
    y_pred = y_pred[target]
  elif hasattr(model, 'output_names') and len(model.output_names) > 1:
    target_idx = model.output_names.index(target)
    y_pred = y_pred[target_idx]

  y_pred = y_pred.flatten()

  result = {
      "MAE": float(sl.metrics.mean_absolute_error(y_true, y_pred)),
      "MSE": float(sl.metrics.mean_squared_error(y_true, y_pred)),
      "RMSE": float(sl.metrics.root_mean_squared_error(y_true, y_pred)),
      "MEDAE": float(sl.metrics.median_absolute_error(y_true, y_pred)),
      "MAPE": float(sl.metrics.mean_absolute_percentage_error(y_true, y_pred)),
  }

  if y_true.min() >= 0 and y_pred.min() >= 0:
    result.update({
      "MSLE": float(sl.metrics.mean_squared_log_error(y_true, y_pred)),
      "RMSLE": float(sl.metrics.root_mean_squared_log_error(y_true, y_pred)),
    })

  filename_prefix = model.name if target in model.name else f"{model.name}_{target}"

  try:
    if archive:
      np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{filename_prefix}.npz"), y_true=y_true.astype(np.float64), y_pred=y_pred.astype(np.float64))
    else:
      np.save(os.path.join(OUTPUT_DATA_DIR, f"{filename_prefix}_predictions.npy"), y_pred.astype(np.float64), allow_pickle=False)
      np.save(os.path.join(OUTPUT_DATA_DIR, f"{filename_prefix}_groundtruth.npy"), y_true.astype(np.float64), allow_pickle=False)
  except Exception as e:
    print(f"Failed to save predicitons: {e}")

  print("\t\t".join(result.keys()))
  print("\t\t".join(["%.4f" % result for result in result.values()]))

  with open(os.path.join(OUTPUT_DATA_DIR, f"{filename_prefix}_regression_report.json"), 'w') as file:
    json.dump(result, file)

  return result

def model_classification_report(model: keras.Model, ds: tf.data.Dataset, target: str, archive: bool = False):
  if not isinstance(next(iter(ds))[1], dict):
    y_true = np.concatenate([y for x, y in ds], axis=0).flatten()
  else:
    y_true = np.concatenate([y[target] for x, y in ds], axis=0).flatten()

  y_pred = model.predict(ds)

  if isinstance(y_pred, dict):
    y_pred = y_pred[target]
  elif hasattr(model, 'output_names') and len(model.output_names) > 1:
    target_idx = model.output_names.index(target)
    y_pred = y_pred[target_idx]

  filename_prefix = model.name if target in model.name else f"{model.name}_{target}"

  try:
    if archive:
      np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{filename_prefix}.npz"), y_true=y_true.astype(np.float64), y_pred=y_pred.astype(np.float64))
    else:
      np.save(os.path.join(OUTPUT_DATA_DIR, f"{filename_prefix}_predictions.npy"), y_pred.astype(np.float64), allow_pickle=False)
      np.save(os.path.join(OUTPUT_DATA_DIR, f"{filename_prefix}_groundtruth.npy"), y_true.astype(np.float64), allow_pickle=False)

  except Exception as e:
    print(f"Failed to save predictions: {e}")

  y_pred = np.argmax(y_pred, axis=1)

  print(sl.metrics.classification_report(y_true, y_pred, digits=4))
  report = sl.metrics.classification_report(y_true, y_pred, digits=4, output_dict=True)

  report.update({
      "balanced_accuracy": sl.metrics.balanced_accuracy_score(y_true, y_pred),
  })

  with open(os.path.join(OUTPUT_DATA_DIR, f"{filename_prefix}_classification_report.json"), 'w') as file:
    json.dump(report, file)

  confusion_matrix = sl.metrics.confusion_matrix(y_true, y_pred)
  confusion_disp = sl.metrics.ConfusionMatrixDisplay(confusion_matrix=confusion_matrix)
  confusion_disp.plot()
  plt.title("Confusion Matrix")
  plt.savefig(os.path.join(GRAPHIC_DIR, f"{filename_prefix}_confusion_matrix.svg"))
  plt.show()
  return report

def model_save_files(model: keras.Model, ds: tf.data.Dataset = None):
  history = model.history.history
  with open(os.path.join(OUTPUT_DATA_DIR, f"{model.name}_history.json"), 'w') as file:
    json.dump(history, file)

  model.save(
      os.path.join(MODEL_DIR, f"{model.name}.keras"),
      overwrite=True
  )

  if ds is not None:
    evaluation = model.evaluate(
        ds,
        return_dict=True,
        verbose=1,
    )
    with open(os.path.join(OUTPUT_DATA_DIR, f"{model.name}_evaluation.json"), 'w') as file:
      json.dump(evaluation, file)

  keras.utils.plot_model(
    model,
    to_file=os.path.join(GRAPHIC_DIR, f"{model.name}.png"),
    show_shapes=True,
    show_dtype=True,
    show_layer_names=False,
    dpi=400,
    show_layer_activations=True,
  )
  try:
    keras.utils.plot_model(
      model,
      to_file=os.path.join(GRAPHIC_DIR, f"{model.name}.svg"),
      show_shapes=True,
      show_dtype=True,
      show_layer_names=False,
      show_layer_activations=True,
    )
  except:
    pass

### Visualization Functions

In [ ]:
def visualize_umap_embeddings(
  data: np.ndarray,
  n_neighbors: int = 20,
  min_dist: float = 0.1,
  n_components: int = 2,
  metric='euclidean',
  title=''
):
    fit = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=n_components,
        metric=metric,
        unique=True,
    )
    u = fit.fit_transform(data)
    fig = plt.figure()
    if n_components == 1:
        ax = fig.add_subplot(111)
        ax.scatter(u[:,0], range(len(u)))
    if n_components == 2:
        ax = fig.add_subplot(111)
        ax.scatter(u[:,0], u[:,1])
    if n_components == 3:
        ax = fig.add_subplot(111, projection='3d')
        ax.scatter(u[:,0], u[:,1], u[:,2], s=100)
    plt.title(title, fontsize=18)

### Model Comparison

In [ ]:
def evaluate_regression(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
  if y_true.ndim > 1:
      y_true = np.argmax(y_true, axis=1)
  if y_pred.ndim > 1:
      y_pred = np.argmax(y_pred, axis=1)

  def logcosh_error(y_true, y_pred):
    error = np.subtract(y_pred, y_true)
    return np.mean(np.log((np.exp(error) + np.exp(-error))/2))

  return {
    'mae': sl.metrics.mean_absolute_error(y_true, y_pred),
    'mse': sl.metrics.mean_squared_error(y_true, y_pred),
    'rmse': sl.metrics.root_mean_squared_error(y_true, y_pred),
    'mape': sl.metrics.mean_absolute_percentage_error(y_true, y_pred),
    'msle': sl.metrics.mean_squared_log_error(y_true, y_pred),
    'medae': sl.metrics.median_absolute_error(y_true, y_pred),
    'logcosh': logcosh_error(y_true, y_pred),
    'max_error': sl.metrics.max_error(y_true, y_pred),
  }

def evaluate_multiclass_classification(y_true: np.ndarray, y_pred: np.ndarray, num_samples: Optional[int] = None) -> Dict[str, float]:
  if y_true.ndim > 1:
      y_true = np.argmax(y_true, axis=1)
  if y_pred.ndim > 1:
      y_pred = np.argmax(y_pred, axis=1)

  return {
    'accuracy': sl.metrics.accuracy_score(y_true, y_pred),
    'accuracy_balanced': sl.metrics.balanced_accuracy_score(y_true, y_pred),
    'accuracy_balanced_adjusted': sl.metrics.balanced_accuracy_score(y_true, y_pred, adjusted=True),
    'f1_micro': sl.metrics.f1_score(y_true, y_pred, average='micro'),
    'f1_macro': sl.metrics.f1_score(y_true, y_pred, average='macro'),
    'f1_weighted': sl.metrics.f1_score(y_true, y_pred, average='weighted'),
    'precision_micro': sl.metrics.precision_score(y_true, y_pred, average='micro', zero_division='warn'),
    'precision_macro': sl.metrics.precision_score(y_true, y_pred, average='macro'),
    'precision_weighted': sl.metrics.precision_score(y_true, y_pred, average='weighted'),
    'recall_micro': sl.metrics.recall_score(y_true, y_pred, average='micro'),
    'recall_macro': sl.metrics.recall_score(y_true, y_pred, average='macro'),
    'recall_weighted': sl.metrics.recall_score(y_true, y_pred, average='weighted'),
  }

def evaluate_embeddings(
  embeddings: np.ndarray,
  original_embeddings: Optional[np.ndarray] = None,
  num_samples: Optional[int] = None,
  return_df: bool = False
) -> Union[dict, pd.DataFrame]:
  if embeddings.ndim > 2:
    embeddings = np.squeeze(embeddings)

  if embeddings.ndim != 2:
    raise ValueError(f"Embeddings must be 2D, got {embeddings.ndim}D")

  if original_embeddings is not None:
    if original_embeddings.ndim > 2:
      original_embeddings = np.squeeze(original_embeddings)

    if original_embeddings.ndim != 2:
      raise ValueError(f"Original embeddings must be 2D, got {original_embeddings.ndim}D")
    elif original_embeddings.shape[0] != embeddings.shape[0]:
      raise ValueError(f"Original embeddings must have the same shape ad dimension 0 as embeddings, got {original_embeddings.shape[0]} and {embeddings.shape[0]}")

  indices = np.arange(embeddings.shape[0])
  if num_samples is not None and num_samples <= embeddings.shape[0]:
    indices = np.random.default_rng().choice(embeddings.shape[0], size=num_samples, replace=False)
    embeddings = embeddings[indices]

  eval = {
    'uniformity': uniformity(embeddings).numpy(),
    'effective_rank': effective_rank(embeddings).numpy(),
  }

  if original_embeddings is not None:
    original_embeddings = original_embeddings[indices]

    eval.update({
      'trustworthiness': sl.manifold.trustworthiness(original_embeddings, embeddings, metric='cosine', n_neighbors=15),
      'continuity': continuity(original_embeddings, embeddings, metric='cosine', n_neighbors=15),
    })

  if return_df:
    return pd.DataFrame.from_dict(eval, orient='index', columns=["Value"])
  else:
    return eval

In [ ]:
def df_generate_simple_event_embeddings(
  df_train: pd.DataFrame,
  df_test: Optional[pd.DataFrame] = None,
  cat_attrs: Union[str, List[str]] = [],
  num_attrs: Union[str, List[str]] = [],
) -> np.ndarray:
  if isinstance(cat_attrs, str):
    cat_attrs = [cat_attrs]
  if isinstance(num_attrs, str):
    num_attrs = [num_attrs]

  df_complete = pd.concat([df_train, df_test]) if df_test is not None else df_train

  ohe = sl.preprocessing.OneHotEncoder(sparse_output=False, dtype=np.int8)
  ohe.fit(df_complete[cat_attrs])

  rbe = sl.preprocessing.RobustScaler()
  rbe.fit(df_train[num_attrs])

  train_cat_embeddings = ohe.transform(df_train[cat_attrs])
  train_num_embeddings = rbe.transform(df_train[num_attrs])
  train_embeddings = np.concatenate([train_cat_embeddings, train_num_embeddings], axis=-1, dtype=np.float32)

  if df_test is not None:
    test_cat_embeddings = ohe.transform(df_test[cat_attrs])
    test_num_embeddings = rbe.transform(df_test[num_attrs])
    test_embeddings = np.concatenate([test_cat_embeddings, test_num_embeddings], axis=-1, dtype=np.float32)
    return train_embeddings, test_embeddings

  return train_embeddings


## General Layers

In [ ]:
class PadAndTruncate(keras.Layer):
  def __init__(self, max_seq_len: int, pad_token: float = 0.0, **kwargs):
    kwargs.pop("trainable", None)
    super().__init__(trainable=False, **kwargs)

    self.max_seq_len = max_seq_len
    self.pad_token = pad_token

    self.masking = keras.layers.Masking(mask_value=pad_token)

  def build(self, input_shape: tuple[Optional[int], int, int]):
    self.masking.build(input_shape)

  def compute_output_shape(self, input_shape: tuple[Optional[int], int, int]) -> tuple[Optional[int], int, int]:
    return (input_shape[0], self.max_seq_len, input_shape[2])

  def compute_mask(self, inputs: keras.KerasTensor, mask: Optional[keras.KerasTensor] = None) -> Optional[keras.KerasTensor]:
    return self.masking.compute_mask(inputs)

  def get_config(self) -> dict:
    config = super().get_config()
    config.update({
      "max_seq_len": self.max_seq_len,
      "pad_token": self.pad_token,
    })
    return config

  @classmethod
  def from_config(cls, config: dict) -> 'PadAndTruncate':
    return cls(**config)

  def call(self, inputs: keras.KerasTensor, mask: Optional[keras.KerasTensor] = None) -> keras.KerasTensor:
    batch_size, seq_len, feature_dim = keras.ops.shape(inputs)

    mask = self.masking.compute_mask(inputs, mask)
    def pad_inputs():
      return keras.ops.pad(
        inputs,
        ((0, 0), (0, self.max_seq_len - seq_len), (0, 0)),
        constant_values=self.pad_token
      )

    def trunc_inputs():
      seq_lens = keras.ops.sum(keras.ops.cast(mask, 'int32'), axis=1)

      # Calculate start indices for each sequence to get the last self.seq_len tokens
      start_indices = keras.ops.maximum(
        keras.ops.zeros_like(seq_lens),
        seq_lens - self.max_seq_len
      )

      # Calculate start indices to capture the last max_seq_len tokens
      start_indices = keras.ops.maximum(0, seq_lens - self.max_seq_len)

      # Shape: (batch_size, max_seq_len)
      window_indices = keras.ops.arange(self.max_seq_len)
      window_indices = keras.ops.reshape(window_indices, (1, self.max_seq_len))
      window_indices = keras.ops.repeat(window_indices, batch_size, axis=0)

      # Add start offsets
      start_indices = keras.ops.reshape(start_indices, (batch_size, 1))
      window_indices = window_indices + start_indices

      # Expand to feature dimension and gather
      # Shape: (batch_size, max_seq_len, 1)
      window_indices = keras.ops.expand_dims(window_indices, axis=-1)

      # Gather along sequence dimension
      return keras.ops.take_along_axis(inputs, window_indices, axis=1)

    x = keras.ops.cond(
      keras.ops.less_equal(seq_len, self.max_seq_len),
      pad_inputs,
      trunc_inputs
    )

    return self.masking(x)

### Loss Functions

#### Contrastive Losses

In [ ]:
@keras.utils.register_keras_serializable()
class NTXentLoss(keras.Loss):
  """
    Normalized Temperature-scaled Cross-Entropy (NT-Xent) Loss for contrastive learning.

    This loss function implements the NT-Xent loss introduced in SimCLR, which learns
    representations by maximizing agreement between differently augmented views of the
    same data example via a contrastive loss in the latent space.

    The loss works by:
    1. L2-normalizing the input projections
    2. Computing cosine similarities between all pairs in the batch
    3. Treating diagonal elements as positive pairs, off-diagonal as negatives
    4. Applying temperature scaling and symmetric cross-entropy loss

    Args:
        temperature (float, optional): Temperature parameter for scaling similarities.
            Lower values (0.05-0.1) create sharper distributions and focus on hard
            negatives. Higher values (0.5-1.0) create smoother distributions.
            Defaults to 0.1.
        **kwargs: Additional arguments passed to keras.Loss parent class.

    References:
        - SimCLR: A Simple Framework for Contrastive Learning of Visual Representations
          https://arxiv.org/abs/2002.05709
        - Understanding Contrastive Representation Learning through Alignment and Uniformity
          https://arxiv.org/abs/2005.10242

    Example:
        >>> # Basic usage
        >>> loss_fn = NTXentLoss(temperature=0.07)
        >>> proj1 = keras.random.normal((32, 128))  # batch_size=32, feature_dim=128
        >>> proj2 = keras.random.normal((32, 128))  # Augmented views of same samples
        >>> loss = loss_fn(proj1, proj2)

        >>> # In a training loop
        >>> for (x1, x2), _ in dataset:
        ...     proj1 = model(x1)
        ...     proj2 = model(x2)
        ...     loss = loss_fn(proj1, proj2)
    """
  def __init__(self, temperature: float = 0.1, normalize: bool = True, **kwargs):
    super(NTXentLoss, self).__init__(**kwargs)
    self.temperature = temperature
    self.normalize = normalize

  def call(self, projections_1: keras.KerasTensor, projections_2: keras.KerasTensor) -> keras.KerasTensor:
    projections_1 = keras.ops.convert_to_tensor(projections_1, dtype=keras.backend.floatx())
    projections_2 = keras.ops.convert_to_tensor(projections_2, dtype=keras.backend.floatx())

    batch_size = keras.ops.shape(projections_1)[0]

    if self.normalize:
      projections_1 = keras.utils.normalize(projections_1, order=2, axis=1)
      projections_2 = keras.utils.normalize(projections_2, order=2, axis=1)

    # Concatenate both views: shape (2*batch_size, feature_dim)
    projections = keras.ops.concatenate([projections_1, projections_2], axis=0)

    # Compute full similarity matrix (cosine similarity of l2 norm): (2*batch_size, 2*batch_size)
    similarity_matrix = keras.ops.matmul(projections, keras.ops.transpose(projections))
    similarity_matrix = similarity_matrix / self.temperature

    # Create mask to exclude self-similarities (diagonal)
    mask = keras.ops.eye(batch_size * 2, dtype="bool")
    similarity_matrix = keras.ops.where(
      mask,
      keras.ops.ones_like(similarity_matrix) * -1e9,  # Mask out diagonal (large negative will be zero in softmax)
      similarity_matrix
    )

    # Positive pairs are at specific indices:
    # For sample i: positive is at i+batch_size (if i < batch_size) or i-batch_size (if i >= batch_size)
    # Create labels: [batch_size, batch_size+1, ..., 2*batch_size-1, 0, 1, ..., batch_size-1]
    positives_idx = keras.ops.concatenate([
      keras.ops.arange(batch_size, 2 * batch_size),
      keras.ops.arange(0, batch_size)
    ], axis=0)

    # Compute cross-entropy loss
    loss = keras.losses.sparse_categorical_crossentropy(
      positives_idx,
      similarity_matrix,
      from_logits=True
    )

    return keras.ops.mean(loss)

  @classmethod
  def from_config(cls, config):
    return cls(**config)

  def get_config(self):
    config = super().get_config()
    config.update({"temperature": self.temperature, "normalize": self.normalize})
    return config

def nt_xent_loss(projections_1: keras.KerasTensor, projections_2: keras.KerasTensor, temperature: float = 0.1):
    loss_fn = NTXentLoss(temperature=temperature)
    return loss_fn(projections_1, projections_2)

#### Info-Max Losses

In [ ]:
@keras.utils.register_keras_serializable()
class BarlowTwinsLoss(keras.Loss):
  """
    Barlow Twins Loss for self-supervised representation learning.

    This loss function learns representations by making the cross-correlation matrix
    of twin embeddings close to the identity matrix. This approach:
    - Encourages invariance to augmentations (diagonal elements → 1)
    - Reduces redundancy between features (off-diagonal elements → 0)
    - Naturally prevents representation collapse without negative pairs

    The method is particularly effective for creating well-distributed, uniform
    embeddings while maintaining semantic invariance, making it ideal for scenarios
    where you want to balance alignment and uniformity.

    Mathematical Formulation:
        Loss = Σ(1 - C_ii)² + λ * ΣΣ(i≠j) C_ij²
        where C is the cross-correlation matrix between normalized embeddings

        - First term: Invariance to augmentations (diagonal closeness to 1)
        - Second term: Redundancy reduction (decorrelation of features)

    Args:
        lambda_param (float, optional): Weight for the redundancy reduction term.
            Higher values more aggressively decorrelate features. Typical range
            is [1e-5, 1e-2]. Defaults to 5e-3.
        eps (float, optional): Small epsilon value for numerical stability in
            standardization. Defaults to 1e-5.
        **kwargs: Additional arguments passed to the keras.Loss parent class.

    References:
        - Barlow Twins: Self-Supervised Learning via Redundancy Reduction
          https://arxiv.org/abs/2103.03230
        - Official implementation: https://github.com/facebookresearch/barlowtwins

    Example:
        >>> # Basic usage with default parameters
        >>> loss_fn = BarlowTwinsLoss()
        >>> proj1 = keras.random.normal((32, 128))  # First augmented views
        >>> proj2 = keras.random.normal((32, 128))  # Second augmented views
        >>> loss = loss_fn(proj1, proj2)

        >>> # Emphasizing feature decorrelation
        >>> decorr_loss = BarlowTwinsLoss(lambda_param=1e-2)
        >>> # This strongly encourages feature independence and uniformity

        >>> # In a training loop
        >>> for (x1, x2), _ in dataset:
        ...     z1 = model(x1)  # Projections from first augmentation
        ...     z2 = model(x2)  # Projections from second augmentation
        ...     loss = loss_fn(z1, z2)
  """
  def __init__(self, lambda_param: float = 5e-3, eps: float = 1e-5, **kwargs):
    super().__init__(**kwargs)
    self.lambda_param = lambda_param
    self.eps = eps

  def call(self, projections_1: keras.KerasTensor, projections_2: keras.KerasTensor) -> keras.KerasTensor:
    projections_1 = keras.ops.convert_to_tensor(projections_1, dtype=keras.backend.floatx())
    projections_2 = keras.ops.convert_to_tensor(projections_2, dtype=keras.backend.floatx())

    batch_size = keras.ops.cast(keras.ops.shape(projections_1)[0], keras.ops.dtype(projections_1))

    # Normalize the projections along the batch dimension
    z1_norm = self._standardize(projections_1)
    z2_norm = self._standardize(projections_2)

    # Compute cross-correlation matrix
    c = keras.ops.matmul(keras.ops.transpose(z1_norm), z2_norm) / batch_size
    #c = keras.ops.clip(c, -1.0, 1.0)

    # Loss function
    on_diag = keras.ops.sum(keras.ops.square(1 - keras.ops.diag(c)))
    off_diag = self._off_diagonal_loss(c)

    loss = on_diag + self.lambda_param * off_diag
    return loss

  def _standardize(self, x):
    mean = keras.ops.mean(x, axis=0, keepdims=True)
    var = keras.ops.var(x, axis=0, keepdims=True)
    return (x - mean) / keras.ops.sqrt(var + self.eps)

  def _off_diagonal_loss(self, c):
    """Compute loss for off-diagonal elements"""
    n = keras.ops.shape(c)[0]
    # Create mask to select off-diagonal elements
    mask = 1 - keras.ops.eye(n)
    off_diag_elements = c * mask
    return keras.ops.sum(keras.ops.square(off_diag_elements))

  def get_config(self):
    config = super().get_config()
    config.update({
      "lambda_param": self.lambda_param,
      "eps": self.eps
    })
    return config

In [ ]:
@keras.utils.register_keras_serializable()
class VICRegLoss(keras.Loss):
  """
    VICReg (Variance-Invariance-Covariance Regularization) Loss for self-supervised learning.

    This loss function learns representations by enforcing three key properties:
    1. **Variance**: Preserves information in the embedding space by preventing dimension collapse
    2. **Invariance**: Maximizes agreement between positive pairs (different views of same sample)
    3. **Covariance**: Decorrelates features to encourage diverse, non-redundant representations

    The loss is effective for improving uniformity while maintaining reasonable
    alignment, making it suitable for scenarios where you want to prevent over-clustering
    and encourage more uniformly distributed embeddings.

    Mathematical Formulation:
        Total Loss = sim_coeff * Invariance + var_coeff * Variance + cov_coeff * Covariance

        - Invariance: Mean squared distance between positive pairs
        - Variance: Hinge loss to maintain standard deviation above 1.0 per dimension
        - Covariance: Penalty for off-diagonal elements in the covariance matrix

    Args:
        sim_coeff (float, optional): Weight for the invariance (similarity) term.
            Higher values emphasize pulling positive pairs together. Defaults to 25.0.
        var_coeff (float, optional): Weight for the variance term.
            Higher values more strongly prevent dimension collapse. Defaults to 25.0.
        cov_coeff (float, optional): Weight for the covariance term.
            Higher values enforce more feature decorrelation. Defaults to 1.0.
        eps (float, optional): Small epsilon value for numerical stability in
            standard deviation calculations. Defaults to 1e-4.
        **kwargs: Additional arguments passed to the keras.Loss parent class.

    References:
        - VICReg: Variance-Invariance-Covariance Regularization for Self-Supervised Learning
          https://arxiv.org/abs/2105.04906
        - Official implementation: https://github.com/facebookresearch/vicreg

    Example:
        >>> # Basic usage with default parameters
        >>> loss_fn = VICRegLoss()
        >>> proj1 = keras.random.normal((32, 128))  # First augmented views
        >>> proj2 = keras.random.normal((32, 128))  # Second augmented views of same samples
        >>> loss = loss_fn(proj1, proj2)

        >>> # Emphasizing uniformity and feature diversity
        >>> uniform_loss = VICRegLoss(var_coeff=50.0, cov_coeff=2.0)
        >>> # This configuration strongly encourages uniform feature distribution

        >>> # In a training loop
        >>> model.compile(optimizer='adam', loss=VICRegLoss())
        >>> for (x1, x2), _ in dataset:
        ...     with tf.GradientTape():
        ...         z1 = model(x1)  # Projections from first augmentation
        ...         z2 = model(x2)  # Projections from second augmentation
        ...         loss = loss_fn(z1, z2)

    Note:
        - Unlike contrastive methods, VICReg doesn't require negative pairs or large batch sizes
        - The variance term acts as a regularizer against representation collapse
        - Well-suited for scenarios where batch size is limited or computational resources are constrained
        - Typically produces embeddings with good uniformity and semantic structure
  """
  def __init__(
    self,
    sim_coeff: float = 25.0,
    var_coeff: float = 25.0,
    cov_coeff: float = 1.0,
    eps: float = 1e-4,
    normalize: bool = False,
    **kwargs
  ):
    super().__init__(**kwargs)
    self.sim_coeff = sim_coeff
    self.var_coeff = var_coeff
    self.cov_coeff = cov_coeff
    self.eps = eps
    self.normalize = normalize

  def call(self, projections_1: keras.KerasTensor, projections_2: keras.KerasTensor) -> keras.KerasTensor:
    projections_1 = keras.ops.convert_to_tensor(projections_1, dtype=keras.backend.floatx())
    projections_2 = keras.ops.convert_to_tensor(projections_2, dtype=keras.backend.floatx())

    batch_size = keras.ops.shape(projections_1)[0]

    if self.normalize:
      projections_1 = keras.utils.normalize(projections_1, order=2, axis=1)
      projections_2 = keras.utils.normalize(projections_2, order=2, axis=1)

    # Invariance term (similarity between positive pairs)
    invariance_loss = keras.ops.mean(keras.ops.square(projections_1 - projections_2))

    # Variance term (encourages variance along batch dimension)
    var_z1 = keras.ops.var(projections_1, axis=0)
    var_z2 = keras.ops.var(projections_2, axis=0)
    std_z1 = keras.ops.sqrt(var_z1 + self.eps)
    std_z2 = keras.ops.sqrt(var_z2 + self.eps)
    variance_loss = keras.ops.mean(keras.ops.relu(1 - std_z1)) + keras.ops.mean(keras.ops.relu(1 - std_z2))

    # Covariance term (decorrelates features)
    cov_z1 = self._covariance_loss(projections_1)
    cov_z2 = self._covariance_loss(projections_2)
    covariance_loss = cov_z1 + cov_z2

    loss = (
      self.sim_coeff * invariance_loss +
      self.var_coeff * variance_loss +
      self.cov_coeff * covariance_loss
    )
    return loss

  def _covariance_loss(self, z):
    """Compute covariance regularization term"""
    batch_size = keras.ops.shape(z)[0]
    num_features = keras.ops.shape(z)[1]

    # Center the features
    z = z - keras.ops.mean(z, axis=0, keepdims=True)

    # Compute covariance matrix
    cov = keras.ops.matmul(keras.ops.transpose(z), z) / (keras.ops.cast(batch_size, keras.ops.dtype(z)) - 1.0)

    # Sum of squared off-diagonal elements
    mask = 1.0 - keras.ops.eye(num_features)
    off_diag_sum = keras.ops.sum(keras.ops.square(cov * mask))

    # Normalize by number of off-diagonal elements
    return off_diag_sum / num_features

  def get_config(self):
    config = super().get_config()
    config.update({
      "sim_coeff": self.sim_coeff,
      "var_coeff": self.var_coeff,
      "cov_coeff": self.cov_coeff,
      "eps": self.eps
    })
    return config

### Metric Functions

In [ ]:
def continuity(X, X_embedded, *, n_neighbors=5, metric="euclidean"):
    """
    Measures continuity: extent to which original neighbors are preserved in embedding.
    Higher values indicate better continuity (original neighbors remain close in embedding).

    This metric quantifies how well the original neighborhood relationships from the input space
    are preserved in the low-dimensional embedding. It's computed as:

        continuity = 1 - (2 / (n * k * (2 * n - 3 * k - 1))) *
                    sum_{i=1 to n} sum_{j in N_i^k} max(0, (r'(i, j) - k))

    where:
        n: number of samples
        k: number of neighbors considered (n_neighbors)
        N_i^k: the k nearest neighbors of sample i in the original space
        r'(i, j): rank of sample j in the embedding space neighborhood of sample i

    In essence, it penalizes when original neighbors appear far away in the embedding,
    with the penalty weighted by how far they've moved in the neighborhood ranking.

    Args:
        X: {array-like, sparse matrix} of shape (n_samples, n_features) or
           (n_samples, n_samples) if metric='precomputed'
           Original high-dimensional data or precomputed distance matrix.
        X_embedded: {array-like, sparse matrix} of shape (n_samples, n_components)
                    Low-dimensional embedding of the data.
        n_neighbors: int, default=5
                    Number of neighbors to consider for continuity calculation.
                    Should be less than n_samples/2 for meaningful results.
        metric: str or callable, default='euclidean'
                Distance metric for original space. Use 'precomputed' if X is a
                distance matrix.

    Returns:
        continuity: float
                   Continuity score between 0 and 1, where 1 indicates perfect
                   preservation of original neighborhoods.

    Usage:
        from sklearn.manifold import trustworthiness
        from sklearn.datasets import make_swiss_roll
        from sklearn.manifold import Isomap

        X, _ = make_swiss_roll(1000)
        embedding = Isomap(n_components=2).fit_transform(X)

        cont_score = continuity(X, embedding, n_neighbors=10)
        trust_score = trustworthiness(X, embedding, n_neighbors=10)
        print(f"Continuity: {cont_score:.3f}, Trustworthiness: {trust_score:.3f}")

    Reference:
        Venna & Kaski (2001). "Neighborhood Preservation in Nonlinear Projection Methods: An Experimental Study"
    """
    n_samples = X.shape[0]
    if n_neighbors >= n_samples / 2:
        raise ValueError(
            f"n_neighbors ({n_neighbors}) should be less than n_samples / 2"
            f" ({n_samples / 2})"
        )

    # Compute pairwise distances and nearest neighbors in input space
    dist_X = sl.metrics.pairwise_distances(X, metric=metric)
    if metric == "precomputed":
        dist_X = dist_X.copy()
    np.fill_diagonal(dist_X, np.inf)
    ind_X = np.argsort(dist_X, axis=1)[:, :n_neighbors]

    # Compute pairwise distances and nearest neighbors in output space
    dist_X_embedded = sl.metrics.pairwise_distances(X_embedded)
    np.fill_diagonal(dist_X_embedded, np.inf)
    ind_X_embedded = np.argsort(dist_X_embedded, axis=1)

    # Build inverted index for output space ranks
    inverted_index = np.zeros((n_samples, n_samples), dtype=int)
    inverted_index = np.argsort(np.argsort(dist_X_embedded, axis=1), axis=1)

    # Calculate ranks of input neighbors in output space and compute penalties
    ranks = inverted_index[np.arange(n_samples)[:, np.newaxis], ind_X] - n_neighbors
    c = np.sum(ranks[ranks > 0])
    c = 1.0 - c * (
        2.0 / (n_samples * n_neighbors * (2.0 * n_samples - 3.0 * n_neighbors - 1.0))
    )
    return float(np.clip(c, 0.0, 1.0))

In [ ]:
@keras.utils.register_keras_serializable()
class Alignment(keras.Metric):
  """
    Measures alignment: average squared L2 distance between positive pairs.
    Lower values indicate better alignment (positive pairs are closer together).

    This metric quantifies how well positive pairs (e.g., augmentations of the same image)
    are mapped to nearby points in the embedding space. It's computed as:

        alignment = E[||f(x) - f(x')||^2]

    where x and x' are positive pairs (e.g., two augmented views of the same sample).

    Args:
        axis: Axis along which to sum squared differences (default: 1).
                    Typically 1 for embeddings of shape (batch_size, embedding_dim).
        **kwargs: Additional keyword arguments passed to keras.Metric.

    Usage:
        alignment = Alignment()
        # During training:
        alignment(embeddings_1, embeddings_2)
        print(f"Alignment: {alignment.result():.4f}")

    Reference:
        Wang & Isola (2020). "Understanding Contrastive Representation Learning through Alignment and Uniformity on the Hypersphere"
        https://proceedings.mlr.press/v119/wang20k/wang20k.pdf
  """
  def __init__(
    self,
    l2_normalize: bool = True,
    **kwargs
  ):
    super().__init__(**kwargs)

    self.l2_normalize = l2_normalize

    self.alignment = self.add_variable(
        shape=(),
        initializer='zeros',
        name='alignment',
        dtype='float32'
     )

    self.total_count = self.add_variable(
        shape=(),
        initializer='zeros',
        name='total_count',
        dtype='float32'
    )

  def update_state(self, embeddings_1: keras.KerasTensor, embeddings_2: keras.KerasTensor) -> None:
    embeddings_1 = keras.ops.convert_to_tensor(embeddings_1, dtype=self.alignment.dtype)
    embeddings_2 = keras.ops.convert_to_tensor(embeddings_2, dtype=self.alignment.dtype)

    if self.l2_normalize:
      embeddings_1 = keras.utils.normalize(embeddings_1, axis=1, order=2)
      embeddings_2 = keras.utils.normalize(embeddings_2, axis=1, order=2)

    batch_size = keras.ops.cast(
        keras.ops.shape(embeddings_1)[0],
        dtype=self.alignment.dtype,
    )

    # Sum over features, then average over batch
    squared_diffs = keras.ops.sum(
        keras.ops.square(embeddings_1 - embeddings_2),
        axis=1
    )
    current_alignment = keras.ops.mean(squared_diffs)

    new_total = self.total_count + batch_size
    self.alignment.assign(
        (self.alignment * self.total_count + current_alignment * batch_size) / new_total
    )
    self.total_count.assign(new_total)

  def result(self) -> keras.KerasTensor:
    return self.alignment

  def reset_state(self) -> None:
    self.alignment.assign(0.0)
    self.total_count.assign(0.0)

def alignment(
  embeddings_1: keras.KerasTensor,
  embeddings_2: keras.KerasTensor,
  num_samples: Optional[int] = None,
  l2_normalize: bool = True,
) -> keras.KerasTensor:
  if num_samples is not None and num_samples < keras.ops.shape(embeddings_1)[0]:
    shuffled_indices = keras.random.shuffle(keras.ops.arange(test_embeddings.shape[0]))[:num_samples]
    embeddings_1 = embeddings_1[shuffled_indices]
    embeddings_2 = embeddings_2[shuffled_indices]

  return Alignment(l2_normalize)(embeddings_1, embeddings_2)

In [ ]:
@keras.utils.register_keras_serializable()
class Uniformity(keras.Metric):
  """
    Measures uniformity: how uniformly distributed embeddings are on the hypersphere.
    Lower (more negative) values indicate better uniformity (embeddings spread out evenly).

    This metric quantifies how evenly embeddings are distributed in the representation space.
    Good uniformity prevents dimensional collapse and ensures embeddings utilize the full
    capacity of the embedding space. It's computed as:

        uniformity = log(E[exp(-t * ||f(x) - f(x')||^2)])

    where the expectation is over all distinct pairs of samples, and t is a temperature
    parameter that controls sensitivity to distances.

    Args:
        t: Temperature parameter (default: 2.0). Higher values make the metric more
           sensitive to nearby pairs. The default of 2.0 follows the original paper.
        l2_normalize (default: True): Whether to apply L2 normalization the embeddings before calculation
        eps: Small constant for numerical stability (default: 1e-8).
        **kwargs: Additional keyword arguments passed to keras.Metric.

    Usage:
        uniformity = Uniformity(t=2.0)
        # Embeddings should be L2-normalized for proper hypersphere uniformity
        uniformity(normalized_embeddings)
        print(f"Uniformity: {uniformity.result():.4f}")

    Note:
        - Embeddings should be L2-normalized before computing uniformity
        - Memory usage scales with O(batch_size²) due to pairwise distances
        - For very large batches, consider sampling pairs instead

    Reference:
        Wang & Isola (2020). "Understanding Contrastive Representation Learning through Alignment and Uniformity on the Hypersphere"
        https://proceedings.mlr.press/v119/wang20k/wang20k.pdf
  """
  def __init__(
    self,
    t: float = 2.0,
    l2_normalize: bool = True,
    eps: float = 1e-8,
    **kwargs
  ):
    super().__init__(**kwargs)
    self.t = t
    self.l2_normalize = l2_normalize
    self.eps = eps

    self.uniformity = self.add_variable(
      shape=(),
      initializer='zeros',
      name='uniformity',
      dtype='float32'
    )

    self.total_count = self.add_variable(
      shape=(),
      initializer='zeros',
      name='total_count',
      dtype='float32'
    )

  def update_state(self, embeddings: keras.KerasTensor) -> None:
    embeddings = keras.ops.convert_to_tensor(embeddings, dtype=self.uniformity.dtype)

    batch_size = keras.ops.cast(
      keras.ops.shape(embeddings)[0],
      self.uniformity.dtype
    )

    if self.l2_normalize:
      embeddings = keras.utils.normalize(embeddings, axis=1, order=2)

    # Compute pairwise squared L2 distances: ||a - b||^2 = ||a||^2 + ||b||^2 - 2*<a,b>
    sq_norms = keras.ops.sum(keras.ops.square(embeddings), axis=1, keepdims=True)
    # Shape: (batch_size, batch_size)
    sq_dists = sq_norms + keras.ops.transpose(sq_norms) - 2 * keras.ops.matmul(
      embeddings, keras.ops.transpose(embeddings)
    )

    # Extract upper triangle (excluding diagonal) to avoid duplicate pairs and self-pairs, create mask for upper triangle
    n = keras.ops.shape(sq_dists)[0]
    indices = keras.ops.arange(n)
    i_indices = keras.ops.expand_dims(indices, 1)  # (n, 1)
    j_indices = keras.ops.expand_dims(indices, 0)  # (1, n)
    mask = keras.ops.cast(i_indices < j_indices, sq_dists.dtype)

    # Apply mask and get non-zero elements
    masked_dists = sq_dists * mask
    # Count valid pairs
    num_pairs = keras.ops.cast(n * (n - 1) / 2, self.uniformity.dtype)

    # Compute uniformity: log(mean(exp(-t * sq_dist)))
    exp_terms = keras.ops.exp(-self.t * masked_dists)
    # Sum only upper triangle elements
    sum_exp = keras.ops.sum(exp_terms * mask)
    current_uniformity = keras.ops.log(sum_exp / num_pairs)

    # Update running weighted average
    new_total = self.total_count + batch_size
    self.uniformity.assign(
      (self.uniformity * self.total_count + current_uniformity * batch_size) / new_total
    )
    self.total_count.assign(new_total)

  def result(self) -> keras.KerasTensor:
    return self.uniformity

  def reset_state(self) -> None:
    self.uniformity.assign(0.0)
    self.total_count.assign(0.0)

def uniformity(
    embeddings: keras.KerasTensor,
    t: float = 2.0,
    l2_normalize: bool = True,
    num_samples: Optional[int] = None,
  ) -> keras.KerasTensor:
  if num_samples is not None and num_samples < keras.ops.shape(embeddings)[0]:
    shuffled_embeddings = keras.random.shuffle(embeddings)
    embeddings = shuffled_embeddings[:num_samples]

  return Uniformity(t, l2_normalize)(embeddings)

In [ ]:
@keras.utils.register_keras_serializable()
class EffectiveRank(keras.Metric):
  """
    Measures the effective rank (intrinsic dimensionality) of embedding batches.
    Lower values indicate more compact representation, higher values indicate more dispersed usage of the available dimensions.

    This metric quantifies how many dimensions are effectively used by the embeddings
    by analyzing the singular value distribution. Two methods are available:

    1. Entropy-based: Interprets normalized singular values as a probability distribution
       and computes the exponential of the entropy. Measures information content.

        effective_rank = exp(-sum(p_i * log(p_i)))
        where p_i = s_i / sum(s) are normalized singular values

    2. Stable rank: Measures variance concentration using the ratio of squared L1 norm
       to L2 norm of singular values. More sensitive to dominant dimensions.

        stable_rank = (sum(s_i))^2 / sum(s_i^2)

    A significant divergence between methods indicates "spikey" embeddings where variance is
    concentrated in few dimensions but information is spread across many low-variance dimensions.

    Args:
        eps: Small constant for numerical stability (default: 1e-10).
        method: Calculation method - 'entropy' or 'stable' (default: 'entropy').
                'entropy': Measures information-theoretic intrinsic dimensionality.
                'stable': Measures variance concentration (numerical rank).
        **kwargs: Additional keyword arguments passed to keras.Metric.

    Usage:
        effective_rank = EffectiveRank(method='entropy')
        # During training:
        effective_rank(embeddings)
        print(f"Effective rank: {effective_rank.result():.4f}")

        # Compare methods:
        entropy_rank = EffectiveRank(method='entropy')(embeddings)
        stable_rank = EffectiveRank(method='stable')(embeddings)

    Reference:
        Roy & Vetterli (2007). "The effective rank: A measure of effective dimensionality"
        http://ieeexplore.ieee.org/document/7098875/
  """
  def __init__(
    self,
    eps: float = 1e-10,
    method: Literal['entropy', 'stable'] = 'entropy',
    **kwargs
  ):
    super().__init__(**kwargs)

    self.eps = eps
    self.method = method

    self.effective_rank = self.add_variable(
      shape=(),
      initializer='zeros',
      name='uniformity',
      dtype='float32'
    )

    self.total_count = self.add_variable(
      shape=(),
      initializer='zeros',
      name='total_count',
      dtype='float32'
    )

  def update_state(self, embeddings: keras.KerasTensor) -> None:
    embeddings = keras.ops.convert_to_tensor(embeddings, dtype=self.effective_rank.dtype)

    embeddings_centered = embeddings - keras.ops.mean(embeddings, axis=0)
    u, s, vh = keras.ops.svd(embeddings)

    if 'entropy' == self.method:
      # Entropy-based effective rank
      s_norm = s / (keras.ops.sum(s) + self.eps)
      entropy = -keras.ops.sum(s_norm * keras.ops.log(s_norm + self.eps))
      effective_rank = keras.ops.exp(entropy)
    elif 'stable' == self.method:
      # Stable-rank based effective rank
      sum_s = keras.ops.sum(s)
      sum_s_squared = keras.ops.sum(keras.ops.square(s))
      effective_rank = (keras.ops.square(sum_s)) / (sum_s_squared + self.eps)

    new_total = self.total_count + 1
    self.effective_rank.assign(
      (self.effective_rank * self.total_count + effective_rank) / new_total
    )

  def result(self) -> keras.KerasTensor:
    return self.effective_rank

  def reset_state(self) -> None:
    self.effective_rank.assign(0.0)
    self.total_count.assign(0.0)

def effective_rank(
  embeddings: keras.KerasTensor,
  method: str = 'entropy',
  num_samples: Optional[int] = None,
) -> keras.KerasTensor:

  if num_samples is not None and keras.greater(keras.ops.shape(embeddings)[0], num_samples):
    shuffled_embeddings = keras.random.shuffle(embeddings)
    embeddings = shuffled_embeddings[:num_samples]

  return EffectiveRank(method=method)(embeddings)

In [ ]:
@keras.saving.register_keras_serializable()
class MinMaxScaler(keras.src.layers.preprocessing.data_layer.DataLayer):
  """ Implementation of a MinMax scaler layer for Keras/Tensorflow preprocessing.

  The code is based on the Keras implementation of a norm layer (https://github.com/keras-team/keras/blob/v3.3.3/keras/src/layers/preprocessing/normalization.py) and the scikit-learn implementation (https://github.com/scikit-learn/scikit-learn/blob/6a0838c41/sklearn/preprocessing/_data.py#L288).

  Author:
      Marc C. Hennig (mhennig@hm.edu)
  """
  def __init__(
    self,
    axis: Optional[int] = -1,
    feature_range: Tuple[float, float] = (0, 1),
    min: Optional[float] = None,
    max: Optional[float] = None,
    **kwargs
  ):
    super().__init__(**kwargs)

    # Set `min` and `max` if passed.
    if feature_range is None or len(feature_range) != 2:
      raise ValueError(f"Feature range must be a tuple of length 2. Received feature_range={feature_range}")
    elif feature_range[0] >= feature_range[1]:
      raise ValueError(f"Feature range minimum must be less than the maximum. Received feature_range={feature_range}")
    else:
      feature_range_min, feature_range_max = feature_range

    # Standardize `axis` to a tuple.
    if axis is None:
      axis = ()
    elif isinstance(axis, int):
      axis = (axis,)
    else:
      axis = tuple(axis)

    self.axis = axis

    self.input_min = min
    self.input_max = max

    self.min = None
    self.max = None

    self.input_feature_range_min = feature_range_min
    self.input_feature_range_max = feature_range_max

    self.feature_range_min = None
    self.feature_range_max = None

    self.supports_masking = True
    self._build_input_shape = None

  def build(self, input_shape):
    if input_shape is None:
      return

    ndim = len(input_shape)
    self._build_input_shape = input_shape

    if any(a < -ndim or a >= ndim for a in self.axis):
      raise ValueError(f"All `axis` values must be in the range [-ndim, ndim). Received inputs with ndim={ndim}, while axis={self.axis}")

    # Axes to be kept, replacing negative values with positive equivalents.
    # Sorted to avoid transposing axes.
    self._keep_axis = tuple(sorted([d if d >= 0 else d + ndim for d in self.axis]))
    # All axes to be kept should have known shape.
    for d in self._keep_axis:
      if input_shape[d] is None:
        raise ValueError(f"All `axis` values to be kept must have a known shape. Received axis={self.axis}, inputs.shape={input_shape}, with unknown axis at index {d}")

    # Axes to be reduced.
    self._reduce_axis = tuple(d for d in range(ndim) if d not in self._keep_axis)
    # 1 if an axis should be reduced, 0 otherwise.
    self._reduce_axis_mask = [0 if d in self._keep_axis else 1 for d in range(ndim)]
    # Broadcast any reduced axes.
    self._broadcast_shape = [input_shape[d] if d in self._keep_axis else 1 for d in range(ndim)]
    min_max_shape = tuple(input_shape[d] for d in self._keep_axis)
    self.min_max_shape = min_max_shape

    self.feature_range_min = keras.ops.convert_to_tensor(self.input_feature_range_min)
    self.feature_range_max = keras.ops.convert_to_tensor(self.input_feature_range_max)

    if self.input_min is None:
      self.adapt_min = self.add_weight(
        name="min",
        shape=min_max_shape,
        initializer="zeros",
        trainable=False,
      )
      self.min = keras.ops.reshape(self.adapt_min, self._broadcast_shape)
    else:
      self.min = keras.ops.reshape(keras.ops.convert_to_tensor(self.input_min), self._broadcast_shape)

    if self.input_max is None:
      self.adapt_max = self.add_weight(
        name="max",
        shape=min_max_shape,
        initializer="zeros",
        trainable=False,
      )
      self.max = keras.ops.reshape(self.adapt_max, self._broadcast_shape)

    else:
      self.max = keras.ops.reshape(keras.ops.convert_to_tensor(self.input_max), self._broadcast_shape)

    self.min = keras.ops.cast(self.min, self.compute_dtype)
    self.max = keras.ops.cast(self.max, self.compute_dtype)

    self.built = True

  def adapt(self, data):
    if isinstance(data, np.ndarray) or keras.ops.is_tensor(data):
      input_shape = data.shape
    elif isinstance(data, tf.data.Dataset):
      input_shape = tuple(data.element_spec.shape)
      if len(input_shape) == 1:
        # Batch dataset if it isn't batched
        data = data.batch(128)
      input_shape = tuple(data.element_spec.shape)
    else:
      raise ValueError(f"Unsupported data type: {type(data)}")

    if not self.built:
      self.build(input_shape)
    else:
      for d in self._keep_axis:
        if input_shape[d] != self._build_input_shape[d]:
          raise ValueError(f"The layer was built with input_shape={self._build_input_shape}, but adapt() is being called with data with an incompatible shape, data.shape={input_shape}")

    if isinstance(data, np.ndarray):
      total_min = np.min(data, axis=self._reduce_axis)
      total_max = np.max(data, axis=self._reduce_axis)
    elif keras.ops.is_tensor(data):
      total_min = keras.ops.min(data, axis=self._reduce_axis)
      total_max = keras.ops.max(data, axis=self._reduce_axis)
    elif isinstance(data, tf.data.Dataset):
      total_min = keras.ops.full(self._broadcast_shape, data.element_spec.dtype.max)
      total_max = keras.ops.full(self._broadcast_shape, data.element_spec.dtype.min)
      for batch in data:
        batch = keras.ops.convert_to_tensor(batch, dtype=self.compute_dtype)

        batch_min = keras.ops.min(batch, axis=self._reduce_axis)
        batch_max = keras.ops.max(batch, axis=self._reduce_axis)

        total_min = keras.ops.minimum(total_min, batch_min)
        total_max = keras.ops.maximum(total_max, batch_max)
    else:
      raise ValueError(f"Unsupported data type: {type(data)}")

    self.adapt_min.assign(total_min)
    self.adapt_max.assign(total_max)
    self.finalize_state()

  def finalize_state(self):
    if self.input_min is not None or self.input_max is not None or not self.built:
      return

    # In the adapt case, we make constant tensors for min and max with
    # proper broadcast shape and dtype each time `finalize_state` is called.
    self.min = keras.ops.reshape(self.adapt_min, self._broadcast_shape)
    self.min = keras.ops.cast(self.min, self.compute_dtype)
    self.max = keras.ops.reshape(self.adapt_max, self._broadcast_shape)
    self.max = keras.ops.cast(self.max, self.compute_dtype)

  def call(self, inputs):
    if self.min is None or self.max is None:
      raise ValueError("You must call `.build(input_shape)` on the layer before using it.")

    inputs = self.backend.numpy.convert_to_tensor(inputs, dtype=self.compute_dtype)

    min = self.min
    max = self.max
    feature_range_min = self.feature_range_min
    feature_range_max = self.feature_range_max

    inputs_std = self.backend.numpy.divide(
     self.backend.numpy.subtract(inputs, min),
     self.backend.numpy.subtract(max, min)
    )

    return self.backend.numpy.add(
        self.backend.numpy.multiply(
          inputs_std,
          self.backend.numpy.subtract(feature_range_max, feature_range_min)
    ), feature_range_min)

  def compute_output_shape(self, input_shape):
    return input_shape

  def get_config(self):
    config = super().get_config()
    config.update({
      "axis": self.axis,
      "feature_range": (self.input_feature_range_min, self.input_feature_range_max),
      "min": np.array(self.input_min).tolist() if self.input_min is not None else None,
      "max": np.array(self.input_max).tolist() if self.input_max is not None else None,
    })
    return config

  def load_own_variables(self, store):
    super().load_own_variables(store)
    # Ensure that we call finalize_state after variable loading.
    self.finalize_state()

  def get_build_config(self):
    if self._build_input_shape:
      return {"input_shape": self._build_input_shape}

  def build_from_config(self, config):
    if config:
      self.build(config["input_shape"])

In [ ]:
@keras.saving.register_keras_serializable()
class MeanScaler(keras.src.layers.preprocessing.data_layer.DataLayer):
  """ Implementation of a Mean scaler layer for Keras/Tensorflow preprocessing.

  The code is based on the Keras implementation of a norm layer (https://github.com/keras-team/keras/blob/v3.3.3/keras/src/layers/preprocessing/normalization.py) and the scikit-learn implementation (https://github.com/scikit-learn/scikit-learn/blob/6a0838c41/sklearn/preprocessing/_data.py#L288).

  Author:
      Marc C. Hennig (mhennig@hm.edu)
  """
  def __init__(
    self,
    axis: Optional[int] = -1,
    mean: Optional[float] = None,
    **kwargs
  ):
    super().__init__(**kwargs)

    # Standardize `axis` to a tuple.
    if axis is None:
      axis = ()
    elif isinstance(axis, int):
      axis = (axis,)
    else:
      axis = tuple(axis)

    self.axis = axis

    self.input_mean = mean

    self.supports_masking = True
    self._build_input_shape = None

  def build(self, input_shape):
    if input_shape is None:
      return

    ndim = len(input_shape)
    self._build_input_shape = input_shape

    if any(a < -ndim or a >= ndim for a in self.axis):
      raise ValueError(f"All `axis` values must be in the range [-ndim, ndim). Received inputs with ndim={ndim}, while axis={self.axis}")

    # Axes to be kept, replacing negative values with positive equivalents.
    # Sorted to avoid transposing axes.
    self._keep_axis = tuple(sorted([d if d >= 0 else d + ndim for d in self.axis]))
    # All axes to be kept should have known shape.
    for d in self._keep_axis:
      if input_shape[d] is None:
        raise ValueError(f"All `axis` values to be kept must have a known shape. Received axis={self.axis}, inputs.shape={input_shape}, with unknown axis at index {d}")

    # Axes to be reduced.
    self._reduce_axis = tuple(d for d in range(ndim) if d not in self._keep_axis)
    # 1 if an axis should be reduced, 0 otherwise.
    self._reduce_axis_mask = [0 if d in self._keep_axis else 1 for d in range(ndim)]
    # Broadcast any reduced axes.
    self._broadcast_shape = [input_shape[d] if d in self._keep_axis else 1 for d in range(ndim)]
    min_max_shape = tuple(input_shape[d] for d in self._keep_axis)
    self.min_max_shape = min_max_shape

    if self.input_mean is None:
      self.adapt_mean = self.add_weight(
        name="mean",
        shape=min_max_shape,
        initializer="zeros",
        trainable=False,
      )
      self.mean = keras.ops.reshape(self.adapt_mean, self._broadcast_shape)
    else:
      self.mean = keras.ops.reshape(keras.ops.convert_to_tensor(self.input_mean), self._broadcast_shape)

    self.mean = keras.ops.cast(self.mean, self.compute_dtype)

  def adapt(self, data):
    if isinstance(data, np.ndarray) or keras.ops.is_tensor(data):
      input_shape = data.shape
    elif isinstance(data, tf.data.Dataset):
      input_shape = tuple(data.element_spec.shape)
      if len(input_shape) == 1:
        # Batch dataset if it isn't batched
        data = data.batch(128)
      input_shape = tuple(data.element_spec.shape)
    else:
      raise ValueError(f"Unsupported data type: {type(data)}")

    if not self.built:
      self.build(input_shape)
    else:
      for d in self._keep_axis:
        if input_shape[d] != self._build_input_shape[d]:
          raise ValueError(f"The layer was built with input_shape={self._build_input_shape}, but adapt() is being called with data with an incompatible shape, data.shape={input_shape}")

    if isinstance(data, np.ndarray):
      total_mean = np.mean(data, axis=self._reduce_axis)
    elif keras.ops.is_tensor(data):
      total_mean = keras.ops.mean(data, axis=self._reduce_axis)
    elif isinstance(data, tf.data.Dataset):
      total_mean = keras.ops.zeros(self._mean_and_var_shape)
      total_count = 0
      for batch in data:
        batch = keras.ops.convert_to_tensor(batch, dtype=self.compute_dtype)

        batch_mean = keras.ops.mean(batch, axis=self._reduce_axis)

        if self._reduce_axis:
          batch_reduce_shape = (batch.shape[d] for d in self._reduce_axis)
          batch_count = math.prod(batch_reduce_shape)
        else:
          batch_count = 1

        total_count += batch_count
        batch_weight = float(batch_count) / total_count
        existing_weight = 1.0 - batch_weight
        total_mean = total_mean * existing_weight + batch_mean * batch_weight
    else:
      raise ValueError(f"Unsupported data type: {type(data)}")


    self.adapt_mean.assign(total_mean)
    self.finalize_state()

  def finalize_state(self):
    if self.input_mean is not None or not self.built:
      return

    # In the adapt case, we make constant tensors for mean and variance with
    # proper broadcast shape and dtype each time `finalize_state` is called.
    self.mean = keras.ops.reshape(self.adapt_mean, self._broadcast_shape)
    self.mean = keras.ops.cast(self.mean, self.compute_dtype)

  def call(self, inputs):
    # This layer can be called in tf.data
    # even with another backend after it has been adapted.
    # However it must use backend-native logic for adapt().
    if self.mean is None:
      raise ValueError("You must call `.build(input_shape)` on the layer before using it.")

    inputs = self.backend.numpy.convert_to_tensor(inputs, dtype=self.compute_dtype)
    # Ensure the weights are in the correct backend. Without this, it is
    # possible to cause breakage when using this layer in tf.data.
    # mean = self.convert_weight(self.mean)
    mean = self.mean

    return self.backend.numpy.divide(
        inputs,
        mean
    )

  def compute_output_shape(self, input_shape):
    return input_shape

  def get_config(self):
    config = super().get_config()
    config.update(
      {
        "axis": self.axis,
        "mean": np.array(self.input_mean).tolist(),
      }
    )
    return config

  def load_own_variables(self, store):
    super().load_own_variables(store)
    # Ensure that we call finalize_state after variable loading.
    self.finalize_state()

  def get_build_config(self):
    if self._build_input_shape:
      return {"input_shape": self._build_input_shape}

  def build_from_config(self, config):
    if config:
      self.build(config["input_shape"])

In [ ]:
@keras.saving.register_keras_serializable()
class LogTransform(keras.src.layers.preprocessing.data_layer.DataLayer):
  """
  Log transformation preprocessing layer for Keras.

  Applies log(x + offset) transformation to handle skewed data.
  The offset ensures we can handle zeros.

  Args:
    eps: Small constant added before log to handle zeros (default: 1e-8)
  """

  def __init__(
    self,
    method: Union[Literal['log', 'log1p', 'log10', 'log2']] = 'log1p',
    eps: float = 1e-8,
    **kwargs
  ):
    kwargs.pop("trainable", None)
    super(LogTransform, self).__init__(trainable=False, **kwargs)

    self.method = method
    self.eps = eps

    self.log_func = self._build_transformer_func()

  def _build_transformer_func(self) -> Callable[[float], float]:
    if 'log' == self.method:
      return keras.ops.log
    elif 'log1p' == self.method:
      return keras.ops.log1p
    elif 'log10' == self.method:
      return keras.ops.log10
    elif 'log2' == self.method:
      return keras.ops.log2
    else:
      raise ValueError(f"Unsupported method: {self.method}")

  def build(self, input_shape: tuple) -> None:
    self.built = True

  def compute_output_shape(self, input_shape: tuple) -> tuple:
    return input_shape

  def get_config(self) -> dict:
    config = super(LogTransform, self).get_config()
    config.update({
      'method': self.method,
      'eps': self.eps
    })
    return config

  @classmethod
  def from_config(cls, config: dict) -> 'LogTransform':
    return cls(**config)

  def call(self, inputs: keras.KerasTensor) -> keras.KerasTensor:
    inputs = keras.ops.convert_to_tensor(inputs, dtype='float32')
    return self.log_func(inputs + self.eps)

def log_transform(inputs: keras.KerasTensor, method: str = 'log1p', eps: float = 1e-8) -> keras.KerasTensor:
  return LogTransform(method=method, eps=eps)(inputs)

### Embedding Layers

In [ ]:
@keras.saving.register_keras_serializable()
class StringEmbedding(keras.Layer):
  def __init__(
    self,
    embed_dim: int,
    vocab: List[str],
    embed_init: Union[str, keras.Initializer] = 'glorot_uniform',
    mask_zero: bool = True,
    num_oov_indices: int = 1,
    mask_token: str = "[PAD]",
    oov_token: str = "[UNK]",
    output_mode: str = 'int',
    pad_to_max_tokens: bool = False,
    name: Optional[str] = None,
  ):
    super().__init__(name=name)

    self.embed_dim = embed_dim
    self.vocab = vocab
    self.embed_init = embed_init
    self.mask_zero = mask_zero
    self.num_oov_indices = num_oov_indices
    self.mask_token = mask_token
    self.oov_token = oov_token
    self.output_mode = output_mode
    self.pad_to_max_tokens = pad_to_max_tokens

    self.lookup = keras.layers.StringLookup(
      num_oov_indices=self.num_oov_indices,
      mask_token=self.mask_token,
      oov_token=self.oov_token,
      output_mode=self.output_mode,
      pad_to_max_tokens=self.pad_to_max_tokens,
    )
    self.lookup.adapt(self.vocab)

    self.embedding = keras.layers.Embedding(
      input_dim=len(self.lookup.get_vocabulary()),
      output_dim=self.embed_dim,
      embeddings_initializer=self.embed_init,
      mask_zero=self.mask_zero,
    )

  def build(self, input_shape):
    self.built = True

  def get_config(self):
    base_config = super().get_config()
    config = {
      "embed_dim": self.embed_dim,
      "vocab": json.dumps(np.array(self.vocab).tolist()),
      "embed_init": self.embed_init,
      "mask_zero": self.mask_zero,
      "num_oov_indices": self.num_oov_indices,
      "mask_token": self.mask_token,
      "oov_token": self.oov_token,
      "output_mode": self.output_mode,
      "pad_to_max_tokens": self.pad_to_max_tokens,
      "name": self.name,
    }
    return {**base_config, **config}

  @classmethod
  def from_config(cls, config):
    embed_dim = config["embed_dim"]
    vocab = json.loads(config["vocab"])
    embed_init = config["embed_init"]
    mask_zero = config["mask_zero"]
    num_oov_indices = config["num_oov_indices"]
    mask_token = config["mask_token"]
    oov_token = config["oov_token"]
    output_mode = config["output_mode"]
    pad_to_max_tokens = config["pad_to_max_tokens"]
    name = config["name"]
    return cls(embed_dim, vocab, embed_init, mask_zero, num_oov_indices, mask_token, oov_token, output_mode, pad_to_max_tokens, name)

  def compute_mask(self, input, mask=None):
    input = self.lookup(input)
    return self.embedding.compute_mask(input, mask=mask)

  def call(self, inputs):
    x = self.lookup(inputs)
    x = self.embedding(x)
    return x

In [ ]:
@keras.saving.register_keras_serializable()
class NumericalEmbedding(keras.Layer):
  def __init__(
    self,
    embed_dim: int,
    vocab: List[float] = [],
    kernel_init: Union[str, keras.Initializer] = 'glorot_uniform',
    bias_init: Union[str, keras.Initializer] = 'zeros',
    mask_token: Optional[float] = -1.0,
    name: Optional[str] = None,
  ):
    super().__init__(name=name)

    self.embed_dim = embed_dim
    self.vocab = vocab
    self.kernel_init = kernel_init
    self.bias_init = bias_init
    self.mask_token = mask_token

    if self.vocab is not None and len(self.vocab) > 0:
      self.norm = keras.layers.Normalization()
      self.norm.adapt(self.vocab)
    else:
      self.norm = keras.layers.Identity()

    if self.mask_token is not None:
      self.masking = keras.layers.Masking(mask_value=self.mask_token)
    else:
      self.masking = keras.layers.Identity()

    self.dense = keras.layers.Dense(
      units=self.embed_dim,
      activation='linear',
      kernel_initializer=self.kernel_init,
      bias_initializer=self.bias_init,
    )

  def build(self, input_shape):
    self.built = True

  def get_config(self):
    base_config = super().get_config()
    config = {
      "embed_dim": self.embed_dim,
      "vocab": None if self.vocab is None else json.dumps(np.array(self.vocab, dtype='float32').tolist()),
      "kernel_init": self.kernel_init,
      "bias_init": self.bias_init,
      "mask_token": self.mask_token,
      "name": self.name,
    }
    return {**base_config, **config}

  @classmethod
  def from_config(cls, config):
    embed_dim = config["embed_dim"]
    vocab = None if config["vocab"] is None else json.loads(config["vocab"])
    kernel_init = config["kernel_init"]
    bias_init = config["bias_init"]
    mask_token = config["mask_token"]
    name = config["name"]
    return cls(embed_dim, vocab, kernel_init, bias_init, mask_token, name)

  def compute_mask(self, inputs, mask=None):
    x = keras.ops.expand_dims(inputs, axis=-1)
    x = self.norm(x)
    return self.masking.compute_mask(x, mask=mask)

  def call(self, inputs):
    x = keras.ops.expand_dims(inputs, axis=-1)
    x = self.norm(x)
    x = self.masking(x)
    return self.dense(x)

In [ ]:
@keras.saving.register_keras_serializable()
class TemporalEncoding(keras.layers.Layer):
  def __init__(
    self,
    embed_init: Union[str, keras.Initializer] = 'uniform',
    embed_reg: Optional[keras.Regularizer] = None,
    embed_con: Optional[keras.constraints.Constraint] = None,
  ):
    super().__init__()

    self.embed_init = embed_init
    self.embed_reg = embed_reg
    self.embed_con = embed_con

    self.month_vocab_len = 13
    self.yearday_vocab_len = 367
    self.day_vocab_len = 32
    self.weekday_vocab_len = 7
    self.hour_vocab_len = 24
    self.min_vocab_len = 60
    self.sec_vocab_len = 60

  def build(self, input_shape: List[Tuple[int, int]]):
    self.embed_dim = input_shape[-1]

    self.month_embedding = keras.layers.Embedding(
      input_dim=self.month_vocab_len,
      output_dim=self.embed_dim,
      embeddings_initializer=self.embed_init,
      embeddings_regularizer=self.embed_reg,
      embeddings_constraint=self.embed_con,
      mask_zero=False,
      name="month_embedding",
    )

    self.yearday_embedding = keras.layers.Embedding(
      input_dim=self.yearday_vocab_len,
      output_dim=self.embed_dim,
      embeddings_initializer=self.embed_init,
      embeddings_regularizer=self.embed_reg,
      embeddings_constraint=self.embed_con,
      mask_zero=False,
      name="dayinyear_embedding",
    )

    self.day_embedding = keras.layers.Embedding(
      input_dim=self.day_vocab_len,
      output_dim=self.embed_dim,
      embeddings_initializer=self.embed_init,
      embeddings_regularizer=self.embed_reg,
      embeddings_constraint=self.embed_con,
      mask_zero=False,
      name="dayinmonth_embedding",
    )

    self.weekday_embedding = keras.layers.Embedding(
      input_dim=self.weekday_vocab_len,
      output_dim=self.embed_dim,
      embeddings_initializer=self.embed_init,
      embeddings_regularizer=self.embed_reg,
      embeddings_constraint=self.embed_con,
      mask_zero=False,
      name="weekday_embedding",
    )

    self.hour_embedding = keras.layers.Embedding(
      input_dim=self.hour_vocab_len,
      output_dim=self.embed_dim,
      embeddings_initializer=self.embed_init,
      embeddings_regularizer=self.embed_reg,
      embeddings_constraint=self.embed_con,
      mask_zero=False,
      name="hour_embedding",
    )

    self.min_embedding = keras.layers.Embedding(
      input_dim=self.min_vocab_len,
      output_dim=self.embed_dim,
      embeddings_initializer=self.embed_init,
      embeddings_regularizer=self.embed_reg,
      embeddings_constraint=self.embed_con,
      mask_zero=False,
      name="minute_embedding",
    )

    self.sec_embedding = keras.layers.Embedding(
      input_dim=self.sec_vocab_len,
      output_dim=self.embed_dim,
      embeddings_initializer=self.embed_init,
      embeddings_regularizer=self.embed_reg,
      embeddings_constraint=self.embed_con,
      mask_zero=False,
      name="second_embedding",
    )

  def call(self, inputs, inputs_month=None, inputs_yearday=None, inputs_day=None, inputs_weekday=None, inputs_hour=None, inputs_minute=None, inputs_second=None):
    x_month = 0 if inputs_month is None else self.month_embedding(inputs_month)
    x_yearday = 0 if inputs_yearday is None else self.yearday_embedding(inputs_yearday)
    x_day = 0 if inputs_day is None else self.day_embedding(inputs_day)
    x_weekday = 0 if inputs_weekday is None else self.weekday_embedding(inputs_weekday)
    x_hour = 0 if inputs_hour is None else self.hour_embedding(inputs_hour)
    x_minute = 0 if inputs_minute is None else self.minute_embedding(inputs_minute)
    x_second = 0 if inputs_second is None else self.sec_embedding(inputs_second)

    return x_month + x_yearday + x_day + x_weekday + x_hour + x_minute + x_second

## RNN

### ProcessLSTM

In [ ]:
class BaseProcessRNN(keras.Model, LayerNameMixin, BaseModelMixin):
  """A base RNN model for process mining tasks with shared and task-specific layers. Provides a structured and more flexible reimplementation of the work by Tax et al. 2017 (https://github.com/verenich/ProcessSequencePrediction).

  The default parameters represent the original model architecture with its hyperparameters.

  This model implements a neural architecture for process mining that combines:
  - Multiple input features (activities, temporal features)
  - Shared RNN layers for common feature extraction
  - Task-specific RNN layers for different prediction tasks
  - Configurable normalization and feed-forward layers

  Args:
      max_case_len (int): Maximum length of process cases/sequences
      activity_vocab (List[str]): Vocabulary of possible process activities
      output_dict (Dict[str, dict]): Configuration for output layers. Format: {"task_name": {"units": int, "activation": str}}, Example: {"remaining_time": {'units': 1, 'activation': 'relu'}, "next_activity": {'units': 10, 'activation': 'softmax'}}
      activity_encoding_type (str, optional): Type of activity encoding. Options: "embedding" or "one_hot". Defaults to "one_hot"
      activity_embedding_dim (int): Dimension for activity embeddings. Only used if activity_encoding_type is "embedding". Defaults to 16
      time_encoding_type (str, optional): Type of time feature encoding. Options: "mean", "minmax", or "normal". Defaults to "mean"
      interval_since_last_event_vocab (List[float], optional): Vocabulary for time intervals
      hour_of_day_vocab (List[float], optional): Vocabulary for hour features
      day_of_week_vocab (List[float], optional): Vocabulary for day of week features
      rnn_units_shared (List[int]): Units in each shared RNN layer. Defaults to [100]
      rnn_units_nonshared (List[int]): Units in each task-specific RNN layer. Defaults to [100]
      rnn_type (str): Type of RNN cell. Options: "lstm", "gru", "rnn". Defaults to "lstm"
      rnn_bidirectional (bool): Whether to use bidirectional RNNs. Defaults to False
      rnn_activation (str): Activation function for RNN cells. Defaults to "tanh"
      rnn_recurrent_activation (str): Recurrent activation for LSTM/GRU. Defaults to "sigmoid"
      rnn_go_backwards (bool): Whether to process sequence backwards. Defaults to False
      rnn_unroll (bool): Whether to unroll RNN computation. Defaults to False
      rnn_recurrent_dropout (float): Dropout rate for recurrent connections. Defaults to 0.0
      rnn_dropout (float): Dropout rate for RNN inputs. Defaults to 0.0
      ff_dims (List[int]): Units in each feed-forward layer. Defaults to []
      ff_activation (str): Activation for feed-forward layers. Defaults to "relu"
      ff_dropout (float): Dropout rate for feed-forward layers. Defaults to 0.0
      norm_type (str, optional): Type of normalization. Options: "layer" or "batch". Defaults to "batch"
      norm_epsilon (float): Small constant for numerical stability. Defaults to 0.001
      norm_center (bool): Whether to subtract mean in normalization. Defaults to True
      norm_scale (bool): Whether to multiply by gamma in normalization. Defaults to True
      numeric_mask_token (float): Token used for masking numeric values. Defaults to TOKEN_PAD_NUM
      mask_token (str): Token used for masking categorical values. Defaults to TOKEN_PAD
      oov_token (str): Token used for out-of-vocabulary values. Defaults to TOKEN_OOV

  Author:
      Marc C. Hennig (mhennig@hm.edu)
  """
  def __init__(
    self,
    max_case_len: int,
    activity_vocab: List[str],
    output_dict: Dict[str, dict],
    activity_encoding_type: Optional[Literal['embedding', 'one_hot']] = 'one_hot',
    activity_embedding_dim: int = 16,
    time_encoding_type: Optional[Literal['mean','minmax', 'normal']] = 'mean',
    interval_since_last_event_vocab: Optional[List[float]] = None,
    hour_of_day_vocab: Optional[List[float]] = None,
    day_of_week_vocab: Optional[List[float]] = None,
    rnn_units_shared: List[int] = [100],
    rnn_units_nonshared: List[int] = [100],
    rnn_type: Literal['lstm', 'gru', 'rnn'] = 'lstm',
    rnn_bidirectional: bool = False,
    rnn_activation: str = 'tanh',
    rnn_recurrent_activation: str = 'sigmoid',
    rnn_go_backwards: bool = False,
    rnn_unroll: bool = False,
    rnn_recurrent_dropout: float = 0.0,
    rnn_dropout: float = 0.0,

    ff_dims: List[int] = [],
    ff_activation: str = 'relu',
    ff_dropout: float = 0.0,

    norm_type: Optional[Literal['layer', 'batch']] = 'batch',
    norm_epsilon: float = 0.001,
    norm_center: bool = True,
    norm_scale: bool = True,

    numeric_mask_token: float = TOKEN_PAD_NUM,
    mask_token: str = TOKEN_PAD,
    oov_token: str = TOKEN_OOV,
    **kwargs
  ):
    super().__init__(**kwargs)

    self.max_case_len = max_case_len
    self.output_dict = output_dict
    self.activity_vocab = np.unique(activity_vocab).tolist()
    self.activity_encoding_type = activity_encoding_type
    self.activity_embedding_dim = activity_embedding_dim

    self.rnn_units_shared = rnn_units_shared
    self.rnn_units_nonshared = rnn_units_nonshared
    self.rnn_type = rnn_type

    self.rnn_activation = rnn_activation
    self.rnn_recurrent_activation = rnn_recurrent_activation
    self.rnn_bidirectional = rnn_bidirectional
    self.rnn_go_backwards = rnn_go_backwards
    self.rnn_unroll = rnn_unroll
    self.rnn_recurrent_dropout = rnn_recurrent_dropout
    self.rnn_dropout = rnn_dropout

    self.norm_type = norm_type
    self.norm_epsilon = norm_epsilon
    self.norm_center = norm_center
    self.norm_scale = norm_scale

    self.ff_dims = ff_dims
    self.ff_activation = ff_activation
    self.ff_dropout = ff_dropout

    self.numeric_mask_token = numeric_mask_token
    self.mask_token = mask_token
    self.oov_token = oov_token

    self.activity_encoding = self._build_activity_encoding_layer()

    self.time_encoding_type = time_encoding_type
    self.interval_since_last_event_encoding = self._build_time_encoding_layer(vocab=interval_since_last_event_vocab, name="interval_since_last_event")
    self.hour_of_day_encoding = self._build_time_encoding_layer(vocab=hour_of_day_vocab, name="hour_of_day")
    self.day_of_week_encoding = self._build_time_encoding_layer(vocab=day_of_week_vocab, name="day_of_week")

    self.shared_rnns = self._build_hard_shared_rnn_layer()

    self.non_shared_rnns = []
    self.ffns = []
    self.outputs = []
    for output_name, output_config in self.output_dict.items():
      self.non_shared_rnns.append(self._build_soft_shared_rnn_layer(name=output_name))
      self.ffns.append(self._build_ffn_layer(name=output_name))
      self.outputs.append(keras.layers.Dense(units=output_config['units'], activation=output_config['activation'], name=output_name))

  @property
  def default_loss(self) -> Dict[str, keras.Loss]:
    return {k: self.find_default_loss(v['activation']) for k, v in self.output_dict.items()}

  @property
  def default_metrics(self) -> Dict[str, List[keras.Metric]]:
    return {k: self.find_default_metrics(v['activation']) for k, v in self.output_dict.items()}

  @property
  def default_learning_rate(self) -> Union[float, keras.optimizers.schedules.LearningRateSchedule]:
    return 0.002

  @property
  def default_optimizer(self) -> keras.optimizers.Optimizer:
    return keras.optimizers.Nadam(learning_rate=self.default_learning_rate, beta_1=0.9, beta_2=0.999, epsilon=1e-08, weight_decay=0.004, clipvalue=3)

  @property
  def default_regression_loss(self) -> keras.Loss:
    return keras.losses.MeanAbsoluteError()

  def _build_time_encoding_layer(self, vocab: Optional[List[float]] = None, name: Optional[str] = None, **kwargs) -> keras.layers.Layer:
    """Builds a time encoding layer based on the specified encoding type.

    Args:
        vocab (List[float], optional): Vocabulary of time values for adaptation
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to the layers

    Returns:
        keras.layers.Layer: Sequential layer for time feature encoding

    Raises:
        NotImplementedError: If time_encoding_type is not one of [None, "mean", "minmax", "normal"]
    """
    name = self._generate_layer_name(f"time_encoding_{self.time_encoding_type}", name)
    layer = keras.models.Sequential([keras.layers.Masking(self.numeric_mask_token)], name=name)

    if self.time_encoding_type == None:
      layer.add(keras.layers.Identity(name=name))
    elif 'mean' == self.time_encoding_type:
      lookup = MeanScaler(axis=None)
      lookup.adapt(vocab)
      layer.add(lookup)
    elif 'minmax' == self.time_encoding_type:
      lookup = MinMaxScaler(axis=None)
      lookup.adapt(vocab)
      layer.add(lookup)
    elif 'normal' == self.time_encoding_type:
      lookup = keras.layers.Normalization(axis=None)
      lookup.adapt(vocab)
      layer.add(lookup)
    else:
      raise NotImplementedError(f"Unknown time encoding type {self.time_encoding_type}")

    layer.add(keras.layers.Reshape((-1, 1)))

    return layer

  def _build_activity_encoding_layer(self, name: Optional[str] = None, **kwargs) -> keras.layers.Layer:
    """Builds an activity encoding layer based on the specified encoding type.

    Args:
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to the layers

    Returns:
        keras.layers.Layer: Layer for activity encoding (Identity, OneHot, or Embedding)

    Raises:
        NotImplementedError: If activity_encoding_type is not one of [None, "one_hot", "embedding"]
    """
    name = self._generate_layer_name(f"activity_encoding_{self.activity_encoding_type}", name)

    if self.activity_encoding_type is None:
      layer = keras.layers.Identity(name=name)
    elif 'one_hot' == self.activity_encoding_type:
      lookup = keras.layers.StringLookup(
        mask_token=self.mask_token,
        oov_token=self.oov_token,
        output_mode='one_hot',
        pad_to_max_tokens=False,
      )
      lookup.adapt(self.activity_vocab)

      # Workaround: https://github.com/keras-team/keras/issues/19191#issuecomment-2077711036
      #lookup.build()
      layer = keras.models.Sequential([
        keras.layers.Reshape((-1, 1), dtype='string'),
        keras.layers.TimeDistributed(lookup)
      ], name=name)

    elif 'embedding' == self.activity_encoding_type:
      lookup = keras.layers.StringLookup(
        mask_token=self.mask_token,
        oov_token=self.oov_token,
        output_mode='int',
        pad_to_max_tokens=False,
      )
      lookup.adapt(self.activity_vocab)

      layer = keras.models.Sequential([
        lookup,
        keras.layers.Embedding(
          input_dim=len(lookup.get_vocabulary()),
          output_dim=self.activity_embedding_dim,
          mask_zero=True,
        )
      ], name=name)
    else:
      raise NotImplementedError(f"Unknown activity encoding type {self.activity_encoding_type}")

    return layer

  def _build_hard_shared_rnn_layer(self, name: Optional[str] = None, **kwargs) -> keras.Layer:
    """Builds the shared RNN layers used across all tasks (hard parameter sharing).

    Args:
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to the layers

    Returns:
        keras.Layer: Sequential model containing shared RNN layers with normalization
    """
    name = self._generate_layer_name(f"hardsharing_{self.rnn_type}", name)
    layer = keras.models.Sequential(name=name)

    for units in self.rnn_units_shared:
      layer.add(self._build_rnn_layer(units, return_sequences=True))
      layer.add(self._build_norm_layer())

    return layer

  def _build_soft_shared_rnn_layer(self, name: Optional[str] = None, **kwargs) -> keras.Layer:
    """Builds task-specific RNN layers (soft parameter sharing).

    Args:
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to the layers

    Returns:
        keras.Layer: Sequential model containing task-specific RNN layers with normalization
    """
    name = self._generate_layer_name(f"softsharing_{self.rnn_type}", name)
    layer = keras.models.Sequential(name=name)

    for i, units in enumerate(self.rnn_units_nonshared):
      return_sequences = False if i == len(self.rnn_units_nonshared) - 1 else True

      layer.add(self._build_rnn_layer(units, return_sequences=return_sequences))
      layer.add(self._build_norm_layer())

    return layer

  def _build_rnn_layer(self, units: int, **kwargs) -> keras.Layer:
    """Builds a single RNN layer based on the specified RNN type.

    Args:
        units (int): Number of output units
        **kwargs: Additional keyword arguments passed to the RNN layer

    Returns:
        keras.Layer: RNN layer (LSTM, SimpleRNN, or GRU)

    Raises:
        NotImplementedError: If rnn_type is not one of ["lstm", "rnn", "gru"]
    """
    if 'lstm' == self.rnn_type:
      layer = keras.layers.LSTM(
        units=units,
        activation=self.rnn_activation,
        recurrent_activation=self.rnn_recurrent_activation,
        dropout=self.rnn_dropout,
        recurrent_dropout=self.rnn_recurrent_dropout,
        unroll=self.rnn_unroll,
        go_backwards=self.rnn_go_backwards,
        seed=RANDOM_STATE,
        **kwargs
      )
    elif 'rnn' == self.rnn_type:
      layer = keras.layers.SimpleRNN(
        units=units,
        activation=self.rnn_activation,
        dropout=self.rnn_dropout,
        recurrent_dropout=self.rnn_recurrent_dropout,
        unroll=self.rnn_unroll,
        go_backwards=self.rnn_go_backwards,
        seed=RANDOM_STATE,
        **kwargs
      )
    elif 'gru' == self.rnn_type:
      layer = keras.layers.GRU(
        units=units,
        activation=self.rnn_activation,
        recurrent_activation=self.rnn_recurrent_activation,
        dropout=self.rnn_dropout,
        recurrent_dropout=self.rnn_recurrent_dropout,
        unroll=self.rnn_unroll,
        go_backwards=self.rnn_go_backwards,
        seed=RANDOM_STATE,
        **kwargs
      )
    else:
      raise NotImplementedError(f"Unknown RNN type {self.rnn_type}")

    if self.rnn_bidirectional:
      layer = keras.layers.Bidirectional(layer)

    return layer

  def _build_norm_layer(self, **kwargs) -> keras.Layer:
    """Builds a normalization layer based on the specified normalization type.

    Args:
        **kwargs: Additional keyword arguments passed to the normalization layer

    Returns:
        keras.Layer: Normalization layer (BatchNorm, LayerNorm, or Identity)

    Raises:
        NotImplementedError: If norm_type is not one of [None, "batch", "layer"]
    """
    if self.norm_type is None:
      layer = keras.layers.Identity()
    elif self.norm_type == 'batch':
      layer = keras.layers.BatchNormalization(
        epsilon=self.norm_epsilon,
        center=self.norm_center,
        scale=self.norm_scale,
        **kwargs
      )
    elif self.norm_type == 'layer':
      layer = keras.layers.LayerNormalization(
        epsilon=self.norm_epsilon,
        center=self.norm_center,
        scale=self.norm_scale,
        **kwargs
      )
    else:
      raise NotImplementedError(f"Unknown normalization type {self.norm_type}")

    return layer

  def _build_ffn_layer(self, name: Optional[str] = None, **kwargs) -> keras.layers.Layer:
    """Builds feed-forward layers with optional dropout.

    Args:
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to the layers

    Returns:
        keras.layers.Layer: Sequential model containing feed-forward layers
    """
    name = self._generate_layer_name("ffn", name)
    layer = keras.models.Sequential([keras.layers.Identity()], name=name)

    for ff_dim in self.ff_dims:
      layer.add(keras.layers.Dense(units=ff_dim, activation=self.ff_activation))
      layer.add(keras.layers.Dropout(self.ff_dropout))

    return layer

  def build_graph(self) -> keras.Model:
    """Builds and returns a Keras Model instance with the defined architecture.

    Returns:
        keras.Model: Compiled model with defined inputs and outputs
    """
    in_activity = keras.layers.Input(shape=(self.max_case_len,), name="activity", dtype='string')
    in_interval_since_last_event = keras.layers.Input(shape=(self.max_case_len,), name="time_timestamp_elapsedprev", dtype='float32')
    in_hour_of_day = keras.layers.Input(shape=(self.max_case_len,), name="time_timestamp_hour_raw", dtype='float32')
    in_day_of_week = keras.layers.Input(shape=(self.max_case_len,), name="time_timestamp_weekday_raw", dtype='float32')

    return keras.Model(inputs=[in_activity, in_interval_since_last_event, in_hour_of_day, in_day_of_week], outputs=self.call(in_activity, in_interval_since_last_event, in_hour_of_day, in_day_of_week), name=self.name)

  def call(self, inputs, inputs_interval_since_last_event, inputs_hour_of_day, inputs_day_of_week, training=False):
    """Forward pass of the model.

    Args:
        inputs: Activity input tensor
        inputs_interval_since_last_event: Time interval input tensor
        inputs_hour_of_day: Hour of day input tensor
        inputs_day_of_week: Day of week input tensor
        training (bool, optional): Whether in training mode. Defaults to False

    Returns:
        List[tf.Tensor]: List of output tensors for each task
    """
    x_activity = self.activity_encoding(inputs)
    x_interval = self.interval_since_last_event_encoding(inputs_interval_since_last_event)
    x_hour = self.hour_of_day_encoding(inputs_hour_of_day)
    x_day = self.day_of_week_encoding(inputs_day_of_week)

    x = keras.layers.concatenate([x_activity, x_interval, x_hour, x_day])

    x = self.shared_rnns(x)

    x_outputs = []
    for rnn, ffn, out in zip(self.non_shared_rnns, self.ffns, self.outputs, strict=True):
      x_out = rnn(x)
      x_out = ffn(x_out)
      x_out = out(x_out)
      x_outputs.append(x_out)

    return x_outputs


In [ ]:
lstm = BaseProcessRNN(
    max_case_len=3,
    activity_vocab=['a', 'b', 'c'],
    interval_since_last_event_vocab=np.array([1,2,3]),
    hour_of_day_vocab=np.array([1,2,3]),
    day_of_week_vocab=np.array([1,2,3]),
    output_dict={"remaining_time": {'units': 1, 'activation': 'relu'}, "next_activity": {'units': 10, 'activation': 'softmax'}},
    activity_encoding_type='one_hot'
)

lstm = lstm.build_graph()
lstm.summary()

In [ ]:
keras.utils.plot_model(
  lstm,
  show_shapes=True,
  show_dtype=True,
  show_layer_names=True,
  expand_nested=True,
  show_layer_activations=True,
  show_trainable=True,
)


### DALSTM

In [ ]:
class BaseDARNN(keras.Model, LayerNameMixin, BaseModelMixin):
  def __init__(
    self,
    max_case_len: int,
    activity_vocab: List[str],
    output_dict: Dict[str, dict],

    activity_encoding_type: Optional[Literal['embedding', 'one_hot']] = 'one_hot',
    activity_embedding_dim: int = 16,
    categorical_attrs: Dict[str, List[str]] = {},
    categorical_encoding_type: Optional[Literal['embedding', 'one_hot', 'hashing']] = 'hashing',
    categorical_embedding_dim: int = 16,
    numerical_attrs: Dict[str, Optional[List[float]]] = {},
    numerical_encoding_type: Optional[Literal['mean', 'minmax', 'normal']] = None,

    rnn_units: List[int] = [200, 200, 200, 200],
    rnn_type: Literal['lstm', 'gru', 'rnn'] = 'lstm',
    rnn_bidirectional: bool = False,
    rnn_activation: str = 'tanh',
    rnn_recurrent_activation: str = 'sigmoid',
    rnn_go_backwards: bool = False,
    rnn_unroll: bool = False,
    rnn_recurrent_dropout: float = 0.0,
    rnn_dropout: float = 0.0,

    ff_dims: List[int] = [400, 200, 100, 50],
    ff_activation: str = 'relu',
    ff_dropout: float = 0.0,

    norm_type: Optional[Literal['layer', 'batch']] = None,
    norm_epsilon: float = 0.001,
    norm_center: bool = True,
    norm_scale: bool = True,

    numeric_mask_token: float = TOKEN_PAD_NUM,
    mask_token: str = TOKEN_PAD,
    oov_token: str = TOKEN_OOV,
    **kwargs
  ):
    super().__init__(**kwargs)

    self.max_case_len = max_case_len
    self.output_dict = output_dict
    self.activity_vocab = np.unique(activity_vocab).tolist()
    self.activity_encoding_type = activity_encoding_type
    self.activity_embedding_dim = activity_embedding_dim

    self.categorical_attrs = categorical_attrs
    self.categorical_encoding_type = categorical_encoding_type
    self.categorical_embedding_dim = categorical_embedding_dim
    self.numerical_attrs = numerical_attrs
    self.numerical_encoding_type = numerical_encoding_type

    self.rnn_units = rnn_units
    self.rnn_type = rnn_type

    self.rnn_activation = rnn_activation
    self.rnn_recurrent_activation = rnn_recurrent_activation
    self.rnn_bidirectional = rnn_bidirectional
    self.rnn_go_backwards = rnn_go_backwards
    self.rnn_unroll = rnn_unroll
    self.rnn_recurrent_dropout = rnn_recurrent_dropout
    self.rnn_dropout = rnn_dropout

    self.norm_type = norm_type
    self.norm_epsilon = norm_epsilon
    self.norm_center = norm_center
    self.norm_scale = norm_scale

    self.ff_dims = ff_dims
    self.ff_activation = ff_activation
    self.ff_dropout = ff_dropout

    self.numeric_mask_token = numeric_mask_token
    self.mask_token = mask_token
    self.oov_token = oov_token

    self.activity_encoding = self._build_activity_encoding_layer()
    self.categorical_encodings = {k: self._build_categorical_encoding_layer(v, k) for k, v in self.categorical_attrs.items()}
    self.numerical_encodings = {k: self._build_numerical_encoding_layer(v, k) for k, v in self.numerical_attrs.items()}

    self.rnn = self._build_rnn()

    self.ffns = {k: self._build_ffn_layer(name=k) for k in self.output_dict.keys()}
    self.outputs = {k: keras.layers.Dense(units=v['units'], activation=v['activation'], name=k) for k, v in self.output_dict.items()}

  @property
  def default_loss(self) -> Dict[str, keras.Loss]:
    return {k: self.find_default_loss(v['activation']) for k, v in self.output_dict.items()}

  @property
  def default_metrics(self) -> Dict[str, List[keras.Metric]]:
    return {k: self.find_default_metrics(v['activation']) for k, v in self.output_dict.items()}

  @property
  def default_learning_rate(self) -> Union[float, keras.optimizers.schedules.LearningRateSchedule]:
    return 0.0002

  def default_callbacks(
    self,
    log_file: Optional[str] = None,
    tensorboard_dir: Optional[str] = None,
    backup_dir: Optional[str] = None,
    checkpoint_file: Optional[str] = None,
    **kwargs
  ) -> List[keras.callbacks.Callback]:
    return super().default_callbacks(log_file, tensorboard_dir, backup_dir, checkpoint_file) + [keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1, min_lr=DEFAULT_MIN_LEARNING_RATE)]

  @property
  def default_optimizer(self) -> keras.optimizers.Optimizer:
    return keras.optimizers.Nadam(learning_rate=self.default_learning_rate, beta_1=0.9, beta_2=0.999, epsilon=1e-08, weight_decay=0.006, clipvalue=1)

  @property
  def default_regression_loss(self) -> keras.Loss:
    return keras.losses.MeanAbsoluteError()

  def _build_numerical_encoding_layer(self, vocab: Optional[List[float]] = None, name: Optional[str] = None, **kwargs) -> keras.layers.Layer:
    """Builds a time encoding layer based on the specified encoding type.

    Args:
        vocab (List[float], optional): Vocabulary of time values for adaptation
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to the layers

    Returns:
        keras.layers.Layer: Sequential layer for time feature encoding

    Raises:
        NotImplementedError: If time_encoding_type is not one of [None, "mean", "minmax", "normal"]
    """
    name = self._generate_layer_name(f"num_encoding_{self.numerical_encoding_type}", name)
    layer = keras.models.Sequential([keras.layers.Masking(self.numeric_mask_token)], name=name)

    if self.numerical_encoding_type == None:
      layer.add(keras.layers.Identity(name=name))
    elif 'mean' == self.numerical_encoding_type:
      lookup = MeanScaler(axis=None)
      lookup.adapt(vocab)
      layer.add(lookup)
    elif 'minmax' == self.numerical_encoding_type:
      lookup = MinMaxScaler(axis=None)
      lookup.adapt(vocab)
      layer.add(lookup)
    elif 'normal' == self.numerical_encoding_type:
      lookup = keras.layers.Normalization(axis=None)
      lookup.adapt(vocab)
      layer.add(lookup)
    else:
      raise NotImplementedError(f"Unknown time encoding type {self.numerical_encoding_type}")

    layer.add(keras.layers.Reshape((-1, 1)))

    return layer

  def _build_activity_encoding_layer(self, name: Optional[str] = None, **kwargs) -> keras.layers.Layer:
    """Builds an activity encoding layer based on the specified encoding type.

    Args:
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to the layers

    Returns:
        keras.layers.Layer: Layer for activity encoding (Identity, OneHot, or Embedding)

    Raises:
        NotImplementedError: If activity_encoding_type is not one of [None, "one_hot", "embedding"]
    """
    name = self._generate_layer_name(f"activity_encoding_{self.activity_encoding_type}", name)

    if self.activity_encoding_type is None:
      layer = keras.layers.Identity(name=name)
    elif 'one_hot' == self.activity_encoding_type:
      lookup = keras.layers.StringLookup(
        mask_token=self.mask_token,
        oov_token=self.oov_token,
        output_mode='one_hot',
        pad_to_max_tokens=False,
      )
      lookup.adapt(self.activity_vocab)

      # Workaround: https://github.com/keras-team/keras/issues/19191#issuecomment-2077711036
      #lookup.build()
      layer = keras.models.Sequential([
        keras.layers.Reshape((-1, 1), dtype='string'),
        keras.layers.TimeDistributed(lookup)
      ], name=name)

    elif 'embedding' == self.activity_encoding_type:
      lookup = keras.layers.StringLookup(
        mask_token=self.mask_token,
        oov_token=self.oov_token,
        output_mode='int',
        pad_to_max_tokens=False,
      )
      lookup.adapt(self.activity_vocab)

      layer = keras.models.Sequential([
        lookup,
        keras.layers.Embedding(
          input_dim=len(lookup.get_vocabulary()),
          output_dim=self.activity_embedding_dim,
          mask_zero=True,
        )
      ], name=name)
    else:
      raise NotImplementedError(f"Unknown activity encoding type {self.activity_encoding_type}")

    return layer

  def _build_categorical_encoding_layer(self, vocab: List[str], name: str, **kwargs) -> keras.layers.Layer:
    """Builds an activity encoding layer based on the specified encoding type.

    Args:
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to the layers

    Returns:
        keras.layers.Layer: Layer for activity encoding (Identity, OneHot, or Embedding)

    Raises:
        NotImplementedError: If activity_encoding_type is not one of [None, "one_hot", "embedding"]
    """
    name = self._generate_layer_name(f"cat_encoding_{self.categorical_encoding_type}", name)

    if self.categorical_encoding_type is None:
      layer = keras.layers.Identity(name=name)
    elif 'hashing' == self.categorical_encoding_type:
      lookup = keras.layers.Hashing(
        mask_value=self.mask_token,
        num_bins=len(vocab),
        output_mode='one_hot',
      )

      layer = keras.models.Sequential([
        keras.layers.Reshape((-1, 1), dtype='string'),
        lookup,
      ], name=name)
    elif 'one_hot' == self.categorical_encoding_type:
      lookup = keras.layers.StringLookup(
        mask_token=self.mask_token,
        oov_token=self.oov_token,
        output_mode='one_hot',
        pad_to_max_tokens=False,
      )
      lookup.adapt(vocab)

      # Workaround: https://github.com/keras-team/keras/issues/19191#issuecomment-2077711036
      #lookup.build()
      layer = keras.models.Sequential([
        keras.layers.Reshape((-1, 1), dtype='string'),
        keras.layers.TimeDistributed(lookup)
      ], name=name)

    elif 'embedding' == self.categorical_encoding_type:
      lookup = keras.layers.StringLookup(
        mask_token=self.mask_token,
        oov_token=self.oov_token,
        output_mode='int',
        pad_to_max_tokens=False,
      )
      lookup.adapt(vocab)

      layer = keras.models.Sequential([
        lookup,
        keras.layers.Embedding(
          input_dim=len(lookup.get_vocabulary()),
          output_dim=self.categorical_embedding_dim,
          mask_zero=True,
        )
      ], name=name)
    else:
      raise NotImplementedError(f"Unknown activity encoding type {self.categorical_encoding_type}")

    return layer

  def _build_rnn(self, name: Optional[str] = None, **kwargs) -> keras.Layer:
    name = self._generate_layer_name(f"{self.rnn_type}", name)
    layer = keras.models.Sequential(name=name)

    for units in self.rnn_units[:-1]:
      layer.add(self._build_rnn_layer(units, return_sequences=True))
      layer.add(self._build_norm_layer())

    layer.add(self._build_rnn_layer(self.rnn_units[-1], return_sequences=False))
    layer.add(self._build_norm_layer())

    return layer

  def _build_rnn_layer(self, units: int, **kwargs) -> keras.Layer:
    """Builds a single RNN layer based on the specified RNN type.

    Args:
        units (int): Number of output units
        **kwargs: Additional keyword arguments passed to the RNN layer

    Returns:
        keras.Layer: RNN layer (LSTM, SimpleRNN, or GRU)

    Raises:
        NotImplementedError: If rnn_type is not one of ["lstm", "rnn", "gru"]
    """
    if 'lstm' == self.rnn_type:
      layer = keras.layers.LSTM(
        units=units,
        activation=self.rnn_activation,
        recurrent_activation=self.rnn_recurrent_activation,
        dropout=self.rnn_dropout,
        recurrent_dropout=self.rnn_recurrent_dropout,
        unroll=self.rnn_unroll,
        go_backwards=self.rnn_go_backwards,
        seed=RANDOM_STATE,
        **kwargs
      )
    elif 'rnn' == self.rnn_type:
      layer = keras.layers.SimpleRNN(
        units=units,
        activation=self.rnn_activation,
        dropout=self.rnn_dropout,
        recurrent_dropout=self.rnn_recurrent_dropout,
        unroll=self.rnn_unroll,
        go_backwards=self.rnn_go_backwards,
        seed=RANDOM_STATE,
        **kwargs
      )
    elif 'gru' == self.rnn_type:
      layer = keras.layers.GRU(
        units=units,
        activation=self.rnn_activation,
        recurrent_activation=self.rnn_recurrent_activation,
        dropout=self.rnn_dropout,
        recurrent_dropout=self.rnn_recurrent_dropout,
        unroll=self.rnn_unroll,
        go_backwards=self.rnn_go_backwards,
        seed=RANDOM_STATE,
        **kwargs
      )
    else:
      raise NotImplementedError(f"Unknown RNN type {self.rnn_type}")

    if self.rnn_bidirectional:
      layer = keras.layers.Bidirectional(layer)

    return layer

  def _build_norm_layer(self, **kwargs) -> keras.Layer:
    """Builds a normalization layer based on the specified normalization type.

    Args:
        **kwargs: Additional keyword arguments passed to the normalization layer

    Returns:
        keras.Layer: Normalization layer (BatchNorm, LayerNorm, or Identity)

    Raises:
        NotImplementedError: If norm_type is not one of [None, "batch", "layer"]
    """
    if self.norm_type is None:
      layer = keras.layers.Identity()
    elif self.norm_type == 'batch':
      layer = keras.layers.BatchNormalization(
        epsilon=self.norm_epsilon,
        center=self.norm_center,
        scale=self.norm_scale,
        **kwargs
      )
    elif self.norm_type == 'layer':
      layer = keras.layers.LayerNormalization(
        epsilon=self.norm_epsilon,
        center=self.norm_center,
        scale=self.norm_scale,
        **kwargs
      )
    else:
      raise NotImplementedError(f"Unknown normalization type {self.norm_type}")

    return layer

  def _build_ffn_layer(self, name: Optional[str] = None, **kwargs) -> keras.layers.Layer:
    """Builds feed-forward layers with optional dropout.

    Args:
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to the layers

    Returns:
        keras.layers.Layer: Sequential model containing feed-forward layers
    """
    name = self._generate_layer_name("ffn", name)
    layer = keras.models.Sequential([keras.layers.Identity()], name=name)

    for ff_dim in self.ff_dims:
      layer.add(keras.layers.Dense(units=ff_dim, activation='linear'))
      layer.add(self._build_norm_layer())
      layer.add(self._build_activation_layer(self.ff_activation))
      layer.add(keras.layers.Dropout(self.ff_dropout))

    return layer

  def build_graph(self) -> keras.Model:
    """Builds and returns a Keras Model instance with the defined architecture.

    Returns:
        keras.Model: Compiled model with defined inputs and outputs
    """
    inputs = {"activity": keras.layers.Input(shape=(self.max_case_len,), name="activity", dtype='string')}
    inputs = inputs | {k: keras.layers.Input(shape=(self.max_case_len,), name=k, dtype='string') for k in self.categorical_attrs.keys()}
    inputs = inputs | {k: keras.layers.Input(shape=(self.max_case_len,), name=k, dtype='float32') for k in self.numerical_attrs.keys()}

    return keras.Model(inputs=list(inputs.values()), outputs=self.call(inputs), name=self.name)

  def call(self, inputs):
    """Forward pass of the model.

    Args:
        inputs: Activity input tensor
        inputs_interval_since_last_event: Time interval input tensor
        inputs_hour_of_day: Hour of day input tensor
        inputs_day_of_week: Day of week input tensor
        training (bool, optional): Whether in training mode. Defaults to False

    Returns:
        List[tf.Tensor]: List of output tensors for each task
    """
    x_activity = self.activity_encoding(inputs["activity"])

    x_categorical = {k: v(inputs[k]) for k, v in self.categorical_encodings.items()}
    x_numerical = {k: v(inputs[k]) for k, v in self.numerical_encodings.items()}

    x = keras.layers.concatenate([x_activity] + list(x_categorical.values()) + list(x_numerical.values()))
    x = self.rnn(x)

    x_outputs = {k: self.ffns[k](x) for k in self.output_dict.keys()}
    x_outputs = {k: self.outputs[k](v) for k, v in x_outputs.items()}

    return x_outputs

In [ ]:
dalstm = BaseDARNN(
    max_case_len=3,
    activity_vocab=['a', 'b', 'c'],
    categorical_attrs={"resource": ["r1", "r2"], "organization": ["o1", "o2", "o3"]},
    numerical_attrs={"day": [1, 2], "month": [1.1, 1.2, 1.3]},
    output_dict={"remaining_time": {'units': 1, 'activation': 'relu'}},
)

dalstm = dalstm.build_graph()
dalstm.summary()

In [ ]:
keras.utils.plot_model(
  dalstm,
  show_shapes=True,
  show_dtype=True,
  show_layer_names=True,
  expand_nested=True,
  show_layer_activations=True,
  show_trainable=True,
)


### EmbeddingLSTM

In [ ]:
class EmbeddingLSTM(keras.Model, LayerNameMixin, BaseModelMixin, ConfigSerializerMixin):
  def __init__(
    self,
    output_dict: Dict[str, dict],
    rnn_units_shared: List[int],
    rnn_units_nonshared: List[int] = [],
    rnn_type: Literal['lstm', 'gru', 'rnn'] = 'lstm',
    rnn_bidirectional: bool = True,
    rnn_activation: str = 'tanh',
    rnn_recurrent_activation: str = 'sigmoid',
    rnn_go_backwards: bool = False,
    rnn_unroll: bool = False,
    rnn_recurrent_dropout: float = 0.0,
    rnn_dropout: float = 0.0,

    norm_type: Optional[Literal['layer', 'batch']] = 'layer',

    ff_dims: List[int] = [],
    ff_activation: str = 'leaky_relu',
    ff_dropout: float = 0.0,
    **kwargs
  ):
    super().__init__(**kwargs)

    self.output_dict = output_dict

    self.rnn_units_shared = rnn_units_shared
    self.rnn_units_nonshared = rnn_units_nonshared
    self.rnn_type = rnn_type
    self.rnn_bidirectional = rnn_bidirectional
    self.rnn_activation = rnn_activation
    self.rnn_recurrent_activation = rnn_recurrent_activation
    self.rnn_go_backwards = rnn_go_backwards
    self.rnn_unroll = rnn_unroll
    self.rnn_recurrent_dropout = rnn_recurrent_dropout
    self.rnn_dropout = rnn_dropout

    self.norm_type = norm_type
    self.ff_dims = ff_dims
    self.ff_activation = ff_activation
    self.ff_dropout = ff_dropout

    self.mask_token = 0.0

    self.pad_or_window = None

    self.hardshared_rnn = self._build_hardshared_rnn()
    self.softshared_rnns = {output_name: self._build_softshared_rnn(name=output_name) for output_name in self.output_dict.keys()}
    self.predictions_heads = self._build_prediction_heads()

  @property
  def default_metrics(self):
    return {k: self.find_default_metrics(v['activation']) for k, v in self.output_dict.items()}

  @property
  def default_loss(self):
    return {k: self.find_default_loss(v['activation']) for k, v in self.output_dict.items()}

  def _build_rnn_layer(self, units: int, **kwargs) -> keras.Layer:
    """Builds a single RNN layer based on the specified RNN type.

    Args:
        units (int): Number of output units
        **kwargs: Additional keyword arguments passed to the RNN layer

    Returns:
        keras.Layer: RNN layer (LSTM, SimpleRNN, or GRU)

    Raises:
        NotImplementedError: If rnn_type is not one of ["lstm", "rnn", "gru"]
    """
    if 'lstm' == self.rnn_type:
      layer = keras.layers.LSTM(
        units=units,
        activation=self.rnn_activation,
        recurrent_activation=self.rnn_recurrent_activation,
        dropout=self.rnn_dropout,
        recurrent_dropout=self.rnn_recurrent_dropout,
        unroll=self.rnn_unroll,
        go_backwards=self.rnn_go_backwards,
        seed=RANDOM_STATE,
        **kwargs
      )
    elif 'rnn' == self.rnn_type:
      layer = keras.layers.SimpleRNN(
        units=units,
        activation=self.rnn_activation,
        dropout=self.rnn_dropout,
        recurrent_dropout=self.rnn_recurrent_dropout,
        unroll=self.rnn_unroll,
        go_backwards=self.rnn_go_backwards,
        seed=RANDOM_STATE,
        **kwargs
      )
    elif 'gru' == self.rnn_type:
      layer = keras.layers.GRU(
        units=units,
        activation=self.rnn_activation,
        recurrent_activation=self.rnn_recurrent_activation,
        dropout=self.rnn_dropout,
        recurrent_dropout=self.rnn_recurrent_dropout,
        unroll=self.rnn_unroll,
        go_backwards=self.rnn_go_backwards,
        seed=RANDOM_STATE,
        **kwargs
      )
    else:
      raise NotImplementedError(f"Unknown RNN type {self.rnn_type}")

    if self.rnn_bidirectional:
      layer = keras.layers.Bidirectional(layer)

    return layer

  def _build_ffn_layer(self, name: Optional[str] = None, **kwargs) -> keras.Layer:
    """Builds feed-forward layers with optional dropout.

    Args:
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to the layers

    Returns:
        keras.layers.Layer: Sequential model containing feed-forward layers
    """
    name = self._generate_layer_name("ffn", name)
    layer = keras.models.Sequential([keras.layers.Identity()], name=name)

    for ff_dim in self.ff_dims:
      layer.add(self._build_norm_layer(self.norm_type))
      layer.add(keras.layers.Dense(units=ff_dim, activation='linear'))
      layer.add(self._build_activation_layer(self.ff_activation))
      layer.add(keras.layers.Dropout(self.ff_dropout))

    return layer

  def _build_hardshared_rnn(self, name: Optional[str] = None, **kwargs) -> keras.Layer:
    name = self._generate_layer_name(f"hardshared_{self.rnn_type}", name)

    layer = keras.models.Sequential([keras.layers.Identity()], name=name)
    for units in self.rnn_units_shared[:-1]:
      layer.add(self._build_rnn_layer(units, return_sequences=True))
      layer.add(self._build_norm_layer(self.norm_type))

    if self.rnn_units_nonshared is None or len(self.rnn_units_nonshared) == 0:
      layer.add(self._build_rnn_layer(self.rnn_units_shared[-1], return_sequences=False))
    else:
      layer.add(self._build_rnn_layer(self.rnn_units_shared[-1], return_sequences=True))

    layer.add(self._build_norm_layer(self.norm_type))

    return layer

  def _build_softshared_rnn(self, name: Optional[str] = None, **kwargs) -> keras.Layer:
    name = self._generate_layer_name(f"softshared_{self.rnn_type}", name)

    layer = keras.models.Sequential([keras.layers.Identity()], name=name)
    for units in self.rnn_units_nonshared[:-1]:
      layer.add(self._build_rnn_layer(units, return_sequences=True))
      layer.add(self._build_norm_layer(self.norm_type))

    if len(self.rnn_units_nonshared) > 0:
      layer.add(self._build_rnn_layer(self.rnn_units_nonshared[-1], return_sequences=False))
      layer.add(self._build_norm_layer(self.norm_type))

    return layer

  def _build_prediction_heads(self, **kwargs) -> dict[str, keras.Layer]:
    return {
      output_name: keras.models.Sequential([
         self._build_ffn_layer(name=output_name),
        keras.layers.Dense(output_config['units'], activation='linear'),
        self._build_activation_layer(output_config['activation'], name=output_name),
      ], name=output_name) for output_name, output_config in self.output_dict.items()
    }

  def compute_output_shape(self, input_shape: tuple[Optional[int], int, int]) -> dict[str, Optional[int], int, int]:
    return {output_name: output_config['units'] for output_name, output_config in self.output_dict}

  def build(self, input_shape: tuple[Optional[int], int, int]) -> None:
    batch_size, seq_len, feature_dim = input_shape
    self.max_seq_len = seq_len

    self.pad_or_window = PadAndTruncate(self.max_seq_len, self.mask_token)
    self.pad_or_window.build(input_shape)
    input_shape = self.pad_or_window.compute_output_shape(input_shape)

    self.hardshared_rnn.build(input_shape)
    input_shape = self.hardshared_rnn.compute_output_shape(input_shape)

    for softshared_rnn in self.softshared_rnns.values():
      softshared_rnn.build(input_shape)

    for output_name, prediction_head in self.predictions_heads.items():
      head_input_shape = self.softshared_rnns[output_name].compute_output_shape(input_shape)
      prediction_head.build(head_input_shape)

    self.built = True

  def freeze_base_model(self, include_softshared: bool = False) -> None:
    self.hardshared_rnn.trainable = False
    if include_softshared:
      for softshared_rnn in self.softshared_rnns.values():
        softshared_rnn.trainable = False

  def unfreeze_base_model(self, include_softshared: bool = False) -> None:
    self.hardshared_rnn.trainable = True
    if include_softshared:
      for softshared_rnn in self.softshared_rnns.values():
        softshared_rnn.trainable = True

  def replace_prediction_head(self, output_dict: Dict[str, dict]) -> keras.Model:
    config = self.get_config()
    config['output_dict'] = output_dict

    model = self.__class__.from_config(config)
    model.build(**self.get_build_config())
    model.hardshared_rnn.set_weights(self.hardshared_rnn.get_weights())

    return model

  def build_graph(self, seq_len: int, feature_dim: int, batch_size: int = None) -> keras.Model:
    inputs = keras.layers.Input(shape=(seq_len, feature_dim), batch_size=batch_size, ragged=True)
    return keras.Model(inputs=inputs, outputs=self.call(inputs), name=self.name)

  def get_config(self) -> dict:
    config = super().get_config()
    config.update({
      "output_dict": self.output_dict,
      "rnn_units_shared": self.rnn_units_shared,
      "rnn_units_nonshared": self.rnn_units_nonshared,
      "rnn_type": self.rnn_type,
      "rnn_bidirectional": self.rnn_bidirectional,
      "rnn_activation": self.rnn_activation,
      "rnn_recurrent_activation": self.rnn_recurrent_activation,
      "rnn_go_backwards": self.rnn_go_backwards,
      "rnn_unroll": self.rnn_unroll,
      "rnn_recurrent_dropout": self.rnn_recurrent_dropout,
      "rnn_dropout": self.rnn_dropout,
      "norm_type": self.norm_type,
      "ff_dims": self.ff_dims,
      "ff_activation": self.ff_activation,
      "ff_dropout": self.ff_dropout,
    })
    return self._serialize_config(config)

  @classmethod
  def from_config(cls, config) -> 'EmbeddingLSTM':
    return cls(**config)

  def call(self, inputs: keras.KerasTensor, mask: Optional[keras.KerasTensor] = None):
    inputs = keras.ops.convert_to_tensor(inputs, ragged=False)

    if not self.built:
      self.build(keras.ops.shape(inputs))

    x = self.pad_or_window(inputs)
    x = self.hardshared_rnn(x)

    x = {k: rnn(x) for k, rnn in self.softshared_rnns.items()}
    x = {k: self.predictions_heads[k](v) for k, v in x.items()}
    return x

In [ ]:
lstm = EmbeddingLSTM(
  output_dict={"remaining_time": {'units': 1, 'activation': 'relu'}},
  rnn_units_shared=[128, 64, 32],
  rnn_type='lstm',
  rnn_bidirectional=True,
  ff_dims=[32, 16],
  ff_activation='relu',
)

lstm = lstm.build_graph(seq_len=10, feature_dim=128)
lstm.summary()

In [ ]:
keras.utils.plot_model(
  lstm,
  show_shapes=True,
  show_dtype=True,
  show_layer_names=True,
  expand_nested=True,
  show_layer_activations=True,
)

## Transformer

### EmbeddingTransformer

In [ ]:
class EmbeddingTransformer(keras.Model, LayerNameMixin, BaseModelMixin, ConfigSerializerMixin):
  def __init__(
    self,
    output_dict: Dict[str, dict],
    enc_heads_shared: List[int],
    enc_ff_dims_shared: List[int],
    enc_heads_nonshared: List[int] = [],
    enc_ff_dims_nonshared: List[int] = [],
    enc_dropout: float = 0.0,
    enc_activation: str = 'relu',
    enc_normalize_first: bool = True,
    pos_encoding_type: Optional[Literal['sincos', 'learned', 'rotary', 'temporal']] = 'learned',
    deseq_type: Literal['maxpool', 'avgpool', 'flatten'] = 'maxpool',
    norm_type: Optional[Literal['layer', 'batch']] = 'layer',
    ff_dims: List[int] = [],
    ff_activation: str = 'leaky_relu',
    ff_dropout: float = 0.0,
    **kwargs
  ):
    super().__init__(**kwargs)
    self.supports_jit = False

    if not len(enc_heads_shared) > 0 or not len(enc_ff_dims_shared) > 0:
      raise ValueError(f"enc_heads_nonshared and enc_ff_dim_nonshared must be at least 1")
    elif len(enc_heads_shared) != len(enc_ff_dims_shared):
      raise ValueError(f"enc_heads_shared and enc_ff_dim_shared must have the same length")
    elif len(enc_heads_nonshared) != len(enc_ff_dims_nonshared):
      raise ValueError("enc_heads_nonshared and enc_ff_dim_nonshared must have the same length")

    self.output_dict = output_dict

    self.pos_encoding_type = pos_encoding_type
    self.enc_heads_shared = enc_heads_shared
    self.enc_ff_dims_shared = enc_ff_dims_shared
    self.enc_heads_nonshared = enc_heads_nonshared
    self.enc_ff_dims_nonshared = enc_ff_dims_nonshared
    self.enc_dropout = enc_dropout
    self.enc_activation = enc_activation
    self.enc_normalize_first = enc_normalize_first

    self.deseq_type = deseq_type

    self.norm_type = norm_type
    self.ff_dims = ff_dims
    self.ff_activation = ff_activation
    self.ff_dropout = ff_dropout

    self.mask_token = 0.0

    self.pad_or_window = None
    self.pos_encoding = None

    self.hardshared_encoder = self._build_hardshared_encoder()

    self.softshared_encoders = {
      output_name: keras.models.Sequential([
        self._build_softshared_encoder(name=output_name),
        self._build_desequentialization_layer(name=output_name),
      ], name=f"softshared_encoder_{output_name}") for output_name in self.output_dict.keys()
    }

    self.prediction_heads = self._build_prediction_heads()

  @property
  def default_metrics(self):
    return {k: self.find_default_metrics(v['activation']) for k, v in self.output_dict.items()}

  @property
  def default_loss(self):
    return {k: self.find_default_loss(v['activation']) for k, v in self.output_dict.items()}

  def _build_positional_encoding_layer(self, **kwargs) -> keras.layers.Layer:
    if self.pos_encoding_type is None:
      layer = keras.layers.Identity()
    elif 'sincos' == self.pos_encoding_type:
      layer = keras_nlp.layers.SinePositionEncoding(**kwargs)
    elif 'learned' == self.pos_encoding_type:
      layer = keras_nlp.layers.PositionEmbedding(sequence_length=self.max_seq_len, **kwargs)
    elif 'rotary' == self.pos_encoding_type:
      layer = keras_nlp.layers.RotaryEmbedding(**kwargs)
    else:
      raise NotImplementedError(f"Unknown positional encoding type {self.pos_encoding_type}")

    return layer

  def _build_encoder_layer(self, heads: int, intermediate_dim: int, **kwargs) -> keras.Layer:
    layer = keras_nlp.layers.TransformerEncoder(
      num_heads=heads,
      intermediate_dim=intermediate_dim,
      dropout=self.enc_dropout,
      activation=self.enc_activation,
      normalize_first=self.enc_normalize_first,
      **kwargs
    )

    return layer

  def _build_hardshared_encoder(self, name: Optional[str] = None, **kwargs) -> keras.Layer:
    name = self._generate_layer_name(f"hardshared_encoder", name)

    layer = keras.models.Sequential([keras.layers.Identity()], name=name)
    for num_heads, intermediate_dim in zip(self.enc_heads_shared, self.enc_ff_dims_shared, strict=True):
      layer.add(self._build_encoder_layer(num_heads, intermediate_dim))

    return layer

  def _build_softshared_encoder(self, name: Optional[str] = None, **kwargs) -> keras.Layer:
    name = self._generate_layer_name(f"softshared_encoder", name)

    layer = keras.models.Sequential([keras.layers.Identity()], name=name)
    for num_heads, intermediate_dim in zip(self.enc_heads_nonshared, self.enc_ff_dims_nonshared, strict=True):
      layer.add(self._build_encoder_layer(num_heads, intermediate_dim))

    return layer

  def _build_ffn_layer(self, name: Optional[str] = None, **kwargs) -> keras.Layer:
    """Builds feed-forward layers with optional dropout.

    Args:
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to the layers

    Returns:
        keras.layers.Layer: Sequential model containing feed-forward layers
    """
    name = self._generate_layer_name("ffn", name)
    layer = keras.models.Sequential([keras.layers.Identity()], name=name)

    for ff_dim in self.ff_dims:
      layer.add(self._build_norm_layer(self.norm_type))
      layer.add(keras.layers.Dense(units=ff_dim, activation='linear'))
      layer.add(self._build_activation_layer(self.ff_activation))
      layer.add(keras.layers.Dropout(self.ff_dropout))

    return layer

  def _build_desequentialization_layer(self, name: Optional[str] = None, **kwargs) -> keras.Layer:
    name = self._generate_layer_name(f"deseq_{self.deseq_type}", name)

    layer = None
    if 'maxpool' == self.deseq_type:
      layer = keras.layers.GlobalMaxPooling1D(name=name, **kwargs)
    elif 'avgpool' == self.deseq_type:
      layer = keras.layers.GlobalAveragePooling1D(name=name, **kwargs)
    elif 'flatten' == self.deseq_type:
      layer = keras.layers.Flatten(name=name, **kwargs)
    else:
      raise NotImplementedError(f"Unknown desequentialization type {self.deseq_type}")

    return layer

  def _build_prediction_heads(self, **kwargs) -> dict[str, keras.Layer]:
    return {
      output_name: keras.models.Sequential([
        self._build_ffn_layer(name=output_name),
        keras.layers.Dense(output_config['units'], activation='linear'),
        self._build_activation_layer(output_config['activation']),
      ], name=output_name) for output_name, output_config in self.output_dict.items()
    }

  def freeze_base_model(self, include_softshared: bool = True) -> None:
    self.hardshared_encoder.trainable = False
    if include_softshared:
      for softshared_encoder in self.softshared_encoders.values():
        softshared_encoder.trainable = False

  def unfreeze_base_model(self, include_softshared: bool = True) -> None:
    self.hardshared_encoder.trainable = True
    if include_softshared:
      for softshared_encoder in self.softshared_encoders.values():
        softshared_encoder.trainable = True

  def replace_prediction_head(self, output_dict: Dict[str, dict], name: Optional[str] = None) -> keras.Model:
    config = self.get_config()
    config['output_dict'] = output_dict
    config['name'] = name

    model = self.__class__.from_config(config)
    model.build(**self.get_build_config())
    model.hardshared_encoder.set_weights(self.hardshared_encoder.get_weights())
    return model

  def compute_output_shape(self, input_shape: tuple[Optional[int], int, int]) -> dict[tuple[Optional[int], int, int]]:
    return {output_name: output_config['units'] for output_name, output_config in self.output_dict}

  def build_graph(self, seq_len: int, feature_dim: int, batch_size: int = None) -> keras.Model:
    inputs = keras.layers.Input(shape=(seq_len, feature_dim), batch_size=batch_size, ragged=True, dtype='float32')
    return keras.Model(inputs=inputs, outputs=self.call(inputs), name=self.name)

  def build(self, input_shape: tuple[Optional[int], int, int]) -> None:
    batch_size, seq_len, feature_dim = input_shape
    self.max_seq_len = seq_len

    self.pad_or_window = PadAndTruncate(self.max_seq_len, self.mask_token)
    self.pad_or_window.build(input_shape)
    input_shape = self.pad_or_window.compute_output_shape(input_shape)

    self.pos_encoding = self._build_positional_encoding_layer()
    self.pos_encoding.build(input_shape)

    self.hardshared_encoder.build(input_shape)
    input_shape = self.hardshared_encoder.compute_output_shape(input_shape)

    for softshared_encoder in self.softshared_encoders.values():
      softshared_encoder.build(input_shape)

    for output_name, prediction_head in self.prediction_heads.items():
      head_input_shape = self.softshared_encoders[output_name].compute_output_shape(input_shape)
      prediction_head.build(head_input_shape)

    self.built = True

  @classmethod
  def from_config(cls, config) -> 'EmbeddingTransformer':
    return cls(**config)

  def get_config(self) -> dict:
    config = super().get_config()
    config.update({
      "output_dict": self.output_dict,
      "enc_heads_shared": self.enc_heads_shared,
      "enc_ff_dims_shared": self.enc_ff_dims_shared,
      "enc_heads_nonshared": self.enc_heads_nonshared,
      "enc_ff_dims_nonshared": self.enc_ff_dims_nonshared,
      "enc_dropout": self.enc_dropout,
      "enc_activation": self.enc_activation,
      "enc_normalize_first": self.enc_normalize_first,
      "pos_encoding_type": self.pos_encoding_type,
      "deseq_type": self.deseq_type,
      "norm_type": self.norm_type,
      "ff_dims": self.ff_dims,
      "ff_activation": self.ff_activation,
      "ff_dropout": self.ff_dropout,
      "name": self.name,
    })

    return self._serialize_config(config)

  def call(self, inputs: keras.KerasTensor, mask: Optional[keras.KerasTensor] = None):
    inputs = keras.ops.convert_to_tensor(inputs, ragged=False)

    if not self.built:
      self.build(keras.ops.shape(inputs))

    x = self.pad_or_window(inputs)
    if self.pos_encoding_type is None or 'rotary' == self.pos_encoding_type:
      x = self.pos_encoding(x)
    elif 'learned' == self.pos_encoding_type or 'sincos' == self.pos_encoding_type:
      x = keras.layers.add([self.pos_encoding(x), x])
    else:
      raise NotImplementedError(f"Unknown positional encoding type {self.pos_encoding_type}")

    x = self.hardshared_encoder(x)
    x = {k: enc(x) for k, enc in self.softshared_encoders.items()}
    x = {k: self.prediction_heads[k](v) for k, v in x.items()}

    return x

In [ ]:
trans = EmbeddingTransformer(
  output_dict={"remaining_time": {'units': 1, 'activation': 'relu'}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  ff_dims=[32, 16],
)

trans = trans.build_graph(seq_len=10, feature_dim=128)
trans.summary(expand_nested=True)

In [ ]:
keras.utils.plot_model(
  trans,
  show_shapes=True,
  show_dtype=True,
  show_layer_names=True,
  expand_nested=True,
  show_layer_activations=True,
)

### ProcessTransformer

In [ ]:
class BaseProcessTransformer(keras.Model, LayerNameMixin, BaseModelMixin):
  """A base Transformer model for process mining tasks with shared and task-specific layers. Provides a structured and more flexible reimplementation of the work by Bukhsh et al. 2021 (https://github.com/Zaharah/processtransformer).

  The default parameters represent the original model architecture with its hyperparameters.

  This model implements a neural architecture for process mining that combines:
  - Multiple input features (activities, temporal features)
  - Positional encodings for sequence information
  - Shared Transformer encoder layers for common feature extraction
  - Task-specific Transformer encoder layers for different prediction tasks
  - Configurable desequentialization and feed-forward layers

  The architecture follows a multi-task learning approach where lower layers are shared
  across all tasks while upper layers are task-specific. The model processes sequential
  data using self-attention mechanisms and can handle variable-length sequences up to
  max_case_len.

  Args:
      max_case_len (int): Maximum length of process cases/sequences
      output_dict (Dict[str, dict]): Configuration for output layers. Format: {"task_name": {"units": int, "activation": str}} Example: {"remaining_time": {'units': 1, 'activation': 'relu'}, "next_activity": {'units': 10, 'activation': 'softmax'}}
      activity_vocab (List[str]): Vocabulary of possible process activities
      activity_encoding_type (str, optional): Type of activity encoding. Currently only supports "embedding". Defaults to "embedding"
      time_encoding_type (str, optional): Type of time feature encoding. Options: "mean", "minmax", or "normal". Defaults to "mean"
      interval_since_last_event_vocab (List[float], optional): Vocabulary for time intervals
      hour_of_day_vocab (List[float], optional): Vocabulary for hour features
      day_of_week_vocab (List[float], optional): Vocabulary for day of week features
      activity_embedding_dim (int): Dimension for activity embeddings. Defaults to 36
      pos_encoding_type (str, optional): Type of positional encoding. Options: "sincos", "learned", "rotary", or "temporal". Defaults to "learned"
      enc_heads_shared (List[int]): Number of attention heads in each shared encoder layer. Defaults to [4]
      enc_ff_dims_shared (List[int]): Feed-forward dimensions in each shared encoder layer. Defaults to [64]
      enc_heads_nonshared (List[int]): Number of attention heads in task-specific layers. Defaults to []
      enc_ff_dims_nonshared (List[int]): Feed-forward dimensions in task-specific layers. Defaults to []
      enc_dropout (float): Dropout rate for encoder layers. Defaults to 0.0
      enc_activation (str): Activation function for encoder layers. Defaults to "relu"
      enc_normalize_first (bool): Whether to apply normalization before attention and feed-forward layers. Defaults to False
      deseq_type (str): Method to convert sequences to fixed-size vectors. Options: "maxpool", "avgpool", "flatten". Defaults to "maxpool"
      ff_dims (List[int]): Units in each final feed-forward layer. Defaults to [32, 128]
      ff_dropout (float): Dropout rate for final feed-forward layers. Defaults to 0.0
      ff_activation (str): Activation for final feed-forward layers. Defaults to "relu"
      numeric_mask_token (float): Token used for masking numeric values. Defaults to TOKEN_PAD_NUM
      mask_token (str): Token used for masking categorical values. Defaults to TOKEN_PAD
      oov_token (str): Token used for out-of-vocabulary values. Defaults to TOKEN_OOV

  Raises:
      ValueError: If enc_heads_shared or enc_ff_dims_shared is empty

  Author:
      Marc C. Hennig (mhennig@hm.edu)
  """
  def __init__(
    self,
    max_case_len: int,
    output_dict: Dict[str, dict],
    activity_vocab: List[str],
    activity_encoding_type: Optional[Literal['embedding']] = 'embedding',
    time_encoding_type: Optional[Literal['mean', 'minmax', 'normal']] = 'mean',
    interval_since_last_event_vocab: Optional[List[float]] = None,
    hour_of_day_vocab: Optional[List[float]] = None,
    day_of_week_vocab: Optional[List[float]] = None,
    activity_embedding_dim: int = 36,
    pos_encoding_type: Optional[Literal['sincos', 'learned', 'rotary', 'temporal']] = 'learned',
    enc_heads_shared: List[int] = [4],
    enc_ff_dims_shared: List[int] = [64],
    enc_heads_nonshared: List[int] = [],
    enc_ff_dims_nonshared: List[int] = [],
    enc_dropout: float = 0.0,
    enc_activation: str = 'relu',
    enc_normalize_first: bool = False,
    deseq_type: Literal['maxpool', 'avgpool', 'flatten'] = 'maxpool',
    ff_dims: List[int] = [32, 128],
    ff_dropout: float = 0.0,
    ff_activation: str = 'relu',
    numeric_mask_token: float = TOKEN_PAD_NUM,
    mask_token: str = TOKEN_PAD,
    oov_token: str = TOKEN_OOV,
    **kwargs
  ):
    super().__init__(**kwargs)
    if not len(enc_heads_shared) > 0 or not len(enc_ff_dims_shared) > 0:
      raise ValueError(f"enc_heads_nonshared and enc_ff_dim_nonshared must be at least 1")

    self.max_case_len = max_case_len
    self.output_dict = output_dict

    self.activity_encoding_type = activity_encoding_type
    self.activity_vocab = activity_vocab
    self.activity_embedding_dim = activity_embedding_dim

    self.time_encoding_type = time_encoding_type
    self.pos_encoding_type = pos_encoding_type

    self.enc_heads_shared = enc_heads_shared
    self.enc_ff_dims_shared = enc_ff_dims_shared
    self.enc_heads_nonshared = enc_heads_nonshared
    self.enc_ff_dims_nonshared = enc_ff_dims_nonshared
    self.enc_dropout = enc_dropout
    self.enc_activation = enc_activation
    self.enc_normalize_first = enc_normalize_first

    self.deseq_type = deseq_type

    self.ff_dims = ff_dims
    self.ff_dropout = ff_dropout
    self.ff_activation = ff_activation

    self.numeric_mask_token = numeric_mask_token
    self.mask_token = mask_token
    self.oov_token = oov_token

    self.activity_encoding = self._build_activity_encoding_layer()
    self.interval_since_last_event_encoding = self._build_time_encoding_layer(vocab=interval_since_last_event_vocab, name="interval_since_last_event")
    self.hour_of_day_encoding = self._build_time_encoding_layer(vocab=hour_of_day_vocab, name="hour_of_day")
    self.day_of_week_encoding = self._build_time_encoding_layer(vocab=day_of_week_vocab, name="day_of_week")

    self.pos_encoding = self._build_positional_encoding_layer()

    self.shared_encoder = self._build_hard_shared_transformer_encoder_layer()

    self.non_shared_encoders = []
    self.deseqs = []
    self.ff_layers = []
    self.outputs = []
    for output_name, output_config in self.output_dict.items():
      self.non_shared_encoders.append(self._build_soft_shared_transformer_encoder_layer(name=output_name))
      self.deseqs.append(self._build_desequentialization_layer(name=output_name))
      self.ff_layers.append(self._build_ffn_layer(name=output_name))
      self.outputs.append(keras.layers.Dense(
          units=output_config['units'],
          activation=output_config['activation'],
          name=output_name
      ))

  @property
  def default_loss(self) -> Dict[str, keras.Loss]:
    return {k: self.find_default_loss(v['activation']) for k, v in self.output_dict.items()}

  @property
  def default_metrics(self) -> Dict[str, List[keras.Metric]]:
    return {k: self.find_default_metrics(v['activation']) for k, v in self.output_dict.items()}

  @property
  def default_learning_rate(self) -> Union[float, keras.optimizers.schedules.LearningRateSchedule]:
    return 0.001

  @property
  def default_optimizer(self) -> keras.optimizers.Optimizer:
    return keras.optimizers.Adam(learning_rate=self.default_learning_rate)

  @property
  def default_regression_loss(self) -> keras.Loss:
    return keras.losses.LogCosh()

  def _build_activity_encoding_layer(self, name: Optional[str] = None, **kwargs) -> keras.layers.Layer:
    """Builds an activity encoding layer based on the specified encoding type.

    Currently only supports embedding-based encoding, which converts activity names
    to dense vectors of fixed dimensionality.

    Args:
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to the layers

    Returns:
        keras.layers.Layer: Layer for activity encoding

    Raises:
        NotImplementedError: If activity_encoding_type is not "embedding"
    """
    name = self._generate_layer_name(f"activity_encoding_{self.activity_encoding_type}", name)

    if self.activity_encoding_type is None:
      layer = keras.layers.Identity(name=name)
    elif 'embedding' == self.activity_encoding_type:
      lookup = keras.layers.StringLookup(
        mask_token=self.mask_token,
        oov_token=self.oov_token,
        output_mode='int',
        pad_to_max_tokens=False,
      )
      lookup.adapt(self.activity_vocab)

      layer = keras.models.Sequential([
        lookup,
        keras.layers.Embedding(
          input_dim=len(lookup.get_vocabulary()),
          output_dim=self.activity_embedding_dim,
          mask_zero=True,
        )],
        name=name,
      )
    else:
      raise NotImplementedError(f"Unknown activity encoding type {self.activity_encoding_type}")

    return layer

  def _build_time_encoding_layer(self, vocab: Optional[List[float]] = None, name: Optional[str] = None, **kwargs) -> keras.layers.Layer:
    """Builds a time encoding layer based on the specified encoding type.

    Args:
        vocab (List[float], optional): Vocabulary of time values for adaptation
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to the layers

    Returns:
        keras.layers.Layer: Sequential layer for time feature encoding

    Raises:
        NotImplementedError: If time_encoding_type is not one of [None, "mean", "minmax", "normal"]
    """
    name = self._generate_layer_name(f"time_encoding_{self.time_encoding_type}", name)
    layer = keras.models.Sequential([keras.layers.Masking(self.numeric_mask_token)], name=name)

    if self.time_encoding_type is None:
      layer.add(keras.layers.Identity(name=name))
    elif 'mean' == self.time_encoding_type:
      lookup = MeanScaler(axis=None)
      lookup.adapt(vocab)
      layer.add(lookup)
    elif 'minmax' == self.time_encoding_type:
      lookup = MinMaxScaler(axis=None)
      lookup.adapt(vocab)
      layer.add(lookup)
    elif 'normal' == self.time_encoding_type:
      lookup = keras.layers.Normalization(axis=None)
      lookup.adapt(vocab)
      layer.add(lookup)
    else:
      raise NotImplementedError(f"Unknown time encoding type {self.time_encoding_type}")

    return layer

  def _build_positional_encoding_layer(self, **kwargs) -> keras.layers.Layer:
    """Builds a positional encoding layer to inject sequence order information.

    Supports multiple types of positional encodings:
    - sincos: Sinusoidal position encoding (Vaswani et al., 2017)
    - learned: Trainable position embeddings
    - rotary: Rotary position embeddings (Su et al., 2021)
    - temporal: Time-aware position encoding (not implemented)

    Args:
        **kwargs: Additional keyword arguments passed to the position encoding layer

    Returns:
        keras.layers.Layer: Positional encoding layer

    Raises:
        NotImplementedError: If pos_encoding_type is "temporal" or unknown
    """
    if self.pos_encoding_type is None:
      layer = keras.layers.Identity()
    elif 'sincos' == self.pos_encoding_type:
      layer = keras_nlp.layers.SinePositionEncoding(**kwargs)
    elif 'learned' == self.pos_encoding_type:
      layer = keras_nlp.layers.PositionEmbedding(sequence_length=self.max_case_len, **kwargs)
    elif 'rotary' == self.pos_encoding_type:
      layer = keras_nlp.layers.RotaryEmbedding(**kwargs)
    elif 'temporal' == self.pos_encoding_type:
      raise NotImplementedError(f"Temporal positional encoding is not yet implemented")
    else:
      raise NotImplementedError(f"Unknown positional encoding type {self.pos_encoding_type}")

    return layer

  def _build_desequentialization_layer(self, name: Optional[str] = None, **kwargs) -> keras.layers.Layer:
    """Builds a layer to convert variable-length sequences to fixed-size vectors.

    Supports multiple pooling strategies:
    - maxpool: Global max pooling across the sequence
    - avgpool: Global average pooling across the sequence
    - flatten: Flattens the entire sequence (preserves all information but loses
      sequence invariance)

    Args:
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to the layer

    Returns:
        keras.layers.Layer: Desequentialization layer

    Raises:
        NotImplementedError: If deseq_type is unknown
    """
    name = self._generate_layer_name(f"{self.deseq_type}", name)

    if 'maxpool' == self.deseq_type:
      layer = keras.layers.GlobalMaxPooling1D(name=name, **kwargs)
    elif 'avgpool' == self.deseq_type:
      layer = keras.layers.GlobalAveragePooling1D(name=name, **kwargs)
    elif 'flatten' == self.deseq_type:
      layer = keras.layers.Flatten(name=name, **kwargs)
    else:
      raise NotImplementedError(f"Unknown desequentialization type {self.deseq_type}")

    return layer

  def _build_hard_shared_transformer_encoder_layer(self, name: Optional[str] = None, **kwargs) -> List[keras.Layer]:
    """Builds the shared Transformer encoder layers used across all tasks.

    Creates a stack of Transformer encoder layers with specified number of attention
    heads and feed-forward dimensions. These layers learn general process patterns
    shared across all prediction tasks.

    Args:
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to TransformerEncoder

    Returns:
        List[keras.Layer]: List of shared Transformer encoder layers
    """
    name = self._generate_layer_name(f"hardsharing_transformer_encoder", name)
    layer = keras.models.Sequential(name=name)

    for num_heads, ff_dim in zip(self.enc_heads_shared, self.enc_ff_dims_shared, strict=True):
      layer.add(keras_nlp.layers.TransformerEncoder(
          intermediate_dim=ff_dim,
          num_heads=num_heads,
          dropout=self.enc_dropout,
          activation=self.enc_activation,
          normalize_first=self.enc_normalize_first,
          **kwargs
      ))

    return layer

  def _build_soft_shared_transformer_encoder_layer(self, name: Optional[str] = None, **kwargs) -> keras.Layer:
    """Builds task-specific Transformer encoder layers.

    Creates a stack of Transformer encoder layers that are specific to each prediction
    task, allowing specialization for different objectives.

    Args:
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to TransformerEncoder

    Returns:
        keras.Layer: Sequential model containing task-specific Transformer encoder layers
    """
    name = self._generate_layer_name(f"softsharing_transformer_encoder", name)
    layer = keras.models.Sequential([keras.layers.Identity()], name=name)

    for num_heads, ff_dim in zip(self.enc_heads_nonshared, self.enc_ff_dims_nonshared, strict=True):
      layer.add(keras_nlp.layers.TransformerEncoder(
          intermediate_dim=ff_dim,
          num_heads=num_heads,
          dropout=self.enc_dropout,
          activation=self.enc_activation,
          normalize_first=self.enc_normalize_first,
          **kwargs
      ))

    return layer

  def _build_ffn_layer(self, name: Optional[str] = None, **kwargs) -> keras.layers.Layer:
    """Builds feed-forward layers with optional dropout.

    Args:
        name (str, optional): Name prefix for the layer
        **kwargs: Additional keyword arguments passed to the layers

    Returns:
        keras.layers.Layer: Sequential model containing feed-forward layers
    """
    name = self._generate_layer_name("ffn", name)
    layer = keras.models.Sequential(name=name)

    for ff_dim in self.ff_dims:
      layer.add(keras.layers.Dense(units=ff_dim, activation=self.ff_activation))
      layer.add(keras.layers.Dropout(self.ff_dropout))

    return layer

  def build_graph(self) -> keras.Model:
    """Builds and returns a Keras Model instance with the defined architecture.

    Returns:
        keras.Model: Compiled model with defined inputs and outputs
    """
    in_activity = keras.layers.Input(shape=(self.max_case_len,), name="activity", dtype='string')
    in_interval_since_last_event = keras.layers.Input(shape=(1,), name="time_timestamp_elapsedprev", dtype='float32')
    in_hour_of_day = keras.layers.Input(shape=(1,), name="time_timestamp_hour_raw", dtype='float32')
    in_day_of_week = keras.layers.Input(shape=(1,), name="time_timestamp_weekday_raw", dtype='float32')

    return keras.Model(inputs=[in_activity, in_interval_since_last_event, in_hour_of_day, in_day_of_week], outputs=dict(zip(self.output_dict.keys(), self.call(in_activity, in_interval_since_last_event, in_hour_of_day, in_day_of_week))), name=self.name)

  def call(self, inputs, inputs_interval_since_last_event, inputs_hour_of_day, inputs_day_of_week, training=False):
    x_activity = self.activity_encoding(inputs)
    x_interval = self.interval_since_last_event_encoding(inputs_interval_since_last_event)
    x_hour = self.hour_of_day_encoding(inputs_hour_of_day)
    x_day = self.day_of_week_encoding(inputs_day_of_week)

    if self.pos_encoding_type is None or 'rotary' == self.pos_encoding_type:
      x_activity = self.pos_encoding(x_activity)
    elif 'learned' == self.pos_encoding_type or 'sincos' == self.pos_encoding_type:
      x_activity = keras.layers.add([self.pos_encoding(x_activity), x_activity])
    else:
      raise NotImplementedError(f"Unknown positional encoding type {self.pos_encoding_type}")

    x_activity = self.shared_encoder(x_activity)

    x_outputs = []
    for encoder, deseq, ff_layer, output in zip(self.non_shared_encoders, self.deseqs, self.ff_layers, self.outputs, strict=True):
      x_nonshared = encoder(x_activity)
      x_nonshared = deseq(x_nonshared)
      x_nonshared = keras.layers.concatenate([x_nonshared, x_interval, x_hour, x_day])
      x_nonshared = ff_layer(x_nonshared)
      x_outputs.append(output(x_nonshared))

    return x_outputs

In [ ]:
trans = BaseProcessTransformer(
    max_case_len=10,
    activity_vocab=['a', 'b', 'c'],
    interval_since_last_event_vocab=np.array([1,2,3]),
    hour_of_day_vocab=np.array([1,2,3]),
    day_of_week_vocab=np.array([1,2,3]),
    output_dict={"remaining_time": {'units': 1, 'activation': 'relu'}, "next_activity": {'units': 10, 'activation': 'softmax'}},
)

trans = trans.build_graph()
trans.summary()

In [ ]:
keras.utils.plot_model(
  trans,
  show_shapes=True,
  show_dtype=True,
  show_layer_names=True,
  expand_nested=True,
  show_layer_activations=True,
  show_trainable=True,
)


### SetTransformer

#### General Layers

In [ ]:
@keras.saving.register_keras_serializable()
class InputEncoder(keras.Model, LayerNameMixin, ConfigSerializerMixin):
  def __init__(
    self,
    embed_dim: int,
    num_attrs: Dict[str, Optional[List[int]]],
    cat_attrs: Dict[str, List[str]],
    cat_embed_dim: Optional[int] = None,
    cat_encoding_type: Literal['one_hot', 'embedding', 'hashing'] = 'embedding',
    num_embed_dim: Optional[int] = None,
    num_encoding_type: Optional[Literal['mean', 'minmax', 'normal', 'log_normal']] = None,
    normalization_type: Optional[Literal['layer']] = 'layer',
    combination_type: Literal['single', 'multi', 'dict'] = 'multi',
    numeric_mask_token: float = TOKEN_PAD_NUM,
    mask_token: str = TOKEN_PAD,
    oov_token: str = TOKEN_OOV,
    **kwargs
  ):
    super().__init__(**kwargs)

    self.embed_dim = embed_dim
    self.num_attrs = num_attrs
    self.cat_attrs = cat_attrs

    self.cat_embed_dim = self.embed_dim if 'single' == combination_type or cat_embed_dim is None else cat_embed_dim
    self.cat_encoding_type = cat_encoding_type

    self.num_embed_dim = self.embed_dim if 'single' == combination_type or num_embed_dim is None else num_embed_dim
    self.num_encoding_type = num_encoding_type

    self.normalization_type = normalization_type
    self.combination_type = combination_type

    self.numeric_mask_token = numeric_mask_token
    self.mask_token = mask_token
    self.oov_token = oov_token

    self.numerical_encoders = { k: self._build_numerical_encoding_layer(vocab=v, name=k) for k, v in num_attrs.items() }
    self.categorical_encoders = { k: self._build_categorical_encoding_layer(vocab=v, name=k) for k, v in cat_attrs.items() }

  @property
  def num_features(self) -> int:
    return len(self.num_attrs) + len(self.cat_attrs)

  def _build_numerical_encoding_layer(self, vocab: Optional[List[float]] = None, name: Optional[str] = None, **kwargs) -> keras.layers.Layer:
    name = self._generate_layer_name(f"num_encoding_{self.num_encoding_type}", name)
    if vocab is not None:
      vocab = np.array(vocab)

    layer = keras.models.Sequential([keras.layers.Masking(self.numeric_mask_token)], name=name)

    if self.num_encoding_type is None or vocab is None:
      name = self._generate_layer_name(f"num_encoding_none", name)
      layer = keras.models.Sequential([keras.layers.Masking(self.numeric_mask_token)], name=name)
      layer.add(keras.layers.Identity())
    elif 'mean' == self.num_encoding_type:
      lookup = MeanScaler(axis=None)
      lookup.adapt(vocab)
      layer.add(lookup)
    elif 'minmax' == self.num_encoding_type:
      lookup = MinMaxScaler(axis=None)
      lookup.adapt(vocab)
      layer.add(lookup)
    elif 'normal' == self.num_encoding_type:
      lookup = keras.layers.Normalization(axis=None)
      lookup.adapt(vocab)
      layer.add(lookup)
    elif 'log_normal' == self.num_encoding_type:
      lookup = keras.layers.Normalization(axis=None)
      lookup.adapt(log_transform(vocab))
      layer.add(LogTransform('log1p'))
      layer.add(lookup)
    else:
      raise NotImplementedError(f"Unknown numerical encoding type {self.num_encoding_type}")

    layer.add(keras.layers.Reshape((-1, 1)))
    if self.num_embed_dim > 1:
      layer.add(keras.layers.Dense(self.num_embed_dim))
      layer.add(self._build_norm_layer())

    return layer

  def _build_norm_layer(self) -> keras.layers.Layer:
    if self.normalization_type is None:
      return keras.layers.Identity()
    elif 'layer' == self.normalization_type:
      return keras.layers.LayerNormalization()
    else:
      raise NotImplementedError(f"Unknown normalization type {self.normalization_type}")


  def _build_categorical_encoding_layer(self, vocab: Optional[List[float]] = None, name: Optional[str] = None, **kwargs) -> keras.layers.Layer:
    name = self._generate_layer_name(f"cat_encoding_{self.cat_encoding_type}", name)

    if vocab is not None:
      vocab = np.array(vocab)

    if self.cat_encoding_type is None:
      layer = keras.layers.Identity(name=name)
    elif 'one_hot' == self.cat_encoding_type:
      lookup = keras.layers.StringLookup(
        mask_token=self.mask_token,
        oov_token=self.oov_token,
        output_mode='one_hot',
        pad_to_max_tokens=False,
      )
      lookup.adapt(vocab)

      # Workaround: https://github.com/keras-team/keras/issues/19191#issuecomment-2077711036
      lookup.build()
      layer = keras.models.Sequential([
        keras.layers.Reshape((-1, 1), dtype='string'),
        keras.layers.TimeDistributed(lookup),
        self._build_norm_layer(),
      ], name=name)
    elif 'hashing' == self.cat_encoding_type:
      lookup = keras.layers.Hashing(
        mask_value=self.mask_token,
        num_bins=self.cat_embed_dim,
        output_mode='one_hot',
      )

      layer = keras.models.Sequential([
        keras.layers.Reshape((-1, 1), dtype='string'),
        lookup,
      ], name=name)
    elif 'embedding' == self.cat_encoding_type:
      lookup = keras.layers.StringLookup(
        mask_token=self.mask_token,
        oov_token=self.oov_token,
        output_mode='int',
        pad_to_max_tokens=False,
      )
      lookup.adapt(vocab)

      layer = keras.models.Sequential([
        lookup,
        keras.layers.Embedding(
          input_dim=len(lookup.get_vocabulary()),
          output_dim=self.cat_embed_dim,
          mask_zero=True,
        ),
        self._build_norm_layer(),
      ], name=name)
    else:
      raise NotImplementedError(f"Unknown categorical encoding type {self.cat_encoding_type}")

    return layer

  def build_graph(self, batch_size: Optional[int] = None) -> keras.Model:
    num_inputs = { k: keras.layers.Input((1,), batch_size=batch_size, dtype='float32', name=k) for k in self.num_attrs.keys() }
    cat_inputs = { k: keras.layers.Input((1,), batch_size=batch_size, dtype='string', name=k) for k in self.cat_attrs.keys() }

    return keras.Model(inputs=list(num_inputs.values()) + list(cat_inputs.values()), outputs=self.call(num_inputs | cat_inputs), name=self.name)

  def build(self, input_shape: tuple[Optional[int], int, int]) -> None:
    batch_size = input_shape[0]

    for encoder in self.numerical_encoders.values():
      encoder.build((batch_size, 1,))
    for encoder in self.categorical_encoders.values():
      encoder.build((batch_size, 1,))

    self.built = True

  def get_config(self) -> dict:
    config = super().get_config()
    config.update({
      "embed_dim": self.embed_dim,
      "num_attrs": self.num_attrs,
      "cat_attrs": self.cat_attrs,
      "cat_encoding_type": self.cat_encoding_type,
      "num_encoding_type": self.num_encoding_type,
      "normalization_type": self.normalization_type,
      "combination_type": self.combination_type,
      "numeric_mask_token": self.numeric_mask_token,
      "mask_token": self.mask_token,
      "oov_token": self.oov_token,
    })
    config = self._serialize_config(config)
    return config

  @classmethod
  def from_config(cls, config: dict) -> "InputEncoder":
    return cls(**config)

  def compute_output_shape(self, input_shape: Dict[str, tuple[Optional[int], int, int]]) -> Union[tuple[Optional[int], int, int], tuple[tuple[Optional[int], int, int], tuple[Optional[int], int, int]]]:
    batch_size = input_shape[0]

    if 'single' == self.combination_type:
      return (batch_size, self.num_features, self.embed_dim)
    elif 'multi' == self.combination_type:
      return (batch_size, len(self.cat_attrs), self.cat_embed_dim), ((batch_size, len(self.num_attrs), self.num_embed_dim))
    else:
      raise NotImplementedError(f"Unknown combination type {self.combination_type}")

  def call(self, inputs: Dict[str, keras.KerasTensor]) -> keras.KerasTensor:
    x_num = []
    num_attrs = inputs.keys() & self.num_attrs.keys()
    for k in num_attrs:
      x_num.append(self.numerical_encoders[k](inputs[k]))

    x_cat = []
    cat_attrs = inputs.keys() & self.cat_attrs.keys()
    for k in cat_attrs:
      x_cat.append(self.categorical_encoders[k](inputs[k]))

    if 'single' == self.combination_type:
      return keras.ops.concatenate(x_cat + x_num, axis=1)
    elif 'multi' == self.combination_type:
      return keras.ops.concatenate(x_cat, axis=1), keras.ops.concatenate(x_num, axis=1)
    elif 'dict' == self.combination_type:
      return dict(zip(cat_attrs, x_cat)), dict(zip(num_attrs, x_num))
    else:
      raise NotImplementedError(f"Unknown combination type {self.combination_type}")

In [ ]:
class MultiheadAttentionBlock(keras.Layer):
    def __init__(
      self,
      embed_dim: Optional[int] = None,
      hidden_dim: Optional[int] = None,
      num_heads: int = 1,
      dropout: float = 0.0,
      hidden_activation: str = "relu",
      **kwargs
    ):
      super().__init__(**kwargs)

      self.embed_dim = embed_dim
      self.hidden_dim = hidden_dim
      self.num_heads = num_heads
      self.dropout = dropout
      self.hidden_activation = hidden_activation

      self.norm1 = keras.layers.LayerNormalization()
      self.norm2 = keras.layers.LayerNormalization()

    def build(self, query_shape: tuple[Optional[int], int, int], key_shape: Tuple[Optional[int], int, int]) -> None:
      batch_size, seq_len, query_embed_dim = query_shape
      batch_size, seq_len, key_embed_dim = key_shape

      self.embed_dim = self.embed_dim if self.embed_dim is not None else query_embed_dim

      self.attn = keras.layers.MultiHeadAttention(
        key_dim=key_embed_dim // self.num_heads,
        num_heads=self.num_heads,
        dropout=self.dropout,
      )
      self.attn.build(query_shape, key_shape)
      out_shape = self.attn.compute_output_shape(query_shape, key_shape)

      self.norm1.build(out_shape)
      self.norm2.build(out_shape)

      if self.hidden_dim is None:
        self.hidden_dim = query_embed_dim * 2

      self.ffn = keras.Sequential([
        keras.layers.Dense(self.hidden_dim, activation=self.hidden_activation),
        keras.layers.Dropout(self.dropout),
        keras.layers.Dense(self.embed_dim)
      ])
      self.ffn.build(out_shape)

      self.built = True

    def compute_output_shape(self, query_shape: Tuple[Optional[int], int, int], key_shape: tuple[Optional[int], int, int]) ->tuple[Optional[int], int, int]:
      out_shape = self.attn.compute_output_shape(query_shape, key_shape)
      out_shape = self.ffn.compute_output_shape(out_shape)
      return out_shape

    def call(self, query: keras.KerasTensor, key: keras.KerasTensor, mask: Optional[keras.KerasTensor] = None, training: bool = False):
      attn_output = self.attn(query, key, attention_mask=mask, training=training)
      x = self.norm1(query + attn_output, training=training)
      ffn_output = self.ffn(x, training=training)
      return self.norm2(x + ffn_output, training=training)

#### Set Attention & Pooling Layers

In [ ]:
@keras.saving.register_keras_serializable()
class SetAttentionBlock(keras.Layer):
    def __init__(
        self,
        embed_dim: Optional[int] = None,
        hidden_dim: Optional[int] = None,
        num_heads: int = 1,
        dropout: float = 0.0,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.dropout = dropout

    def build(self, input_shape: tuple[Optional[int], int, int]) -> None:
      self.mab = MultiheadAttentionBlock(
        embed_dim=self.embed_dim,
        hidden_dim=self.hidden_dim,
        num_heads=self.num_heads,
        dropout=self.dropout,
      )
      self.mab.build(input_shape, input_shape)

      self.built = True

    def compute_output_shape(self, input_shape: tuple[Optional[int], int, int]) -> tuple[Optional[int], int, int]:
      return self.mab.compute_output_shape(input_shape, input_shape)

    def get_config(self) -> dict:
      config = super().get_config()
      config.update({
        "embed_dim": self.embed_dim,
        "hidden_dim": self.hidden_dim,
        "num_heads": self.num_heads,
        "dropout": self.dropout,
      })
      return config

    @classmethod
    def from_config(cls, config: dict) -> "SetAttentionBlock":
      return cls(**config)

    def call(self, inputs: keras.KerasTensor, mask: Optional[keras.KerasTensor] = None, training: bool = False) -> keras.KerasTensor:
      return self.mab(inputs, inputs, mask=mask, training=training)

In [ ]:
@keras.saving.register_keras_serializable()
class InducedSetAttentionBlock(keras.Layer):
    def __init__(
      self,
      num_inductions: int,
      embed_dim: Optional[int] = None,
      hidden_dim: Optional[int] = None,
      num_heads: int = 1,
      dropout: float = 0.0,
      **kwargs
    ):
      super().__init__(**kwargs)

      self.num_inductions = num_inductions

      self.embed_dim = embed_dim
      self.hidden_dim = hidden_dim
      self.num_heads = num_heads
      self.dropout = dropout

    def build(self, inputs_shape: tuple[Optional[int], int, int]) -> None:
      batch_size, num_features, feature_dim = inputs_shape

      self.embed_dim = self.embed_dim if self.embed_dim is not None else feature_dim

      self.mab1 = MultiheadAttentionBlock(
        embed_dim=self.embed_dim,
        hidden_dim=self.hidden_dim,
        num_heads=self.num_heads,
        dropout=self.dropout
      )
      self.mab1.build((batch_size, self.num_inductions, self.embed_dim), inputs_shape)

      self.mab2 = MultiheadAttentionBlock(
        embed_dim=self.embed_dim,
        hidden_dim=self.hidden_dim,
        num_heads=self.num_heads,
        dropout=self.dropout
      )
      self.mab2.build(inputs_shape, self.mab1.compute_output_shape((batch_size, self.num_inductions, self.embed_dim), inputs_shape))

      self.inductions = self.add_weight(
        shape=(1, self.num_inductions, self.embed_dim),
        initializer="glorot_uniform",
        trainable=True,
        name="inducing_points"
      )

    def compute_output_shape(self, inputs_shape: tuple[Optional[int], int, int]) ->  tuple[Optional[int], int, int]:
      batch_size, num_features, feature_dim = inputs_shape
      return (batch_size, num_features, self.embed_dim)

    @classmethod
    def from_config(clas, config: dict) -> "InducedSetAttentionBlock":
      return InducedSetAttentionBlock(**config)

    def get_config(self) -> dict:
      config = super().get_config()
      config.update({
        "num_inductions": self.num_inductions,
        "embed_dim": self.embed_dim,
        "hidden_dim": self.hidden_dim,
        "num_heads": self.num_heads,
        "dropout": self.dropout,
      })
      return config

    def call(self, inputs: keras.KerasTensor, mask: Optional[keras.KerasTensor] = None, training: bool = False):
      batch_size = keras.ops.shape(inputs)[0]

      inducing_points = keras.ops.tile(self.inductions, (batch_size, 1, 1))
      if 'tensorflow' == keras.backend.backend():
        inducing_points = tf.ensure_shape(inducing_points, (None, keras.ops.shape(self.inductions)[1], keras.ops.shape(self.inductions)[2]))

      h = self.mab1(inducing_points, inputs, mask=mask, training=training)

      return self.mab2(inputs, h, mask=mask, training=training)

In [ ]:
@keras.saving.register_keras_serializable()
class PoolingByMultiheadAttention(keras.Layer):
    def __init__(
      self,
      num_seeds: int,
      embed_dim: Optional[int] = None,
      hidden_dim: Optional[int] = None,
      num_heads: int = 1,
      dropout: float = 0.0,
      **kwargs
    ):
      super().__init__(**kwargs)
      self.num_seeds = num_seeds

      self.embed_dim = embed_dim
      self.hidden_dim = hidden_dim
      self.num_heads = num_heads
      self.dropout = dropout

    def build(self, input_shape: tuple[Optional[int], int, int]):
      batch_size, num_features, feature_dim = input_shape

      self.embed_dim = self.embed_dim if self.embed_dim is not None else feature_dim

      self.mab = MultiheadAttentionBlock(
        embed_dim=self.embed_dim,
        hidden_dim=self.hidden_dim,
        num_heads=self.num_heads,
        dropout=self.dropout
      )

      self.seeds = self.add_weight(
        shape=(1, self.num_seeds, feature_dim),
        initializer='glorot_uniform',
        trainable=True,
        name="seed_vectors"
      )

      self.mab.build((batch_size, self.num_seeds, feature_dim), input_shape)

    def compute_output_shape(self, input_shape: tuple[Optional[int], int, int]) -> tuple[Optional[int], int, int]:
      batch_size, num_features, feature_dim = input_shape

      return self.mab.compute_output_shape((batch_size, self.num_seeds, feature_dim), input_shape)

    def get_config(self) -> dict:
      config = super().get_config()
      config.update({"num_seeds": self.num_seeds})
      return config

    @classmethod
    def from_config(cls, config: dict) -> "PoolingByMultiheadAttention":
      return cls(**config)

    def call(self, inputs: keras.KerasTensor, training: bool = False) -> keras.KerasTensor:
      batch_size = tf.shape(inputs)[0]

      seeds = keras.ops.tile(self.seeds, (batch_size, 1, 1))
      if 'tensorflow' == keras.backend.backend():
        seeds = tf.ensure_shape(seeds, (None, keras.ops.shape(self.seeds)[1], keras.ops.shape(self.seeds)[2]))

      return self.mab(seeds, inputs, training=training)

#### Augmentation Layers

In [ ]:
@keras.saving.register_keras_serializable()
class SetAugmentationLayer(keras.Layer):
  def __init__(
    self,
    set_element_mask_apply_prob: float = 1.0,
    set_element_mask_prob: float = 0.1,
    set_shuffle_apply_prob: float = 1.0,
    num_cutmix_apply_prob: float = 1.0,
    num_cutmix_alpha: float = 1.0,
    num_cutmix_beta: Optional[float] = None,
    num_mixup_apply_prob: float = 1.0,
    num_mixup_alpha: float = 0.2,
    num_mixup_beta: Optional[float] = None,
    num_noise_apply_prob: float = 1.0,
    num_noise_mean: float = 0.0,
    num_noise_min_std: float = 0.01,
    num_noise_max_std: float = 0.1,
    num_scaling_apply_prob: float = 1.0,
    num_scaling_min: float = 0.9,
    num_scaling_max: float = 1.1,
    num_feature_mask_apply_prob: float = 1.0,
    num_feature_mask_prob: float = 0.1,
    cat_cutmix_apply_prob: float = 1.0,
    cat_cutmix_alpha: float = 1.0,
    cat_cutmix_beta: Optional[float] = None,
    cat_element_mask_apply_prob: float = 1.0,
    cat_element_mask_prob: float = 0.1,
    cat_feature_mask_apply_prob: float = 1.0,
    cat_feature_mask_prob: float = 0.15,
  **kwargs
  ):
    kwargs.pop("trainable", None)
    super().__init__(trainable=False, **kwargs)

    self.set_element_mask_apply_prob = set_element_mask_apply_prob
    self.set_element_mask_prob = set_element_mask_prob
    self.set_shuffle_apply_prob = set_shuffle_apply_prob

    self.num_cutmix_apply_prob = num_cutmix_apply_prob
    self.num_cutmix_alpha = num_cutmix_alpha
    self.num_cutmix_beta = num_cutmix_beta or num_cutmix_alpha
    self.num_mixup_apply_prob = num_mixup_apply_prob
    self.num_mixup_alpha = num_mixup_alpha
    self.num_mixup_beta = num_mixup_beta or num_mixup_alpha
    self.num_noise_apply_prob = num_noise_apply_prob
    self.num_noise_mean = num_noise_mean
    self.num_noise_min_std = num_noise_min_std
    self.num_noise_max_std = num_noise_max_std
    self.num_scaling_apply_prob = num_scaling_apply_prob
    self.num_scaling_min = num_scaling_min
    self.num_scaling_max = num_scaling_max
    self.num_feature_mask_apply_prob = num_feature_mask_apply_prob
    self.num_feature_mask_prob = num_feature_mask_prob

    self.cat_cutmix_apply_prob = cat_cutmix_apply_prob
    self.cat_cutmix_alpha = cat_cutmix_alpha
    self.cat_cutmix_beta = cat_cutmix_beta or cat_cutmix_alpha
    self.cat_element_mask_apply_prob = cat_element_mask_apply_prob
    self.cat_element_mask_prob = cat_element_mask_prob
    self.cat_feature_mask_apply_prob = cat_feature_mask_apply_prob
    self.cat_feature_mask_prob = cat_feature_mask_prob

  def augment_sets(self, data: keras.KerasTensor) -> keras.KerasTensor:
    return self._augment_sets(
      data,
      element_mask_apply_prob=self.set_element_mask_apply_prob,
      element_mask_prob=self.set_element_mask_prob,
      shuffle_apply_prob=self.set_shuffle_apply_prob,
    )

  def augment_numerical_features(self, data: keras.KerasTensor) -> keras.KerasTensor:
    return self._augment_numerical_features(
      data,
      cutmix_apply_prob=self.num_cutmix_apply_prob,
      cutmix_alpha=self.num_cutmix_alpha,
      cutmix_beta=self.num_cutmix_beta,
      mixup_apply_prob=self.num_mixup_apply_prob,
      mixup_alpha=self.num_mixup_alpha,
      mixup_beta=self.num_mixup_beta,
      noise_apply_prob=self.num_noise_apply_prob,
      noise_mean=self.num_noise_mean,
      noise_min_std=self.num_noise_min_std,
      noise_max_std=self.num_noise_max_std,
      scaling_apply_prob=self.num_scaling_apply_prob,
      scaling_min=self.num_scaling_min,
      scaling_max=self.num_scaling_max,
      feature_mask_apply_prob=self.num_feature_mask_apply_prob,
      feature_mask_prob=self.num_feature_mask_prob,
    )

  def augment_categorical_features(self, data: keras.KerasTensor) -> keras.KerasTensor:
    return self._augment_categorical_features(
      data,
      cutmix_apply_prob=self.cat_cutmix_apply_prob,
      cutmix_alpha=self.cat_cutmix_alpha,
      cutmix_beta=self.cat_cutmix_beta,
      element_mask_apply_prob=self.cat_element_mask_apply_prob,
      element_mask_prob=self.cat_element_mask_prob,
      feature_mask_apply_prob=self.cat_feature_mask_apply_prob,
      feature_mask_prob=self.cat_feature_mask_prob,
    )

  @classmethod
  def _augment_sets(
    cls,
    sets: keras.KerasTensor,
    element_mask_apply_prob: float = 1.0,
    element_mask_prob: float = 0.1,
    shuffle_apply_prob: float = 1.0,
  ) -> keras.KerasTensor:
    batch_size, num_features, feature_dim = keras.ops.shape(sets)

    # Random subset sampling (element-level dropout)
    if element_mask_apply_prob > 0 and element_mask_prob > 0:
      condition = keras.ops.cast(
        keras.ops.less(keras.random.uniform(()), element_mask_apply_prob),
        dtype=keras.ops.dtype(sets)
      )
      # Only generate mask if we might use it
      random_mask = keras.random.uniform((batch_size, num_features, 1))
      element_mask = keras.ops.cast(
        keras.ops.greater(random_mask, element_mask_prob),
        dtype=keras.ops.dtype(sets)
      )

      sets = sets * (condition * element_mask + (1 - condition))

    # Shuffle (enforce permutation invariance)
    if shuffle_apply_prob > 0:
      condition = keras.ops.less(keras.random.uniform(()), shuffle_apply_prob)
      sets = keras.ops.cond(condition, lambda: keras.random.shuffle(sets, axis=1), lambda: sets)

    return sets

  @classmethod
  def _set_cutmix(
    cls,
    data: keras.KerasTensor,
    alpha: float = 1.0,
    beta: Optional[float] = None,
  ) -> keras.KerasTensor:
    """
    CutMix augmentation for sets.

    Replaces a random contiguous chunk of set elements with elements
    from another sample in the batch.

    Args:
        sets: Input sets [batch_size, num_features, feature_dim]
        alpha: Beta distribution parameter
               - 1.0 (default): uniform mixing ratio
               - <1.0: prefer extreme cuts (small or large)
               - >1.0: prefer balanced 50/50 cuts

    Returns:
        Augmented data with same shape

    Example:
        Input Set A: [e1, e2, e3, e4, e5]
        Input Set B: [f1, f2, f3, f4, f5]
        With cut_start=2, cut_size=2:
        Output: [e1, e2, f3, f4, e5]

    Reference:
        Yun et al. "CutMix: Regularization Strategy to Train Strong Classifiers with Localizable Features" (2019)
        https://arxiv.org/abs/1905.04899
    """
    batch_size, num_features, feature_dim = keras.ops.shape(data)

    # Sample mixing ratio: lambda ~ Beta(alpha, beta) - lambda indicates proportion to KEEP from original
    beta = beta or alpha
    lam = keras.random.beta((), alpha, beta)

    # Cut size: proportion to replace (1 - lambda)
    cut_size = keras.ops.cast(
        keras.ops.cast(num_features, keras.ops.dtype(data)) * (1.0 - lam),
        'int32'
    )

    # Ensure valid cut size: at least 1, at most (num_features - 1)
    cut_size = keras.ops.clip(cut_size, 1, num_features - 1)

    # Random cut position
    max_start_pos = num_features - cut_size
    cut_start = keras.random.randint((), 0, max_start_pos + 1)
    cut_end = cut_start + cut_size

    # Create position indices [0, 1, 2, ..., num_features-1]
    # Shape: [num_features]
    positions = keras.ops.arange(num_features, dtype='int32')

    # Create boolean mask for cut region
    # Shape: [num_features] -> broadcasts to [batch_size, num_features, feature_dim]
    mask = (positions >= cut_start) & (positions < cut_end)
    mask = keras.ops.cast(mask, data.dtype)
    mask = keras.ops.reshape(mask, (1, num_features, 1))

    # Shuffle batch indices to get mixing pairs
    shuffled_indices = keras.random.shuffle(keras.ops.arange(batch_size))
    shuffled_data = keras.ops.take(data, shuffled_indices, axis=0)

    # Vectorized blend: use shuffled where mask=1, original where mask=0
    mixed = data * (1.0 - mask) + shuffled_data * mask

    return mixed

  @classmethod
  def _augment_numerical_features(
    cls,
    data: keras.KerasTensor,
    cutmix_apply_prob: float = 1.0,
    cutmix_alpha: float = 1.0,
    cutmix_beta: Optional[float] = None,
    mixup_apply_prob: float = 1.0,
    mixup_alpha: float = 0.2,
    mixup_beta: Optional[float] = None,
    noise_apply_prob: float = 1.0,
    noise_mean: float = 0.0,
    noise_min_std: float = 0.01,
    noise_max_std: float = 0.1,
    scaling_apply_prob: float = 1.0,
    scaling_min: float = 0.9,
    scaling_max: float = 1.1,
    feature_mask_apply_prob: float = 1.0,
    feature_mask_prob: float = 0.1,
  ) -> keras.KerasTensor:

    batch_size, num_features, feature_dim = keras.ops.shape(data)

    # 1. Cut-mix numerical features in the batch
    if cutmix_apply_prob > 0:
      condition = keras.ops.less(keras.random.uniform(()), cutmix_apply_prob)
      data = keras.ops.cond(condition, lambda: cls._set_cutmix(data, cutmix_alpha, cutmix_beta), lambda: data)

    # 2. Mix-up numerical features in the batch
    if mixup_apply_prob > 0:
      condition = keras.ops.less(keras.random.uniform(()), mixup_apply_prob)
      data = keras.ops.cond(condition, lambda: cls._numerical_mixup(data, mixup_alpha, mixup_beta), lambda: data)


    # 3. Gaussian noise (additive noise)
    if noise_apply_prob > 0:
      condition = keras.ops.cast(
        keras.ops.less(keras.random.uniform(()), noise_apply_prob),
        dtype=keras.ops.dtype(data)
      )

      # Generate noise
      noise_std = keras.random.uniform((), noise_min_std, noise_max_std)
      noise = keras.random.normal(keras.ops.shape(data), mean=noise_mean, stddev=noise_std)

      data = data + condition * noise

    # 4. Random scaling / jittering (multiplicative noise)
    if scaling_apply_prob > 0:
      condition = keras.ops.cast(
        keras.ops.less(keras.random.uniform(()), scaling_apply_prob),
        dtype=keras.ops.dtype(data)
      )

      scale_factor = keras.random.uniform(
        (batch_size, num_features, 1),
        scaling_min,
        scaling_max
      )
      data = data * (condition * scale_factor + (1 - condition))

    # 5. Feature-level dropout
    if feature_mask_apply_prob > 0 and feature_mask_prob > 0:
      condition = keras.ops.cast(
        keras.ops.less(keras.random.uniform(()), feature_mask_apply_prob),
        dtype=keras.ops.dtype(data)
      )
      # Only generate mask if we might use it
      random_mask = keras.random.uniform((batch_size, 1, feature_dim))
      feature_mask = keras.ops.cast(
        keras.ops.greater(random_mask, feature_mask_prob),
        dtype=keras.ops.dtype(data)
      )

      data = data * (condition * feature_mask + (1 - condition))

    return data

  @classmethod
  def _numerical_mixup(
    cls,
    data: keras.KerasTensor,
    alpha: float = 0.2,
    beta: Optional[float] = None,
  ) -> keras.KerasTensor:
    """
    MixUp augmentation for numerical features.

    Performs smooth interpolation between pairs of samples in the batch.
    This creates "virtual" training examples that lie between real samples,
    encouraging the model to learn smooth decision boundaries.

    Args:
        data: Numerical features [batch_size, num_features, feature_dim]
        alpha: Beta distribution parameter controlling mixing strength
               - Small alpha (e.g., 0.1): Strong mixing, more extreme ratios
               - Large alpha (e.g., 4.0): Weak mixing, closer to 50/50

    Returns:
        Augmented data with shape as input

    Example:
        Sample A: [5.0, 3.2, 8.1]
        Sample B: [2.0, 7.5, 1.3]
        With lambda=0.3:
        Output: 0.3*A + 0.7*B = [2.9, 6.035, 3.34]

    Reference:
        Zhang et al. "mixup: Beyond Empirical Risk Minimization" (2018)
        https://arxiv.org/abs/1710.09412
    """
    batch_size, num_features, feature_dim = keras.ops.shape(data)

    # Sample mixing coefficient
    beta = beta or alpha
    lam = keras.random.beta((batch_size, 1, 1), alpha, beta)

    # Shuffle batch
    indices = keras.random.shuffle(keras.ops.arange(batch_size))
    shuffled = keras.ops.take(data, indices, axis=0)

    # Mix
    mixed = lam * data + (1 - lam) * shuffled

    return mixed

  @classmethod
  def _augment_categorical_features(
    cls,
    data: keras.KerasTensor,
    cutmix_apply_prob: float = 1.0,
    cutmix_alpha: float = 1.0,
    cutmix_beta: Optional[float] = None,
    element_mask_apply_prob: float = 1.0,
    element_mask_prob: float = 0.1,
    feature_mask_apply_prob: float = 1.0,
    feature_mask_prob: float = 0.15,
  ) -> keras.KerasTensor:
    batch_size, num_features, feature_dim = keras.ops.shape(data)

    # 1. Cut-mix numerical features in the batch
    if cutmix_apply_prob > 0:
      condition = keras.ops.less(keras.random.uniform(()), cutmix_apply_prob)
      data = keras.ops.cond(condition, lambda: cls._set_cutmix(data, cutmix_alpha, cutmix_beta), lambda: data)

    # 2. Element-level dropout: randomly zero out entire categorical embeddings
    if element_mask_apply_prob > 0 and element_mask_prob > 0:
      condition = keras.ops.cast(
        keras.ops.less(keras.random.uniform(()), element_mask_apply_prob),
        dtype=keras.ops.dtype(data)
      )
      # Only generate mask if we might use it
      random_mask = keras.random.uniform((batch_size, num_features, 1))
      element_mask = keras.ops.cast(
        keras.ops.greater(random_mask, element_mask_prob),
        dtype=keras.ops.dtype(data)
      )

      data = data * (condition * element_mask + (1 - condition))

    # 3. Feature-level dropout: randomly mask specific embedding dimensions
    if feature_mask_apply_prob > 0 and feature_mask_prob > 0:
      condition = keras.ops.cast(
        keras.ops.less(keras.random.uniform(()), feature_mask_apply_prob),
        dtype=keras.ops.dtype(data)
      )
      # Only generate mask if we might use it
      random_mask = keras.random.uniform((batch_size, 1, feature_dim))
      feature_mask = keras.ops.cast(
        keras.ops.greater(random_mask, feature_mask_prob),
        dtype=keras.ops.dtype(data)
      )

      data = data * (condition * feature_mask + (1 - condition))

    return data

  def build(self, cat_inputs_shape: tuple[Optional[int], int, int], num_inputs_shape: tuple[Optional[int], int, int]) -> None:
    self.built = True

  def compute_output_shape(self, cat_inputs_shape: tuple[Optional[int], int, int], num_inputs_shape: tuple[Optional[int], int, int]) -> tuple[Optional[int], int, int]:
    if len(cat_inputs_shape) != 3 or len(cat_inputs_shape) != 3:
      raise ValueError(f"Input tensors must be 3-dimensional, but are {len(cat_inputs_shape)} and {len(num_inputs_shape)}")
    elif cat_inputs_shape[0] != num_inputs_shape[0]:
      raise ValueError(f"Input tensors must have the same batch size, but have {cat_inputs_shape[0]} and {num_inputs_shape[0]}")
    elif cat_inputs_shape[2] != num_inputs_shape[2]:
      raise ValueError(f"Input tensors must have the same feature dimension, but have {cat_inputs_shape[2]} and {num_inputs_shape[2]}")

    return (cat_inputs_shape[0], cat_inputs_shape[1] + num_inputs_shape[1], cat_inputs_shape[2])

  def get_config(self) -> dict:
    config = super().get_config()
    config.update({
      "set_element_mask_apply_prob": self.set_element_mask_apply_prob,
      "set_element_mask_prob": self.set_element_mask_prob,
      "set_shuffle_apply_prob": self.set_shuffle_apply_prob,
      "num_cutmix_apply_prob": self.num_cutmix_apply_prob,
      "num_cutmix_alpha": self.num_cutmix_alpha,
      "num_cutmix_beta": self.num_cutmix_beta,
      "num_mixup_apply_prob": self.num_mixup_apply_prob,
      "num_mixup_alpha": self.num_mixup_alpha,
      "num_mixup_beta": self.num_mixup_beta,
      "num_noise_apply_prob": self.num_noise_apply_prob,
      "num_noise_mean": self.num_noise_mean,
      "num_noise_min_std": self.num_noise_min_std,
      "num_noise_max_std": self.num_noise_max_std,
      "num_scaling_apply_prob": self.num_scaling_apply_prob,
      "num_scaling_min": self.num_scaling_min,
      "num_scaling_max": self.num_scaling_max,
      "num_feature_mask_apply_prob": self.num_feature_mask_apply_prob,
      "num_feature_mask_prob": self.num_feature_mask_prob,
      "cat_cutmix_apply_prob": self.cat_cutmix_apply_prob,
      "cat_cutmix_alpha": self.cat_cutmix_alpha,
      "cat_cutmix_beta": self.cat_cutmix_beta,
      "cat_element_mask_apply_prob": self.cat_element_mask_apply_prob,
      "cat_element_mask_prob": self.cat_element_mask_prob,
      "cat_feature_mask_apply_prob": self.cat_feature_mask_apply_prob,
      "cat_feature_mask_prob": self.cat_feature_mask_prob,
    })
    return config

  @classmethod
  def from_config(cls, config: dict) -> "SetAugmentationLayer":
    return cls(**config)

  @tf.custom_gradient
  def call(self, cat_inputs: keras.KerasTensor, num_inputs: keras.KerasTensor) -> keras.KerasTensor:
    x_num = self.augment_numerical_features(num_inputs)
    x_cat = self.augment_categorical_features(cat_inputs)

    x = keras.ops.concatenate([x_cat, x_num], axis=1)

    x = self.augment_sets(x)

    def grad(upstream: keras.KerasTensor) -> tuple[keras.KerasTensor, keras.KerasTensor]:
      return upstream[:,:keras.ops.shape(cat_inputs)[1],:], upstream[:,keras.ops.shape(cat_inputs)[1]:,:]
      #return keras.ops.zeros_like(upstream[:,:keras.ops.shape(cat_inputs)[1],:]), keras.ops.zeros_like(upstream[:,keras.ops.shape(cat_inputs)[1]:,:])

    return x, grad

In [ ]:
WEAK_SET_AUGMENTATION = {
    # Set-level
    'set_element_mask_apply_prob': 0.3,
    'set_element_mask_prob': 0.05,
    'set_shuffle_apply_prob': 0.0,

    # Numerical
    'num_cutmix_apply_prob': 0.2,
    'num_cutmix_alpha': 0.8,
    'num_cutmix_beta': 0.8,
    'num_mixup_apply_prob': 0.2,
    'num_mixup_alpha': 0.8,
    'num_mixup_beta': 0.8,
    'num_noise_apply_prob': 0.3,
    'num_noise_mean': 0.0,
    'num_noise_min_std': 0.005,
    'num_noise_max_std': 0.02,
    'num_scaling_apply_prob': 0.3,
    'num_scaling_min': 0.95,
    'num_scaling_max': 1.05,
    'num_feature_mask_apply_prob': 0.3,
    'num_feature_mask_prob': 0.05,

    # Categorical
    'cat_cutmix_apply_prob': 0.2,
    'cat_cutmix_alpha': 0.8,
    'cat_cutmix_beta': 0.8,
    'cat_element_mask_apply_prob': 0.3,
    'cat_element_mask_prob': 0.05,
    'cat_feature_mask_apply_prob': 0.3,
    'cat_feature_mask_prob': 0.05,
}

WEAK_SET_AUGMENTATION_NOMIX = {
    # Set-level
    'set_element_mask_apply_prob': 0.3,
    'set_element_mask_prob': 0.05,
    'set_shuffle_apply_prob': 0.0,

    # Numerical
    'num_cutmix_apply_prob': 0.0,
    'num_mixup_apply_prob': 0.0,

    'num_noise_apply_prob': 0.3,
    'num_noise_mean': 0.0,
    'num_noise_min_std': 0.005,
    'num_noise_max_std': 0.02,
    'num_scaling_apply_prob': 0.3,
    'num_scaling_min': 0.95,
    'num_scaling_max': 1.05,
    'num_feature_mask_apply_prob': 0.3,
    'num_feature_mask_prob': 0.05,

    # Categorical
    'cat_cutmix_apply_prob': 0.0,

    'cat_element_mask_apply_prob': 0.3,
    'cat_element_mask_prob': 0.05,
    'cat_feature_mask_apply_prob': 0.3,
    'cat_feature_mask_prob': 0.05,
}

In [ ]:
MEDIUM_SET_AUGMENTATION = {
    # Set-level
    'set_element_mask_apply_prob': 0.7,
    'set_element_mask_prob': 0.15,
    'set_shuffle_apply_prob': 0.0,

    # Numerical
    'num_cutmix_apply_prob': 0.6,
    'num_cutmix_alpha': 0.4,
    'num_cutmix_beta': 0.4,
    'num_mixup_apply_prob': 0.6,
    'num_mixup_alpha': 0.4,
    'num_mixup_beta': 0.4,
    'num_noise_apply_prob': 0.6,
    'num_noise_mean': 0.0,
    'num_noise_min_std': 0.01,
    'num_noise_max_std': 0.05,
    'num_scaling_apply_prob': 0.6,
    'num_scaling_min': 0.9,
    'num_scaling_max': 1.1,
    'num_feature_mask_apply_prob': 0.7,
    'num_feature_mask_prob': 0.1,

    # Categorical
    'cat_cutmix_apply_prob': 0.5,
    'cat_cutmix_alpha': 0.4,
    'cat_cutmix_beta': 0.4,
    'cat_element_mask_apply_prob': 0.6,
    'cat_element_mask_prob': 0.1,
    'cat_feature_mask_apply_prob': 0.6,
    'cat_feature_mask_prob': 0.1,
}

MEDIUM_SET_AUGMENTATION_NOMIX = {
    # Set-level
    'set_element_mask_apply_prob': 0.7,
    'set_element_mask_prob': 0.15,
    'set_shuffle_apply_prob': 0.0,

    # Numerical
    'num_cutmix_apply_prob': 0.0,
    'num_mixup_apply_prob': 0.0,

    'num_noise_apply_prob': 0.6,
    'num_noise_mean': 0.0,
    'num_noise_min_std': 0.01,
    'num_noise_max_std': 0.05,
    'num_scaling_apply_prob': 0.6,
    'num_scaling_min': 0.9,
    'num_scaling_max': 1.1,
    'num_feature_mask_apply_prob': 0.7,
    'num_feature_mask_prob': 0.1,

    # Categorical
    'cat_cutmix_apply_prob': 0.0,

    'cat_element_mask_apply_prob': 0.6,
    'cat_element_mask_prob': 0.1,
    'cat_feature_mask_apply_prob': 0.6,
    'cat_feature_mask_prob': 0.1,
}

In [ ]:
STRONG_SET_AUGMENTATION = {
    # Set-level
    'set_element_mask_apply_prob': 0.9,
    'set_element_mask_prob': 0.25,
    'set_shuffle_apply_prob': 0.0,

    # Numerical
    'num_cutmix_apply_prob': 0.8,
    'num_cutmix_alpha': 0.2,
    'num_cutmix_beta': 0.2,
    'num_mixup_apply_prob': 0.8,
    'num_mixup_alpha': 0.2,
    'num_mixup_beta': 0.2,
    'num_noise_apply_prob': 0.8,
    'num_noise_mean': 0.0,
    'num_noise_min_std': 0.03,
    'num_noise_max_std': 0.1,
    'num_scaling_apply_prob': 0.8,
    'num_scaling_min': 0.8,
    'num_scaling_max': 1.2,
    'num_feature_mask_apply_prob': 0.9,
    'num_feature_mask_prob': 0.2,

    # Categorical
    'cat_cutmix_apply_prob': 0.8,
    'cat_cutmix_alpha': 0.2,
    'cat_cutmix_beta': 0.2,
    'cat_element_mask_apply_prob': 0.8,
    'cat_element_mask_prob': 0.2,
    'cat_feature_mask_apply_prob': 0.8,
    'cat_feature_mask_prob': 0.2,
}

STRONG_SET_AUGMENTATION_NOMIX = {
    # Set-level
    'set_element_mask_apply_prob': 0.9,
    'set_element_mask_prob': 0.25,
    'set_shuffle_apply_prob': 0.0,

    # Numerical
    'num_cutmix_apply_prob': 0.0,
    'num_mixup_apply_prob': 0.0,

    'num_noise_apply_prob': 0.8,
    'num_noise_mean': 0.0,
    'num_noise_min_std': 0.03,
    'num_noise_max_std': 0.1,
    'num_scaling_apply_prob': 0.8,
    'num_scaling_min': 0.8,
    'num_scaling_max': 1.2,
    'num_feature_mask_apply_prob': 0.9,
    'num_feature_mask_prob': 0.2,

    # Categorical
    'cat_cutmix_apply_prob': 0.0,

    'cat_element_mask_apply_prob': 0.8,
    'cat_element_mask_prob': 0.2,
    'cat_feature_mask_apply_prob': 0.8,
    'cat_feature_mask_prob': 0.2,
}

In [ ]:
COMPONENT_WISE_ASYMMETRIC_SET_AUGMENTATIONS = {
    'view1': {
        # Focus on interpolation-based augmentations
        'set_element_mask_apply_prob': 0.5,
        'set_element_mask_prob': 0.1,
        'set_shuffle_apply_prob': 0.0,

        'num_cutmix_apply_prob': 0.9,
        'num_cutmix_alpha': 0.3,
        'num_mixup_apply_prob': 0.9,
        'num_mixup_alpha': 0.3,
        'num_noise_apply_prob': 0.2,
        'num_noise_max_std': 0.02,
        'num_feature_mask_apply_prob': 0.3,
        'num_feature_mask_prob': 0.05,

        'cat_cutmix_apply_prob': 0.8,
        'cat_element_mask_apply_prob': 0.4,
    },
    'view2': {
        # Focus on corruption-based augmentations
        'set_element_mask_apply_prob': 0.8,
        'set_element_mask_prob': 0.2,
        'set_shuffle_apply_prob': 0.0,

        'num_cutmix_apply_prob': 0.3,
        'num_cutmix_alpha': 0.6,
        'num_mixup_apply_prob': 0.3,
        'num_mixup_alpha': 0.6,
        'num_noise_apply_prob': 0.8,
        'num_noise_max_std': 0.08,
        'num_feature_mask_apply_prob': 0.8,
        'num_feature_mask_prob': 0.15,

        'cat_cutmix_apply_prob': 0.3,
        'cat_element_mask_apply_prob': 0.8,
    }
}

#### Base Set Transformer

In [ ]:
@keras.saving.register_keras_serializable()
class SetTransformer(keras.Model, LayerNameMixin, ConfigSerializerMixin):
  def __init__(
    self,
    num_inductions: int,
    num_seeds: int = 1,
    embed_dim: Optional[int] = None,
    hidden_dim: Optional[int] = None,
    output_dim: Optional[int] = None,
    encoder_num_heads: int = 1,
    num_encoder_blocks: int = 2,
    encoder_dropout: float = 0.0,
    pooling_dropout: float = 0.0,
    decoder_num_heads: int = 2,
    num_decoder_blocks: int = 2,
    decoder_dropout: float = 0.0,
    flatten: bool = True,
    **kwargs
  ):
    super().__init__(**kwargs)

    self.num_inductions = num_inductions
    self.num_seeds = num_seeds

    self.encoder_num_heads = encoder_num_heads
    self.num_encoder_blocks = num_encoder_blocks
    self.encoder_dropout = encoder_dropout

    self.pooling_dropout = pooling_dropout

    self.decoder_num_heads = decoder_num_heads
    self.num_decoder_blocks = num_decoder_blocks
    self.decoder_dropout = decoder_dropout

    self.embed_dim = embed_dim
    self.hidden_dim = hidden_dim
    self.output_dim = embed_dim if output_dim is None else output_dim

    self.flatten = flatten

  def build(self, input_shape: tuple[Optional[int], int, int]) -> None:
    batch_size, num_features, feature_dim = input_shape

    self.embed_dim = self.embed_dim if self.embed_dim is not None else feature_dim
    self.output_dim = self.output_dim if self.output_dim is not None else self.embed_dim

    self.input_proj = keras.Sequential([
      keras.layers.Dense(self.embed_dim),
      keras.layers.LayerNormalization(),
    ], name="input_proj")

    self.input_proj.build(input_shape)

    self.encoder = keras.Sequential([
      InducedSetAttentionBlock(
        embed_dim=self.embed_dim,
        hidden_dim=self.hidden_dim,
        num_inductions=self.num_inductions,
        num_heads=self.encoder_num_heads,
        dropout=self.encoder_dropout,
      ) for _ in range(self.num_encoder_blocks)
    ], name="isab_encoder")
    self.encoder.add(keras.layers.Identity())
    self.encoder.build((batch_size, num_features, self.embed_dim))

    self.pooling = PoolingByMultiheadAttention(
      embed_dim=self.embed_dim,
      hidden_dim=self.hidden_dim,
      num_seeds=self.num_seeds,
      num_heads=self.encoder_num_heads,
      name="mha_pooling",
      dropout=self.pooling_dropout,
    )
    self.pooling.build((batch_size, num_features, self.embed_dim))

    self.decoder = keras.Sequential([
      SetAttentionBlock(
        embed_dim=self.embed_dim,
        hidden_dim=self.hidden_dim,
        num_heads=self.decoder_num_heads,
        dropout=self.decoder_dropout,
      ) for _ in range(self.num_decoder_blocks)
    ], name="sab_secoder")
    self.decoder.add(keras.layers.Identity())
    self.decoder.build((batch_size, self.num_seeds, self.embed_dim))

    self.output_proj = keras.layers.Dense(self.output_dim, name="output_proj") if self.output_dim != self.embed_dim else keras.layers.Identity()
    self.output_proj.build((batch_size, self.num_seeds, self.embed_dim))

    self.flattener = keras.layers.Flatten() if self.flatten else keras.layers.Identity()
    self.flattener.build((batch_size, self.num_seeds, self.output_dim))

  def compute_output_shape(self, input_shape: tuple[Optional[int], int, int]) -> tuple[Optional[int], int, int]:
    self.batch_size, self.num_features, self.feature_dim = input_shape
    output_shape = self.input_proj.compute_output_shape(input_shape)
    output_shape = self.encoder.compute_output_shape(output_shape)
    output_shape = self.pooling.compute_output_shape(output_shape)
    output_shape = self.decoder.compute_output_shape(output_shape)
    output_shape = self.output_proj.compute_output_shape(output_shape)
    output_shape = self.flattener.compute_output_shape(output_shape)
    return output_shape

  def get_config(self) -> dict:
    config = super().get_config()
    config.update({
      "num_inductions": self.num_inductions,
      "num_seeds": self.num_seeds,
      "encoder_num_heads": self.encoder_num_heads,
      "num_encoder_blocks": self.num_encoder_blocks,
      "encoder_dropout": self.encoder_dropout,
      "pooling_dropout": self.pooling_dropout,
      "decoder_num_heads": self.decoder_num_heads,
      "num_decoder_blocks": self.num_decoder_blocks,
      "decoder_dropout": self.decoder_dropout,
      "embed_dim": self.embed_dim,
      "hidden_dim": self.hidden_dim,
      "output_dim": self.output_dim,
      "flatten": self.flatten,
    })
    config = self._serialize_config(config)
    return config

  @classmethod
  def from_config(cls, config: dict):
    return cls(**config)

  def build_graph(self, batch_size: int, num_features: int, feature_dim: int) -> keras.Model:
    input_shape = (batch_size, num_features, feature_dim)
    inputs = keras.layers.Input(shape=input_shape[1:], batch_size=input_shape[0])
    self.build(inputs.shape)
    return keras.Model(inputs=[inputs], outputs=self.call(inputs), name=self.name)

  def call(self, inputs: keras.KerasTensor, training: bool = False) -> keras.KerasTensor:
    x = self.input_proj(inputs, training=training)
    x = self.encoder(x, training=training)
    x = self.pooling(x, training=training)
    x = self.decoder(x, training=training)
    x = self.output_proj(x, training=training)
    x = self.flattener(x)
    return x

In [ ]:
st = SetTransformer(
  num_inductions=1,
  num_seeds=1,
  encoder_num_heads=2,
)

model = st.build_graph(32, 10, 64)
model.summary()

#### Contrastive SetTransformer

In [ ]:
@keras.saving.register_keras_serializable(package="custom")
class ContrastiveProcessSetTransformer(keras.Model, LayerNameMixin):
  def __init__(
    self,
    input_encoder: InputEncoder,
    embedding_model: SetTransformer,
    augmentation_1: SetAugmentationLayer,
    augmentation_2: Optional[SetAugmentationLayer] = None,
    loss: keras.Loss = BarlowTwinsLoss(lambda_param=1e-4),

    projection_dim: int = 128,
    projection_hidden_dims: Union[int, List[int]] = [512, 256],
    projection_hidden_activation: str = "relu",
    projection_dropout: float = 0.1,
    debug: bool = False,
    **kwargs
  ):
      super().__init__(**kwargs)

      self.input_encoder = input_encoder
      self.set_transformer = embedding_model

      self.augmentation_1 = augmentation_1
      self.augmentation_2 = augmentation_2 or augmentation_1

      if isinstance(projection_hidden_dims, int):
        projection_hidden_dims = [projection_hidden_dims]
      self.projection_hidden_dims = projection_hidden_dims
      self.projection_hidden_activation = projection_hidden_activation
      self.projection_dim = projection_dim
      self.projection_dropout = projection_dropout

      self.projection_head = self._build_projection_head()
      self.debug = debug

      lr_scheduler = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.00001,
        decay_steps=2000,
        alpha=0.0001,
        warmup_target=0.0001,
        warmup_steps=200,
      )

      self.optimizer = keras.optimizers.AdamW(learning_rate=lr_scheduler, weight_decay=0.004)#, clipnorm=3.0) #global_clipnorm=2.0)

      self.default_callbacks = [
        keras.callbacks.TerminateOnNaN(),
        #keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=5, mode='min', factor=0.5, min_lr=DEFAULT_MIN_LEARNING_RATE, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, mode='min', restore_best_weights=True, verbose=1),
        keras.callbacks.BackupAndRestore(os.path.join(MODEL_BACKUP_DIR, self.name), save_freq='epoch', double_checkpoint=True, delete_checkpoint=False),
        keras.callbacks.ModelCheckpoint(filepath=os.path.join(MODEL_CHECKPOINT_DIR, f"{self.name}.keras"), save_best_only=True, save_weights_only=False, monitor='val_loss', mode='auto', save_freq='epoch', verbose=1),
        keras.callbacks.CSVLogger(filename=os.path.join(OUTPUT_DATA_DIR, f"{self.name}_history.csv"), separator=";", append=True),
        keras.callbacks.TensorBoard(log_dir=os.path.join(OUTPUT_DATA_DIR, f"{self.name}_tensorboard"), update_freq='epoch', histogram_freq=5, embeddings_freq=5, write_graph=True, write_images=True)
      ]

      self.default_loss = loss
      self.loss_tracker = keras.metrics.Mean(name="loss")
      self.alignment_tracker = Alignment(l2_normalize=True)
      self.uniformity_tracker = Uniformity()
      self.effective_rank_tracker = EffectiveRank()

  def build(self, input_shape: tuple[int, int, int]) -> None:
    batch_size = input_shape[0]

    self.input_encoder.build(input_shape)
    output_shape_cat, output_shape_num = self.input_encoder.compute_output_shape(input_shape)

    self.augmentation_1.build(output_shape_cat, output_shape_num)
    self.augmentation_2.build(output_shape_cat, output_shape_num)
    output_shape = self.augmentation_1.compute_output_shape(output_shape_cat, output_shape_num)

    self.set_transformer.build(output_shape)
    output_shape = self.set_transformer.compute_output_shape(output_shape)

    self.projection_head.build(output_shape)
    output_shape = self.projection_head.compute_output_shape(output_shape)

    self.built = True

  def compute_output_shape(self, input_shape: tuple[int, int, int]) -> tuple[int, int, int]:
    batch_size = input_shape[0]

    output_shape_cat, output_shape_num = self.input_encoder.compute_output_shape(input_shape)
    output_shape = (batch_size, output_shape_cat[1] + output_shape_num[1], self.set_transformer.embed_dim)
    output_shape = self.set_transformer.compute_output_shape(output_shape)
    output_shape = self.projection_head.compute_output_shape(output_shape)
    return output_shape

  @property
  def callbacks(self) -> List[keras.callbacks.Callback]:
    return self.default_callbacks

  @property
  def metrics(self) -> List[keras.Metric]:
    return [self.loss_tracker, self.alignment_tracker, self.uniformity_tracker, self.effective_rank_tracker]

  def train_step(self, data: keras.KerasTensor) -> dict:
    # Unpack the data
    sets = data[0] if isinstance(data, tuple) else data

    # Create two augmented views of each set
    with tf.GradientTape() as tape:
      cat_attrs, num_attrs = self.input_encoder(sets, training=True)

      augmented_sets_1 = self.augmentation_1(cat_attrs, num_attrs, training=True)
      augmented_sets_2 = self.augmentation_2(cat_attrs, num_attrs, training=True)

      # Get representations for both views
      projections_1 = self(augmented_sets_1, training=True)
      projections_2 = self(augmented_sets_2, training=True)

      # Compute contrastive loss
      loss = keras.ops.mean(self.default_loss(projections_1, projections_2))

    # Compute gradients and update weights
    trainable_vars = self.trainable_variables
    gradients = tape.gradient(loss, trainable_vars)
    self.optimizer.apply_gradients(zip(gradients, trainable_vars))

    # Update metrics
    self.loss_tracker.update_state(loss)
    self.effective_rank_tracker.update_state(keras.ops.concatenate([projections_1, projections_2], axis=0))
    self.alignment_tracker.update_state(projections_1, projections_2)
    self.uniformity_tracker.update_state(keras.ops.concatenate([projections_1, projections_2], axis=0))
    grad_norm = keras.ops.sqrt(sum(keras.ops.sum(keras.ops.square(g)) for g in gradients))

    return {
      "loss": self.loss_tracker.result(),
      "alignment": self.alignment_tracker.result(),
      "uniformity": self.uniformity_tracker.result(),
      "effective_rank": self.effective_rank_tracker.result(),
      "gradient_norm": grad_norm,
    }

  def test_step(self, data: keras.KerasTensor) -> dict:
    # Unpack the data
    sets = data[0] if isinstance(data, tuple) else data

    # Create two augmented views of each set
    cat_attrs, num_attrs = self.input_encoder(sets, training=False)

    augmented_sets_1 = self.augmentation_1(cat_attrs, num_attrs, training=False)
    augmented_sets_2 = self.augmentation_2(cat_attrs, num_attrs, training=False)

    projections_1 = self(augmented_sets_1, training=False)
    projections_2 = self(augmented_sets_2, training=False)

    loss = self.default_loss(projections_1, projections_2)

    # Update metrics
    self.loss_tracker.update_state(loss)
    self.effective_rank_tracker.update_state(keras.ops.concatenate([projections_1, projections_2], axis=0))
    self.alignment_tracker.update_state(projections_1, projections_2)
    self.uniformity_tracker.update_state(keras.ops.concatenate([projections_1, projections_2], axis=0))

    return {
      "loss": self.loss_tracker.result(),
      "alignment": self.alignment_tracker.result(),
      "uniformity": self.uniformity_tracker.result(),
      "effective_rank": self.effective_rank_tracker.result(),
    }

  def _build_projection_head(self, name:str ="projection") -> keras.Layer:
    projection_head = keras.Sequential([], name=name)
    for dim in self.projection_hidden_dims:
      projection_head.add(keras.layers.Dense(dim, activation=self.projection_hidden_activation))
      projection_head.add(keras.layers.Dropout(self.projection_dropout))

    projection_head.add(keras.layers.Dense(self.projection_dim, use_bias=False))
    return projection_head

  def build_embedding_model(self, trainable: bool = False) -> keras.Layer:
    ie = self.input_encoder.build_graph()
    embed_model = keras.Model(inputs=ie.inputs, outputs=self.set_transformer(keras.ops.concatenate(ie.outputs, axis=1)))

    embed_model.trainable = trainable
    return embed_model

  @classmethod
  def from_config(cls, config: dict) -> "ContrastiveProcessSetTransformer":
    return cls(
      input_encoder=InputEncoder.from_config(config.pop('input_encoder')),
      embedding_model=SetTransformer.from_config(config.pop('embedding_model')),
      augmentation_1=SetAugmentationLayer.from_config(config.pop('augmentation_1')),
      augmentation_2=SetAugmentationLayer.from_config(config.pop('augmentation_2')) if config.get('augmentation_2') else config.pop('augmentation_2'),
      loss=keras.saving.deserialize_keras_object(config.pop("loss")),
      **config
    )

  def get_config(self) -> dict:
    config = super().get_config()
    config.update({
      "input_encoder": self.input_encoder.get_config(),
      "embedding_model": self.set_transformer.get_config(),
      "augmentation_1": self.augmentation_1.get_config(),
      "augmentation_2": self.augmentation_2.get_config(),
      "loss": keras.saving.serialize_keras_object(self.default_loss),
      "projection_dim": self.projection_dim,
      "projection_hidden_dims": self.projection_hidden_dims,
      "projection_hidden_activation": self.projection_hidden_activation,
      "projection_dropout": self.projection_dropout,
      "debug": self.debug,
    })
    return config

  def call(self, inputs: keras.KerasTensor, training: bool = False) -> keras.KerasTensor:
    x = self.set_transformer(inputs, training=training)
    x = self.projection_head(x, training=training)
    return x

### TabTransformer

#### Augmentation Layers

In [ ]:
class TabularAugmentationLayer(keras.Layer):
  def __init__(
    self,
    # Feature-level augmentations (preserve column order)
    num_feature_mask_apply_prob: float = 1.0,
    num_feature_mask_prob: float = 0.1,
    cat_feature_mask_apply_prob: float = 1.0,
    cat_feature_mask_prob: float = 0.15,

    # Batch mixing augmentations (safe for ordered data)
    num_cutmix_apply_prob: float = 1.0,
    num_cutmix_alpha: float = 1.0,
    num_cutmix_beta: Optional[float] = None,
    num_mixup_apply_prob: float = 1.0,
    num_mixup_alpha: float = 0.2,
    num_mixup_beta: Optional[float] = None,

    # Noise-based augmentations (preserve order)
    num_noise_apply_prob: float = 1.0,
    num_noise_mean: float = 0.0,
    num_noise_min_std: float = 0.01,
    num_noise_max_std: float = 0.1,
    num_scaling_apply_prob: float = 1.0,
    num_scaling_min: float = 0.9,
    num_scaling_max: float = 1.1,

    # Categorical augmentations
    cat_cutmix_apply_prob: float = 1.0,
    cat_cutmix_alpha: float = 1.0,
    cat_cutmix_beta: Optional[float] = None,
    **kwargs
  ):
    kwargs.pop('trainable', None)
    super().__init__(trainable=False, **kwargs)

    # Feature masking (preserves order)
    self.num_feature_mask_apply_prob = num_feature_mask_apply_prob
    self.num_feature_mask_prob = num_feature_mask_prob
    self.cat_feature_mask_apply_prob = cat_feature_mask_apply_prob
    self.cat_feature_mask_prob = cat_feature_mask_prob

    # Batch mixing (safe for ordered data)
    self.num_cutmix_apply_prob = num_cutmix_apply_prob
    self.num_cutmix_alpha = num_cutmix_alpha
    self.num_cutmix_beta = num_cutmix_beta or num_cutmix_alpha
    self.num_mixup_apply_prob = num_mixup_apply_prob
    self.num_mixup_alpha = num_mixup_alpha
    self.num_mixup_beta = num_mixup_beta or num_mixup_alpha

    self.cat_cutmix_apply_prob = cat_cutmix_apply_prob
    self.cat_cutmix_alpha = cat_cutmix_alpha
    self.cat_cutmix_beta = cat_cutmix_beta or cat_cutmix_alpha

    # Noise and scaling (preserve order)
    self.num_noise_apply_prob = num_noise_apply_prob
    self.num_noise_mean = num_noise_mean
    self.num_noise_min_std = num_noise_min_std
    self.num_noise_max_std = num_noise_max_std
    self.num_scaling_apply_prob = num_scaling_apply_prob
    self.num_scaling_min = num_scaling_min
    self.num_scaling_max = num_scaling_max

  def augment_numerical_features(self, data: keras.KerasTensor) -> keras.KerasTensor:
    """Augment tabular data while preserving row order."""
    return self._augment_numerical_features(
      data,
      feature_mask_apply_prob=self.num_feature_mask_apply_prob,
      feature_mask_prob=self.num_feature_mask_prob,
      cutmix_apply_prob=self.num_cutmix_apply_prob,
      cutmix_alpha=self.num_cutmix_alpha,
      cutmix_beta=self.num_cutmix_beta,
      mixup_apply_prob=self.num_mixup_apply_prob,
      mixup_alpha=self.num_mixup_alpha,
      mixup_beta=self.num_mixup_beta,
      noise_apply_prob=self.num_noise_apply_prob,
      noise_mean=self.num_noise_mean,
      noise_min_std=self.num_noise_min_std,
      noise_max_std=self.num_noise_max_std,
      scaling_apply_prob=self.num_scaling_apply_prob,
      scaling_min=self.num_scaling_min,
      scaling_max=self.num_scaling_max,
    )

  def augment_categorical_features(self, data: keras.KerasTensor) -> keras.KerasTensor:
    """Augment categorical features for tabular data."""
    return self._augment_categorical_features(
      data,
      cutmix_apply_prob=self.cat_cutmix_apply_prob,
      cutmix_alpha=self.cat_cutmix_alpha,
      cutmix_beta=self.cat_cutmix_beta,
      feature_mask_apply_prob=self.cat_feature_mask_apply_prob,
      feature_mask_prob=self.cat_feature_mask_prob,
    )

  @classmethod
  def _augment_numerical_features(
    cls,
    data: keras.KerasTensor,
    feature_mask_apply_prob: float = 1.0,
    feature_mask_prob: float = 0.1,
    cutmix_apply_prob: float = 1.0,
    cutmix_alpha: float = 1.0,
    cutmix_beta: Optional[float] = None,
    mixup_apply_prob: float = 1.0,
    mixup_alpha: float = 0.2,
    mixup_beta: Optional[float] = None,
    noise_apply_prob: float = 1.0,
    noise_mean: float = 0.0,
    noise_min_std: float = 0.01,
    noise_max_std: float = 0.1,
    scaling_apply_prob: float = 1.0,
    scaling_min: float = 0.9,
    scaling_max: float = 1.1,
  ) -> keras.KerasTensor:
    batch_size, num_features, feature_dim = keras.ops.shape(data)

    # 1. Feature-level dropout (preserves order)
    if feature_mask_apply_prob > 0 and feature_mask_prob > 0:
      condition = keras.ops.cast(
        keras.ops.less(keras.random.uniform(()), feature_mask_apply_prob),
        dtype=keras.ops.dtype(data)
      )
      # Only generate mask if we might use it
      random_mask = keras.random.uniform((batch_size, 1, feature_dim))
      feature_mask = keras.ops.cast(
        keras.ops.greater(random_mask, feature_mask_prob),
        dtype=keras.ops.dtype(data)
      )

      data = data * (condition * feature_mask + (1 - condition))

    # 2. CutMix across batch dimension (safe for ordered columns)
    if cutmix_apply_prob > 0:
      condition = keras.ops.less(keras.random.uniform(()), cutmix_apply_prob)
      data = keras.ops.cond(condition, lambda: cls._tabular_cutmix(data, cutmix_alpha, cutmix_beta), lambda: data)

    # 3. MixUp across batch dimension (safe for ordered columns)
    if mixup_apply_prob > 0:
      condition = keras.ops.less(keras.random.uniform(()), mixup_apply_prob)
      data = keras.ops.cond(condition, lambda: cls._numerical_mixup(data, mixup_alpha, mixup_beta), lambda: data)

    # 4. Gaussian noise
    if noise_apply_prob > 0:
      condition = keras.ops.cast(
        keras.ops.less(keras.random.uniform(()), noise_apply_prob),
        dtype=keras.ops.dtype(data)
      )

      # Generate noise
      noise_std = keras.random.uniform((), noise_min_std, noise_max_std)
      noise = keras.random.normal(keras.ops.shape(data), mean=noise_mean, stddev=noise_std)

      data = data + condition * noise

    # 4. Random scaling / jittering (multiplicative noise)
    if scaling_apply_prob > 0:
      condition = keras.ops.cast(
        keras.ops.less(keras.random.uniform(()), scaling_apply_prob),
        dtype=keras.ops.dtype(data)
      )

      scale_factor = keras.random.uniform(
        (batch_size, num_features, 1),
        scaling_min,
        scaling_max
      )
      data = data * (condition * scale_factor + (1 - condition))

    return data

  @classmethod
  def _augment_categorical_features(
    cls,
    data: keras.KerasTensor,
    cutmix_apply_prob: float = 1.0,
    cutmix_alpha: float = 1.0,
    cutmix_beta: Optional[float] = None,
    feature_mask_apply_prob: float = 1.0,
    feature_mask_prob: float = 0.15,
  ) -> keras.KerasTensor:
    batch_size, num_features, feature_dim = keras.ops.shape(data)

    # 1. CutMix for categorical features
    if cutmix_apply_prob > 0:
      condition = keras.ops.less(keras.random.uniform(()), cutmix_apply_prob)
      data = keras.ops.cond(condition, lambda: cls._tabular_cutmix(data, cutmix_alpha, cutmix_beta), lambda: data)

    # 2. Feature-level dropout
    if feature_mask_apply_prob > 0 and feature_mask_prob > 0:
      condition = keras.ops.cast(
        keras.ops.less(keras.random.uniform(()), feature_mask_apply_prob),
        dtype=keras.ops.dtype(data)
      )

      # Only generate mask if we might use it
      random_mask = keras.random.uniform((batch_size, 1, feature_dim))
      feature_mask = keras.ops.cast(
        keras.ops.greater(random_mask, feature_mask_prob),
        dtype=keras.ops.dtype(data)
      )

      # Blend: data * [condition * mask + (1 - condition)]
      # This equals: data * mask when condition=1, data when condition=0
      data = data * (condition * feature_mask + (1 - condition))

    return data

  @classmethod
  def _tabular_cutmix(
    cls,
    data: keras.KerasTensor,
    alpha: float = 1.0,
    beta: Optional[float] = None,
  ) -> keras.KerasTensor:
    """
    CutMix adaptation for tabular data.

    Instead of shuffling columns (which breaks order), we mix entire rows
    or contiguous segments of features from different samples.
    """
    batch_size, num_columns, column_dim = keras.ops.shape(data)

    # Sample mixing ratio
    beta = beta or alpha
    lam = keras.random.beta((), alpha, beta)

    # Determine how many columns to replace (from another sample)
    cut_size = keras.ops.cast(
        keras.ops.cast(num_columns, 'float32') * (1.0 - lam),
        'int32'
    )
    cut_size = keras.ops.clip(cut_size, 1, num_columns - 1)

    # Random contiguous segment
    max_start_pos = num_columns - cut_size
    cut_start = keras.random.randint((), 0, max_start_pos + 1, dtype='int32')
    cut_end = cut_start + cut_size

    # Create position indices [0, 1, 2, ..., num_columns-1]
    # Shape: [num_columns]
    positions = keras.ops.arange(num_columns, dtype='int32')

    # Create boolean mask for cut region
    # Shape: [num_columns] -> broadcasts to [batch_size, num_columns, column_dim]
    mask = keras.ops.logical_and(
      keras.ops.greater_equal(positions, cut_start),
      keras.ops.less(positions, cut_end)
    )
    mask = keras.ops.cast(mask, data.dtype)
    mask = keras.ops.reshape(mask, (1, num_columns, 1))

    # Shuffle batch indices to get mixing pairs
    shuffled_indices = keras.random.shuffle(keras.ops.arange(batch_size))
    shuffled_data = keras.ops.take(data, shuffled_indices, axis=0)

    # Vectorized blend: use shuffled where mask=1, original where mask=0
    mixed = data * (1.0 - mask) + shuffled_data * mask

    return mixed

  @classmethod
  def _numerical_mixup(
    cls,
    data: keras.KerasTensor,
    alpha: float = 0.2,
    beta: Optional[float] = None,
  ) -> keras.KerasTensor:
    """MixUp for tabular data - safe as it preserves column order."""
    batch_size, num_columns, column_dim = keras.ops.shape(data)

    # Sample mixing coefficient
    beta = beta or alpha
    lam = keras.random.beta((batch_size, 1, 1), alpha, beta)

    # Shuffle batch and mix
    indices = keras.random.shuffle(keras.ops.arange(batch_size))
    shuffled = keras.ops.take(data, indices, axis=0)
    mixed = lam * data + (1 - lam) * shuffled

    return mixed

  def build(self, cat_inputs_shape: tuple[Optional[int], int, int], num_inputs_shape: tuple[Optional[int], int, int]) -> None:
    self.built = True

  def compute_output_shape(self, cat_inputs_shape: tuple[Optional[int], int, int], num_inputs_shape: tuple[Optional[int], int, int]) -> tuple[tuple[Optional[int], int, int], tuple[Optional[int], int, int]]:
    if len(cat_inputs_shape) != 3 or len(cat_inputs_shape) != 3:
      raise ValueError(f"Input tensors must be 3-dimensional, but are {len(cat_inputs_shape)} and {len(num_inputs_shape)}")
    elif cat_inputs_shape[0] != num_inputs_shape[0]:
      raise ValueError(f"Input tensors must have the same batch size, but have {cat_inputs_shape[0]} and {num_inputs_shape[0]}")

    return cat_inputs_shape, num_inputs_shape

  @tf.custom_gradient
  def call(self, cat_inputs: keras.KerasTensor, num_inputs: keras.KerasTensor) -> keras.KerasTensor:
    x_num = self.augment_numerical_features(num_inputs)
    x_cat = self.augment_categorical_features(cat_inputs)

    def grad(upstream_cat: keras.KerasTensor, upstream_num: keras.KerasTensor) -> tuple[keras.KerasTensor, keras.KerasTensor]:
      return upstream_cat, upstream_num

    return (x_cat, x_num), grad

In [ ]:
WEAK_TAB_AUGMENTATION = {
    'num_feature_mask_apply_prob': 0.3,
    'num_feature_mask_prob': 0.05,
    'cat_feature_mask_apply_prob': 0.3,
    'cat_feature_mask_prob': 0.05,

    'num_cutmix_apply_prob': 0.2,
    'num_cutmix_alpha': 0.8,
    'num_mixup_apply_prob': 0.2,
    'num_mixup_alpha': 0.8,

    'num_noise_apply_prob': 0.3,
    'num_noise_max_std': 0.02,
    'num_scaling_apply_prob': 0.3,
    'num_scaling_min': 0.95,
    'num_scaling_max': 1.05,

    'cat_cutmix_apply_prob': 0.2,
    'cat_cutmix_alpha': 0.8,
}

WEAK_TAB_AUGMENTATION_NOMIX = {
    'num_feature_mask_apply_prob': 0.3,
    'num_feature_mask_prob': 0.05,
    'cat_feature_mask_apply_prob': 0.3,
    'cat_feature_mask_prob': 0.05,

    'num_cutmix_apply_prob': 0.0,
    'num_mixup_apply_prob': 0.0,

    'num_noise_apply_prob': 0.3,
    'num_noise_max_std': 0.02,
    'num_scaling_apply_prob': 0.3,
    'num_scaling_min': 0.95,
    'num_scaling_max': 1.05,

    'cat_cutmix_apply_prob': 0.0,
}

MEDIUM_TAB_AUGMENTATION = {
    'num_feature_mask_apply_prob': 0.6,
    'num_feature_mask_prob': 0.1,
    'cat_feature_mask_apply_prob': 0.6,
    'cat_feature_mask_prob': 0.1,

    'num_cutmix_apply_prob': 0.5,
    'num_cutmix_alpha': 0.4,
    'num_mixup_apply_prob': 0.5,
    'num_mixup_alpha': 0.4,

    'num_noise_apply_prob': 0.6,
    'num_noise_max_std': 0.05,
    'num_scaling_apply_prob': 0.6,
    'num_scaling_min': 0.9,
    'num_scaling_max': 1.1,

    'cat_cutmix_apply_prob': 0.5,
    'cat_cutmix_alpha': 0.4,
}

MEDIUM_TAB_AUGMENTATION_NOMIX = {
    'num_feature_mask_apply_prob': 0.6,
    'num_feature_mask_prob': 0.1,
    'cat_feature_mask_apply_prob': 0.6,
    'cat_feature_mask_prob': 0.1,

    'num_cutmix_apply_prob': 0.0,
    'num_mixup_apply_prob': 0.0,

    'num_noise_apply_prob': 0.6,
    'num_noise_max_std': 0.05,
    'num_scaling_apply_prob': 0.6,
    'num_scaling_min': 0.9,
    'num_scaling_max': 1.1,

    'cat_cutmix_apply_prob': 0.0,
}

STRONG_TAB_AUGMENTATION = {
    'num_feature_mask_apply_prob': 0.9,
    'num_feature_mask_prob': 0.2,
    'cat_feature_mask_apply_prob': 0.9,
    'cat_feature_mask_prob': 0.2,

    'num_cutmix_apply_prob': 0.8,
    'num_cutmix_alpha': 0.2,
    'num_mixup_apply_prob': 0.8,
    'num_mixup_alpha': 0.2,

    'num_noise_apply_prob': 0.8,
    'num_noise_max_std': 0.1,
    'num_scaling_apply_prob': 0.8,
    'num_scaling_min': 0.8,
    'num_scaling_max': 1.2,

    'cat_cutmix_apply_prob': 0.8,
    'cat_cutmix_alpha': 0.2,
}

STRONG_TAB_AUGMENTATION_NOMIX = {
    'num_feature_mask_apply_prob': 0.9,
    'num_feature_mask_prob': 0.2,
    'cat_feature_mask_apply_prob': 0.9,
    'cat_feature_mask_prob': 0.2,

    'num_cutmix_apply_prob': 0.0,
    'num_mixup_apply_prob': 0.0,

    'num_noise_apply_prob': 0.8,
    'num_noise_max_std': 0.1,
    'num_scaling_apply_prob': 0.8,
    'num_scaling_min': 0.8,
    'num_scaling_max': 1.2,

    'cat_cutmix_apply_prob': 0.0,
}

#### Base Tab Transformer

In [ ]:
@keras.saving.register_keras_serializable()
class TabTransformer(keras.Model):
    def __init__(
        self,
        num_encoder_blocks: int = 1,
        encoder_intermediate_dim: int = 128,
        encoder_num_heads: int = 4,
        encoder_dropout: float = 0.0,
        encoder_activation: str = "relu",
        encoder_normalize_first: bool = True,
        ffn_dims: Union[int, List[int]] = 128,
        ffn_dropout: float = 0.0,
        ffn_activation: str = "leaky_relu",
        ffn_normalization_type: Optional[Literal['layer', 'batch']] = None,
        **kwargs
    ):
        super().__init__(**kwargs)

        if isinstance(ffn_dims, int):
          ffn_dims = [ffn_dims]

        self.num_encoder_blocks = num_encoder_blocks
        self.encoder_intermediate_dim = encoder_intermediate_dim
        self.encoder_num_heads = encoder_num_heads
        self.encoder_dropout = encoder_dropout
        self.encoder_activation = encoder_activation
        self.encoder_normalize_first = encoder_normalize_first

        self.ffn_dims = ffn_dims
        self.ffn_dropout = ffn_dropout
        self.ffn_activation = ffn_activation
        self.ffn_normalization_type = ffn_normalization_type

        self.encoders = self._build_encoders()
        self.ffn = self._build_ffn()

        self.flatten_cat = keras.layers.Flatten(name="flatten_cat")
        self.flatten_num = keras.layers.Flatten(name="flatten_num")
        self.norm_num = keras.layers.LayerNormalization(name="layernorm_num")

        self.concat = keras.layers.Concatenate()

    def _build_encoders(self, **kwargs) -> List[keras.Layer]:
      encoders = []

      for i in range(self.num_encoder_blocks):
        layer = keras_nlp.layers.TransformerEncoder(
          intermediate_dim=self.encoder_intermediate_dim,
          num_heads=self.encoder_num_heads,
          dropout=self.encoder_dropout,
          activation=self.encoder_activation,
          normalize_first=self.encoder_normalize_first,
          name=f"transformer_encoder_{i}",
          **kwargs
        )

        encoders.append(layer)

      return encoders

    def _build_ffn(self, **kwargs) -> keras.Layer:
      layer = keras.Sequential(name="ffn", **kwargs)

      for dim in self.ffn_dims:
        layer.add(self._build_norm_layer(self.ffn_normalization_type))
        layer.add(keras.layers.Dense(dim, activation=self.ffn_activation))
        layer.add(keras.layers.Dropout(self.ffn_dropout))

      return layer

    def _build_norm_layer(self, norm_type: Optional[Literal['layer', 'batch']]) -> keras.Layer:
      if norm_type is None:
        return keras.layers.Identity()
      elif 'layer' == norm_type:
        return keras.layers.LayerNormalization()
      elif 'batch' == norm_type:
        return keras.layers.BatchNormalization()
      else:
        raise NotImplementedError(f"Unknown normalization type {self.norm_type}")

    def build(self, cat_inputs_shape: Tuple[Optional[int], int, int], num_inputs_shape: Tuple[Optional[int], int, int]) -> None:
      batch_size, num_cat_columns, cat_feature_dim = cat_inputs_shape
      batch_size, num_num_columns, num_feature_dim = num_inputs_shape

      self.column_embedding = keras_nlp.layers.PositionEmbedding(sequence_length=num_cat_columns)
      self.column_embedding.build(cat_inputs_shape)

      for encoder in self.encoders:
        encoder.build(cat_inputs_shape)

      self.flatten_cat.build(cat_inputs_shape)
      cat_out_shape = self.flatten_cat.compute_output_shape(cat_inputs_shape)

      self.norm_num.build(num_inputs_shape)
      self.flatten_num.build(num_inputs_shape)
      num_out_shape = self.flatten_num.compute_output_shape(num_inputs_shape)

      self.concat.build([cat_out_shape, num_out_shape])
      out_shape = self.concat.compute_output_shape([cat_out_shape, num_out_shape])

      self.ffn.build(out_shape)

    def build_graph(self, batch_size: int, cat_features: int, cat_feature_dim: int, num_features: int, num_feature_dim: int) -> keras.Model:
      cat_input_shape = (batch_size, cat_features, cat_feature_dim)
      num_input_shape = (batch_size, num_features, num_feature_dim)

      cat_inputs = keras.layers.Input(shape=cat_input_shape[1:], batch_size=cat_input_shape[0])
      num_inputs = keras.layers.Input(shape=num_input_shape[1:], batch_size=num_input_shape[0])
      self.build(cat_inputs.shape, num_inputs.shape)

      return keras.Model(inputs=[cat_inputs, num_inputs], outputs=self.call(cat_inputs, num_inputs), name=self.name)

    def compute_output_shape(self, cat_inputs_shape: Tuple[Optional[int], int, int], num_inputs_shape: Tuple[Optional[int], int, int]) -> Tuple[Optional[int], int, int]:
      cat_out_shape = self.flatten_cat.compute_output_shape(cat_inputs_shape)
      num_out_shape = self.flatten_num.compute_output_shape(num_inputs_shape)

      out_shape = self.concat.compute_output_shape([cat_out_shape, num_out_shape])
      out_shape = self.ffn.compute_output_shape(out_shape)

      return out_shape

    def get_config(self) -> dict:
      config = super().get_config()
      config.update({
        "num_encoder_blocks": self.num_encoder_blocks,
        "encoder_intermediate_dim": self.encoder_intermediate_dim,
        "encoder_num_heads": self.encoder_num_heads,
        "encoder_dropout": self.encoder_dropout,
        "encoder_activation": self.encoder_activation,
        "encoder_normalize_first": self.encoder_normalize_first,
        "ffn_dims": self.ffn_dims,
        "ffn_dropout": self.ffn_dropout,
        "ffn_activation": self.ffn_activation,
        "ffn_normalization_type": self.ffn_normalization_type,
      })

      return config

    @classmethod
    def from_config(cls, config: dict):
      return cls(**config)

    def call(self, cat_inputs: keras.KerasTensor, num_inputs: keras.KerasTensor, mask: Optional[keras.KerasTensor] = None, training: bool = False) -> keras.KerasTensor:
      x_cat = cat_inputs + self.column_embedding(cat_inputs, training=training)
      for encoder in self.encoders:
        x_cat = encoder(x_cat, attention_mask=mask, training=training)
      x_cat = self.flatten_cat(x_cat)

      x_num = self.norm_num(num_inputs, training=training)
      x_num = self.flatten_num(x_num)

      x = self.concat([x_cat, x_num])
      x = self.ffn(x, training=training)
      return x


In [ ]:
tt = TabTransformer()

model = tt.build_graph(32, 10, 64, 10, 1)
model.summary()

In [ ]:
keras.utils.plot_model(
  model,
  show_shapes=True,
  show_layer_names=True,
  expand_nested=True,
)

#### Contrastive TabTransformer

In [ ]:
@keras.saving.register_keras_serializable(package="custom")
class ContrastiveProcessTabTransformer(keras.Model, LayerNameMixin):
  def __init__(
    self,
    input_encoder: InputEncoder,
    embedding_model: TabTransformer,
    augmentation_1: TabularAugmentationLayer,
    augmentation_2: Optional[SetAugmentationLayer] = None,
    loss: keras.Loss = BarlowTwinsLoss(lambda_param=1e-4),

    temperature: float = 0.1,
    projection_dim: int = 128,
    projection_hidden_dims: Union[int, List[int]] = [512, 256],
    projection_hidden_activation: str = "leaky_relu",
    projection_dropout: float = 0.0,
    debug: bool = False,
    **kwargs
  ):
      super().__init__(**kwargs)

      self.input_encoder = input_encoder
      self.tab_transformer = embedding_model

      self.augmentation_1 = augmentation_1
      self.augmentation_2 = augmentation_2 or augmentation_1

      if isinstance(projection_hidden_dims, int):
        projection_hidden_dims = [projection_hidden_dims]
      self.projection_hidden_dims = projection_hidden_dims
      self.projection_hidden_activation = projection_hidden_activation
      self.projection_dim = projection_dim
      self.projection_dropout = projection_dropout

      self.projection_head = self._build_projection_head()
      self.debug = debug

      lr_scheduler = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.00001,
        decay_steps=2000,
        alpha=0.0001,
        warmup_target=0.0001,
        warmup_steps=200,
      )

      #self.optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.004, global_clipnorm=3.0) #, clipvalue=2.0)
      self.optimizer = keras.optimizers.AdamW(learning_rate=lr_scheduler, weight_decay=0.004)#, clipnorm=3.0) #global_clipnorm=2.0)

      self.default_callbacks = [
        keras.callbacks.TerminateOnNaN(),
      #  keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=5, mode='min', factor=0.5, min_lr=DEFAULT_MIN_LEARNING_RATE, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, mode='min', restore_best_weights=True, verbose=1),
        keras.callbacks.BackupAndRestore(os.path.join(MODEL_BACKUP_DIR, self.name), save_freq='epoch', double_checkpoint=True, delete_checkpoint=False),
        keras.callbacks.ModelCheckpoint(filepath=os.path.join(MODEL_CHECKPOINT_DIR, f"{self.name}.keras"), save_best_only=True, save_weights_only=False, monitor='val_loss', mode='auto', save_freq='epoch', verbose=1),
        keras.callbacks.CSVLogger(filename=os.path.join(OUTPUT_DATA_DIR, f"{self.name}_history.csv"), separator=";", append=True),
        keras.callbacks.TensorBoard(log_dir=os.path.join(OUTPUT_DATA_DIR, f"{self.name}_tensorboard"), update_freq='epoch', histogram_freq=5, embeddings_freq=5, write_graph=True, write_images=True, profile_batch=(1,5))
      ]

      self.default_loss = loss

      self.loss_tracker = keras.metrics.Mean(name="loss")
      self.alignment_tracker = Alignment(l2_normalize=True)
      self.uniformity_tracker = Uniformity()
      self.effective_rank_tracker = EffectiveRank()

  def build(self, input_shape_cat: tuple[int, int, int], input_shape_num: tuple[int, int, int]) -> None:
    batch_size = input_shape_cat[0]

    self.input_encoder.build(input_shape_cat)
    output_shape_cat, output_shape_num = self.input_encoder.compute_output_shape(input_shape_cat)

    self.augmentation_1.build(output_shape_cat, output_shape_num)
    self.augmentation_2.build(output_shape_cat, output_shape_num)
    output_shape_cat, output_shape_num = self.augmentation_1.compute_output_shape(output_shape_cat, output_shape_num)

    self.tab_transformer.build(output_shape_cat, output_shape_num)
    output_shape = self.tab_transformer.compute_output_shape(output_shape_cat, output_shape_num)

    self.projection_head.build(output_shape)
    output_shape = self.projection_head.compute_output_shape(output_shape)

    self.built = True

  def compute_output_shape(self, input_shape: tuple[int, int, int]) -> tuple[int, int, int]:
    batch_size = input_shape_cat[0]

    output_shape_cat, output_shape_num = self.input_encoder.compute_output_shape(input_shape)
    output_shape = self.tab_transformer.compute_output_shape(output_shape_cat, output_shape_num)
    output_shape = self.flattener.compute_output_shape(output_shape)
    output_shape = self.projection_head.compute_output_shape(output_shape)
    return output_shape

  @property
  def callbacks(self) -> List[keras.callbacks.Callback]:
    return self.default_callbacks

  @property
  def metrics(self) -> List[keras.Metric]:
    return [self.loss_tracker, self.alignment_tracker, self.uniformity_tracker, self.effective_rank_tracker]

  def train_step(self, data: keras.KerasTensor) -> dict:
    # Unpack the data
    tabs = data[0] if isinstance(data, tuple) else data

    # Create two augmented views of each set
    with tf.GradientTape() as tape:
      cat_attrs, num_attrs = self.input_encoder(tabs, training=True)

      #with tape.stop_recording():
      augmented_tabs_cat_1, augmented_tabs_num_1 = self.augmentation_1(cat_attrs, num_attrs)
      augmented_tabs_cat_2, augmented_tabs_num_2 = self.augmentation_2(cat_attrs, num_attrs)

      # Get representations for both views
      projections_1 = self(augmented_tabs_cat_1, augmented_tabs_num_1, training=True)
      projections_2 = self(augmented_tabs_cat_2, augmented_tabs_num_2, training=True)

      # Compute contrastive loss
      loss = keras.ops.mean(self.default_loss(projections_1, projections_2))

    # Compute gradients and update weights
    trainable_vars = self.trainable_variables
    gradients = tape.gradient(loss, trainable_vars)
    self.optimizer.apply_gradients(zip(gradients, trainable_vars))

    # Update metrics
    self.loss_tracker.update_state(loss)
    #self.effective_rank_tracker.update_state(keras.ops.concatenate([projections_1, projections_2], axis=0))
    #self.alignment_tracker.update_state(projections_1, projections_2)
    #self.uniformity_tracker.update_state(keras.ops.concatenate([projections_1, projections_2], axis=0))
    grad_norm = keras.ops.sqrt(sum(keras.ops.sum(keras.ops.square(g)) for g in gradients))

    debug_statistics = self._debug_statistics(projections_1, projections_2) if self.debug else {}

    return {
      "loss": self.loss_tracker.result(),
      #"alignment": self.alignment_tracker.result(),
      #"uniformity": self.uniformity_tracker.result(),
      #"effective_rank": self.effective_rank_tracker.result(),
      "gradient_norm": grad_norm,
      **debug_statistics
    }

  def test_step(self, data: keras.KerasTensor) -> dict:
    # Unpack the data
    sets = data[0] if isinstance(data, tuple) else data

    # Create two augmented views of each set
    cat_attrs, num_attrs = self.input_encoder(sets, training=False)

    augmented_tabs_cat_1, augmented_tabs_num_1 = self.augmentation_1(cat_attrs, num_attrs, training=False)
    augmented_tabs_cat_2, augmented_tabs_num_2 = self.augmentation_2(cat_attrs, num_attrs, training=False)

    projections_1 = self(augmented_tabs_cat_1, augmented_tabs_num_1, training=False)
    projections_2 = self(augmented_tabs_cat_2, augmented_tabs_num_2, training=False)

    loss = self.default_loss(projections_1, projections_2)

    # Update metrics
    self.loss_tracker.update_state(loss)
    self.effective_rank_tracker.update_state(keras.ops.concatenate([projections_1, projections_2], axis=0))
    self.alignment_tracker.update_state(projections_1, projections_2)
    self.uniformity_tracker.update_state(keras.ops.concatenate([projections_1, projections_2], axis=0))

    return {
      "loss": self.loss_tracker.result(),
      "alignment": self.alignment_tracker.result(),
      "uniformity": self.uniformity_tracker.result(),
      "effective_rank": self.effective_rank_tracker.result(),
    }

  def _debug_statistics(self, projections_1: keras.KerasTensor, projections_2: keras.KerasTensor) -> dict:
    projections_1 = keras.utils.normalize(projections_1, axis=1, order=2)
    projections_2 = keras.utils.normalize(projections_2, axis=1, order=2)

    similarities = keras.ops.matmul(projections_1, keras.ops.transpose(projections_2))
    sim_pos = keras.ops.sum(projections_1 * projections_2, axis=1)  # Positive pair similarities
    sim_matrix = keras.ops.matmul(projections_1, keras.ops.transpose(projections_2))

    stats = {
      # 1. Check embeddings are normalized
      'Z1_norm_min': float(keras.ops.min(keras.ops.norm(projections_1, axis=1))),
      'Z1_norm_max': float(keras.ops.max(keras.ops.norm(projections_1, axis=1))),
      'Z1_norm_mean': float(keras.ops.mean(keras.ops.norm(projections_1, axis=1))),

      'Z2_norm_min': float(keras.ops.min(keras.ops.norm(projections_2, axis=1))),
      'Z2_norm_max': float(keras.ops.max(keras.ops.norm(projections_2, axis=1))),
      'Z2_norm_mean': float(keras.ops.mean(keras.ops.norm(projections_2, axis=1))),

      # 2. Check embedding statistics
      'Z1_mean': float(keras.ops.mean(projections_1)),
      'Z1_std': float(keras.ops.std(projections_1)),

      'Z2_mean': float(keras.ops.mean(projections_2)),
      'Z2_std': float(keras.ops.std(projections_2)),

      # 3. Check similarity distribution
      'similarity_mean': float(keras.ops.mean(similarities)),
      'similarity_std': float(keras.ops.std(similarities)),

      # 4. Check specific similarities
      'positive_sim_min': float(keras.ops.min(sim_pos)),
      'positive_sim_max': float(keras.ops.max(sim_pos)),
      'positive_sim_mean': float(keras.ops.mean(sim_pos)),

      'all_sim_min': float(keras.ops.min(sim_matrix)),
      'all_sim_max': float(keras.ops.max(sim_matrix)),
      'all_sim_mean': float(keras.ops.mean(sim_matrix))
    }

    return stats

  def _build_projection_head(self, name: str = "projection") -> keras.Layer:
    projection_head = keras.Sequential([], name=name)
    for dim in self.projection_hidden_dims:
      projection_head.add(keras.layers.Dense(dim, activation=self.projection_hidden_activation))
      projection_head.add(keras.layers.Dropout(self.projection_dropout))

    projection_head.add(keras.layers.Dense(self.projection_dim, use_bias=False))
    return projection_head

  def build_embedding_model(self, trainable: bool = False) -> keras.Model:
    ie = self.input_encoder.build_graph()
    embed_model = keras.Model(inputs=ie.inputs, outputs=self.tab_transformer(ie.outputs[0], ie.outputs[1]))

    embed_model.trainable = trainable
    return embed_model

  @classmethod
  def from_config(cls, config: dict) -> "ContrastiveProcessSetTransformer":
    return cls(
      input_encoder=InputEncoder.from_config(config.pop("input_encoder")),
      embedding_model=TabTransformer.from_config(config.pop("embedding_model")),
      augmentation_1=TabularAugmentationLayer.from_config(config.pop("augmentation_1")),
      augmentation_2=TabularAugmentationLayer.from_config(config.pop("augmentation_2")) if config.get("augmentation_2") else config.pop("augmentation_2"),
      loss=keras.saving.deserialize_keras_object(config.pop("loss")),
      **config
    )

  def get_config(self) -> dict:
    config = super().get_config()
    config.update({
      "input_encoder": self.input_encoder.get_config(),
      "embedding_model": self.tab_transformer.get_config(),
      "augmentation_1": self.augmentation_1.get_config(),
      "augmentation_2": self.augmentation_2.get_config(),
      "loss": keras.saving.serialize_keras_object(self.default_loss),
      "projection_dim": self.projection_dim,
      "projection_hidden_dims": self.projection_hidden_dims,
      "projection_hidden_activation": self.projection_hidden_activation,
      "projection_dropout": self.projection_dropout,
      "debug": self.debug,
    })
    return config

  def call(self, inputs_cat: keras.KerasTensor, inputs_num: keras.KerasTensor, training: bool = False) -> keras.KerasTensor:
    x = self.tab_transformer(inputs_cat, inputs_num, training=training)
    x = self.projection_head(x, training=training)
    return x

# Dataset: Incident Management Process Enriched Event Log

## Input Preparation

In [ ]:
df_servicenow = pd.read_feather(os.path.join(INPUT_DATA_DIR, "incident_event_log_labeled.feather"))
df_servicenow

In [ ]:
df_servicenow_train = pd.read_feather(os.path.join(INPUT_DATA_DIR, "incident_event_log_train.feather"))
df_servicenow_train

In [ ]:
df_servicenow_test = pd.read_feather(os.path.join(INPUT_DATA_DIR, "incident_event_log_test.feather"))
df_servicenow_test

In [ ]:
PROJECT_NAME = "servicenow"
MAX_SEQ_LEN = 13
TRAIN_STEPS_EPOCH = len(df_servicenow_train) // DEFAULT_BATCH_SIZE

TIME_ATTRS = {
    'month': "time_timestamp_month",
    'hour': "time_timestamp_hour",
    'day': "time_timestamp_day",
    'weekday': "time_timestamp_weekday",
    'yearday': "time_timestamp_dayofyear",
}
ACTIVITY_VOCAB = np.array(df_servicenow[EVENTLOG_ACTIVITY].to_list() + [TOKEN_NA, TOKEN_EOC])
NUM_ACTIVITIES = np.unique(ACTIVITY_VOCAB).shape[0]

INTERVAL_SINCE_LAST_EVENT_VOCAB = df_servicenow_train["time:timestamp:elapsedprev:seconds"].to_numpy()
INTERVAL_SINCE_START_VOCAB = df_servicenow_train["time:timestamp:elapsedcycle:seconds"].to_numpy()
DAY_OF_WEEK_VOCAB = df_servicenow_train["time:timestamp:weekday:raw"].to_numpy()
HOUR_OF_DAY_VOCAB = df_servicenow_train["time:timestamp:hour:raw"].to_numpy()

TIME_ATTRS_VOCAB = {
  # Already scaled
  "time_timestamp_month": None,
  "time_timestamp_hour": None,
  "time_timestamp_day": None,
  "time_timestamp_weekday": None,
  "time_timestamp_dayofyear": None,
  # Need Scaling
  "time_timestamp_elapsedprev": INTERVAL_SINCE_LAST_EVENT_VOCAB,
  "time_timestamp_elapsedcycle": INTERVAL_SINCE_START_VOCAB,
}

STATIC_CATEGORICAL_ATTRS = {
    "case_opened_by": df_servicenow["case:opened_by"].unique().to_numpy(dtype=str),
    "case_sys_created_by": df_servicenow["case:sys_created_by"].unique().to_numpy(dtype=str),
}
STATIC_NUMERICAL_ATTRS = {
    "case_notify": None,
}
DYNAMIC_CATEGORICAL_ATTRS = {
    "category": df_servicenow["category"].unique().to_numpy(dtype=str),
    "subcategory": df_servicenow["subcategory"].unique().to_numpy(dtype=str),
    "location": df_servicenow["location"].unique().to_numpy(dtype=str),
    "u_symptom": df_servicenow["u_symptom"].unique().to_numpy(dtype=str),
    "caller_id": df_servicenow["caller_id"].unique().to_numpy(dtype=str),
    "sys_updated_by": df_servicenow["sys_updated_by"].unique().to_numpy(dtype=str),
    "org_group": df_servicenow["org:group"].unique().to_numpy(dtype=str),
    "org_resource": df_servicenow["org:resource"].unique().to_numpy(dtype=str),
    "contact_type": df_servicenow["contact_type"].unique().to_numpy(dtype=str),
}

DYNAMIC_NUMERICAL_ATTRS = {
    "sys_mod_count": None,
    "u_priority_confirmation": None,
    "knowledge": None,
    "priority": None,
    "reopen_count": None,
    "reassignment_count": None,
}

DYNAMIC_RAW_ATTRS = {
    "org_resource_graph": -99,
    "org_group_graph": -99
}

In [ ]:
ds_servicenow_train = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "incident_event_log_train_dataset"),
    compression='GZIP'
)

ds_servicenow_train = ds_window_dynamic_attrs(ds_servicenow_train, MAX_SEQ_LEN)

print(len(ds_servicenow_train))

ds_servicenow_train = ds_cache_and_batch(ds_servicenow_train, shuffle=False)

ds_servicenow_train

In [ ]:
!unzip "/content/Data/Input/incident_event_log_tabular_train_dataset.zip" -d "/content/Data/Input/incident_event_log_tabular_train_dataset"
!unzip "/content/Data/Input/incident_event_log_tabular_test_dataset.zip" -d "/content/Data/Input/incident_event_log_tabular_test_dataset"

In [ ]:
ds_servicenow_tab_train = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "incident_event_log_tabular_train_dataset"),
    compression='GZIP'
)

print(len(ds_servicenow_tab_train))
ds_servicenow_tab_train = ds_remove_outputs(ds_servicenow_tab_train)
ds_servicenow_tab_train = ds_cache_and_batch(ds_servicenow_tab_train, batch_size=256, shuffle=False)
ds_servicenow_tab_train

In [ ]:
ds_servicenow_test = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "incident_event_log_test_dataset"),
    compression='GZIP'
)
ds_servicenow_test = ds_window_dynamic_attrs(ds_servicenow_test, MAX_SEQ_LEN)

print(len(ds_servicenow_test))

ds_servicenow_test = ds_cache_and_batch(ds_servicenow_test, shuffle=False)
ds_servicenow_test

In [ ]:
ds_servicenow_tab_test = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "incident_event_log_tabular_test_dataset"),
    compression='GZIP'
)

print(len(ds_servicenow_tab_test))
ds_servicenow_tab_test = ds_remove_outputs(ds_servicenow_tab_test)
ds_servicenow_tab_test = ds_cache_and_batch(ds_servicenow_tab_test, batch_size=256, shuffle=False)
ds_servicenow_tab_test

## LSTM

### Multitask ProcessLSTM

In [ ]:
model_name = f"lstm_{PROJECT_NAME}_multi"

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseProcessRNN(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    interval_since_last_event_vocab=INTERVAL_SINCE_LAST_EVENT_VOCAB,
    hour_of_day_vocab=HOUR_OF_DAY_VOCAB,
    day_of_week_vocab=DAY_OF_WEEK_VOCAB,
    output_dict={"next_activity": {'units': NUM_ACTIVITIES, 'activation': 'softmax'}, "next_time": {'units': 1, 'activation': 'linear'}, "remaining_time": {'units': 1, 'activation': 'linear'}},
    name=model_name,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_servicenow_train,
  validation_data=ds_servicenow_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_servicenow_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_servicenow_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)
model_regression_report(model, ds_servicenow_test, DEFAULT_NEXT_TIME_OUTPUT, archive=True)
model_classification_report(model, ds_servicenow_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

In [ ]:
logs = os.path.join(OUTPUT_LOG_DATA_DIR, model_name)
%tensorboard --logdir "$logs"

## DA-LSTM

### Remaining Time DA-LSTM

In [ ]:
model_name = f"dalstm_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}"

ds_servicenow_dalstm_train = ds_to_single_target(ds_sequentialize_static_attrs(ds_servicenow_train, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_REMAINING_TIME_OUTPUT)
ds_servicenow_dalstm_train = ds_cache_and_batch(ds_servicenow_dalstm_train, shuffle=False)

ds_servicenow_dalstm_test = ds_to_single_target(ds_sequentialize_static_attrs(ds_servicenow_test, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_REMAINING_TIME_OUTPUT)
ds_servicenow_dalstm_test = ds_cache_and_batch(ds_servicenow_dalstm_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseDARNN(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    categorical_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS,
    numerical_attrs=DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS | {"time_timestamp_elapsedcycle": None, "time_timestamp_elapsedprev": None, "time_timestamp_hour": None, "time_timestamp_weekday": None},
    output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}},
    name=model_name,
    rnn_dropout=0.1,
    ff_dropout=0.15,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_servicenow_dalstm_train,
  validation_data=ds_servicenow_dalstm_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_servicenow_dalstm_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_servicenow_dalstm_test, DEFAULT_REMAINING_TIME_OUTPUT)

### Next Activity DA-LSTM

In [ ]:
model_name = f"dalstm_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}"

ds_servicenow_dalstm_train = ds_to_single_target(ds_sequentialize_static_attrs(ds_servicenow_train, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_servicenow_dalstm_train = ds_cache_and_batch(ds_servicenow_dalstm_train, shuffle=False)

ds_servicenow_dalstm_test = ds_to_single_target(ds_sequentialize_static_attrs(ds_servicenow_test, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_servicenow_dalstm_test = ds_cache_and_batch(ds_servicenow_dalstm_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseDARNN(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    categorical_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS,
    numerical_attrs=DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS | {"time_timestamp_elapsedcycle": None, "time_timestamp_elapsedprev": None, "time_timestamp_hour": None, "time_timestamp_weekday": None},
    output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'units': NUM_ACTIVITIES, 'activation': 'softmax'}},
    name=model_name,
    rnn_dropout=0.1,
    ff_dropout=0.15,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_servicenow_dalstm_train,
  validation_data=ds_servicenow_dalstm_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_servicenow_dalstm_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_servicenow_dalstm_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## ProcessTransformer

### Remaining Time ProcessTransformer

In [ ]:
model_name = f"processtransformer_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}"

ds_servicenow_transformer_train = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_servicenow_train, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_REMAINING_TIME_OUTPUT)
ds_servicenow_transformer_train = ds_cache_and_batch(ds_servicenow_transformer_train, shuffle=False)

ds_servicenow_transformer_test = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_servicenow_test, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_REMAINING_TIME_OUTPUT)
ds_servicenow_transformer_test = ds_cache_and_batch(ds_servicenow_transformer_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseProcessTransformer(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    interval_since_last_event_vocab=INTERVAL_SINCE_LAST_EVENT_VOCAB,
    hour_of_day_vocab=HOUR_OF_DAY_VOCAB,
    day_of_week_vocab=DAY_OF_WEEK_VOCAB,
    output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}},
    name=model_name,
    activity_encoding_type='embedding'
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_servicenow_transformer_train,
  validation_data=ds_servicenow_transformer_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_servicenow_transformer_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_servicenow_transformer_test, DEFAULT_REMAINING_TIME_OUTPUT)

In [ ]:
logs = os.path.join(OUTPUT_LOG_DATA_DIR, model_name)
%tensorboard --logdir "$logs"

### Next Activity ProcessTransformer

In [ ]:
model_name = f"processtransformer_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}"

ds_servicenow_transformer_train = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_servicenow_train, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_servicenow_transformer_train = ds_cache_and_batch(ds_servicenow_transformer_train, shuffle=False)

ds_servicenow_transformer_test = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_servicenow_test, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_servicenow_transformer_test = ds_cache_and_batch(ds_servicenow_transformer_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseProcessTransformer(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    interval_since_last_event_vocab=INTERVAL_SINCE_LAST_EVENT_VOCAB,
    hour_of_day_vocab=HOUR_OF_DAY_VOCAB,
    day_of_week_vocab=DAY_OF_WEEK_VOCAB,
    output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'units': NUM_ACTIVITIES, 'activation': 'softmax'}},
    name=model_name,
    activity_encoding_type='embedding'
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_servicenow_transformer_train,
  validation_data=ds_servicenow_transformer_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_servicenow_transformer_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_servicenow_transformer_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

In [ ]:
logs = os.path.join(OUTPUT_LOG_DATA_DIR, model_name)
%tensorboard --logdir "$logs"

## ProcessSetTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cpst = ContrastiveProcessSetTransformer(
    name="servicenow_set_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      num_encoding_type='log_normal',
      embed_dim=64,
    ),
    embedding_model=SetTransformer(
        num_inductions=8,
        num_seeds=2,
        encoder_num_heads=4,
        encoder_dropout=0.1,
        decoder_num_heads=4,
        decoder_dropout=0.1,
    ),
    augmentation_1=SetAugmentationLayer(**MEDIUM_SET_AUGMENTATION),
    augmentation_2=SetAugmentationLayer(**STRONG_SET_AUGMENTATION),

)

cpst.build((512,2,1))

loss = cpst.default_loss
metrics = cpst.metrics

optimizer = cpst.optimizer
callbacks = cpst.callbacks

cpst.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cpst.summary()

In [ ]:
history = cpst.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_servicenow_tab_train,
  validation_data=ds_servicenow_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cpst, history)

In [ ]:
cpst.evaluate(ds_servicenow_tab_train)

In [ ]:
cpst.evaluate(ds_servicenow_tab_test)

In [ ]:
model = cpst.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cpst.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_servicenow_tab_train))
test_embeddings = np.squeeze(model.predict(ds_servicenow_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_servicenow_train,
  df_servicenow_test,
  cat_attrs=[
      EVENTLOG_ACTIVITY,
      EVENTLOG_RESOURCE,
      EVENTLOG_GROUP,
      'caller_id',
      'sys_updated_by',
      'contact_type',
      'location',
      'category',
      'subcategory',
      'u_symptom',
      'case:opened_by',
      'case:sys_created_by'
    ],
  num_attrs=[
      'priority',
      'knowledge',
      'reassignment_count',
      'sys_mod_count',
      'reopen_count',
      'u_priority_confirmation',
      'case:notify',
      "time:timestamp:elapsedcycle:seconds",
      "time:timestamp:elapsedprev:seconds",
      "time:timestamp:month",
      "time:timestamp:dayofyear",
      "time:timestamp:day",
      "time:timestamp:weekday",
      "time:timestamp:hour"
    ],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=10000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, num_samples=10000, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_servicenow_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_servicenow_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/servicenow_set_embedding_test_dataset.zip -d /content/Data/Input/servicenow_set_embedding_test_dataset
!unzip /content/Data/Input/servicenow_set_embedding_train_dataset.zip -d /content/Data/Input/servicenow_set_embedding_train_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)

ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)

ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: ProcessSetTransformer without Mix

In [ ]:
keras.utils.clear_session(free_memory=True)

cpst = ContrastiveProcessSetTransformer(
    name="servicenow_nomix_set_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      num_encoding_type='log_normal',
      embed_dim=64,
    ),
    embedding_model=SetTransformer(
        num_inductions=8,
        num_seeds=2,
        encoder_num_heads=4,
        encoder_dropout=0.1,
        decoder_num_heads=4,
        decoder_dropout=0.1,
    ),
    augmentation_1=SetAugmentationLayer(**MEDIUM_SET_AUGMENTATION_NOMIX),
    augmentation_2=SetAugmentationLayer(**STRONG_SET_AUGMENTATION_NOMIX),

)

cpst.build((512,2,1))

loss = cpst.default_loss
metrics = cpst.metrics

optimizer = cpst.optimizer
callbacks = cpst.callbacks

cpst.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cpst.summary()

In [ ]:
history = cpst.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_servicenow_tab_train,
  validation_data=ds_servicenow_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cpst, history)

In [ ]:
cpst.evaluate(ds_servicenow_tab_train)

In [ ]:
cpst.evaluate(ds_servicenow_tab_test)

In [ ]:
model = cpst.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cpst.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_servicenow_tab_train))
test_embeddings = np.squeeze(model.predict(ds_servicenow_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_servicenow_train,
  df_servicenow_test,
  cat_attrs=[
      EVENTLOG_ACTIVITY,
      EVENTLOG_RESOURCE,
      EVENTLOG_GROUP,
      'caller_id',
      'sys_updated_by',
      'contact_type',
      'location',
      'category',
      'subcategory',
      'u_symptom',
      'case:opened_by',
      'case:sys_created_by'
    ],
  num_attrs=[
      'priority',
      'knowledge',
      'reassignment_count',
      'sys_mod_count',
      'reopen_count',
      'u_priority_confirmation',
      'case:notify',
      "time:timestamp:elapsedcycle:seconds",
      "time:timestamp:elapsedprev:seconds",
      "time:timestamp:month",
      "time:timestamp:dayofyear",
      "time:timestamp:day",
      "time:timestamp:weekday",
      "time:timestamp:hour"
    ],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=10000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, num_samples=10000, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_servicenow_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_servicenow_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/servicenow_nomix_set_embedding_train_dataset.zip -d /content/Data/Input/servicenow_nomix_set_embedding_train_dataset
!unzip /content/Data/Input/servicenow_nomix_set_embedding_test_dataset.zip -d /content/Data/Input/servicenow_nomix_set_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)

ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)

ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: Randomly Initialized SetTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cpst = ContrastiveProcessSetTransformer(
    name="servicenow_random_set_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      num_encoding_type='log_normal',
      embed_dim=64,
    ),
    embedding_model=SetTransformer(
        num_inductions=8,
        num_seeds=2,
        encoder_num_heads=4,
        encoder_dropout=0.1,
        decoder_num_heads=4,
        decoder_dropout=0.1,
    ),
    augmentation_1=SetAugmentationLayer(**MEDIUM_SET_AUGMENTATION),
    augmentation_2=SetAugmentationLayer(**STRONG_SET_AUGMENTATION),
)

cpst.build((256,2,1))

loss = cpst.default_loss
metrics = cpst.metrics

optimizer = cpst.optimizer
callbacks = cpst.callbacks

cpst.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)
cpst.trainable = False
cpst.summary()

In [ ]:
cpst.evaluate(ds_servicenow_tab_train)

In [ ]:
cpst.evaluate(ds_servicenow_tab_test)

In [ ]:
model = cpst.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cpst.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_servicenow_tab_train))
test_embeddings = np.squeeze(model.predict(ds_servicenow_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_servicenow_train,
  df_servicenow_test,
  cat_attrs=[
      EVENTLOG_ACTIVITY,
      EVENTLOG_RESOURCE,
      EVENTLOG_GROUP,
      'caller_id',
      'sys_updated_by',
      'contact_type',
      'location',
      'category',
      'subcategory',
      'u_symptom',
      'case:opened_by',
      'case:sys_created_by'
  ],
  num_attrs=[
      'priority',
      'knowledge',
      'reassignment_count',
      'sys_mod_count',
      'reopen_count',
      'u_priority_confirmation',
      'case:notify',
      "time:timestamp:elapsedcycle:seconds",
      "time:timestamp:elapsedprev:seconds",
      "time:timestamp:month",
      "time:timestamp:dayofyear",
      "time:timestamp:day",
      "time:timestamp:weekday",
      "time:timestamp:hour"
    ],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_servicenow_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_servicenow_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/servicenow_random_set_embedding_train_dataset.zip -d /content/Data/Input/servicenow_random_set_embedding_train_dataset
!unzip /content/Data/Input/servicenow_random_set_embedding_test_dataset.zip -d /content/Data/Input/servicenow_random_set_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, 'servicenow_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, 'servicenow_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, 'servicenow_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)

ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, 'servicenow_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)

ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, 'servicenow_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, 'servicenow_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, 'servicenow_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, 'servicenow_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## ProcessTabTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cptt = ContrastiveProcessTabTransformer(
    name="servicenow_tab_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      embed_dim=64,
      num_encoding_type='log_normal'
    ),
    embedding_model=TabTransformer(
        encoder_num_heads=4,
        encoder_dropout=0.1,
        ffn_dims=[256, 128],
    ),
    augmentation_1=TabularAugmentationLayer(**MEDIUM_TAB_AUGMENTATION),
    augmentation_2=TabularAugmentationLayer(**STRONG_TAB_AUGMENTATION),

)

cptt.build((512, len(cptt.input_encoder.cat_attrs), 32), (512, len(cptt.input_encoder.num_attrs), 32))

loss = cptt.default_loss
metrics = cptt.metrics

optimizer = cptt.optimizer
callbacks = cptt.callbacks

cptt.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cptt.summary()

In [ ]:
history = cptt.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_servicenow_tab_train,
  validation_data=ds_servicenow_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cptt, history)

In [ ]:
cptt.evaluate(ds_servicenow_tab_train)

In [ ]:
cptt.evaluate(ds_servicenow_tab_test)

In [ ]:
model = cptt.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cptt.name}.keras"))

#model = keras.models.load_model("/content/Models/bpic13_tab_embedding.keras", custom_objects={"log1p": keras.ops.log1p})
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_servicenow_tab_train))
test_embeddings = np.squeeze(model.predict(ds_servicenow_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_servicenow_train,
  df_servicenow_test,
  cat_attrs=[
      EVENTLOG_ACTIVITY,
      EVENTLOG_RESOURCE,
      EVENTLOG_GROUP,
      'caller_id',
      'sys_updated_by',
      'contact_type',
      'location',
      'category',
      'subcategory',
      'u_symptom',
      'case:opened_by',
      'case:sys_created_by'
    ],
  num_attrs=[
      'priority',
      'knowledge',
      'reassignment_count',
      'sys_mod_count',
      'reopen_count',
      'u_priority_confirmation',
      'case:notify',
      "time:timestamp:elapsedcycle:seconds",
      "time:timestamp:elapsedprev:seconds",
      "time:timestamp:month",
      "time:timestamp:dayofyear",
      "time:timestamp:day",
      "time:timestamp:weekday",
      "time:timestamp:hour"
    ],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=10000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, num_samples=10000, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_servicenow_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_servicenow_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/servicenow_tab_embedding_train_dataset.zip -d /content/Data/Input/servicenow_tab_embedding_train_dataset
!unzip /content/Data/Input/servicenow_tab_embedding_test_dataset.zip -d /content/Data/Input/servicenow_tab_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: ProcessTabTransformer without Mix

In [ ]:
keras.utils.clear_session(free_memory=True)

cptt = ContrastiveProcessTabTransformer(
    name="servicenow_nomix_tab_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      embed_dim=64,
      num_encoding_type='log_normal'
    ),
    embedding_model=TabTransformer(
        encoder_num_heads=4,
        encoder_dropout=0.1,
        ffn_dims=[256, 128],
    ),
    augmentation_1=TabularAugmentationLayer(**MEDIUM_TAB_AUGMENTATION_NOMIX),
    augmentation_2=TabularAugmentationLayer(**STRONG_TAB_AUGMENTATION_NOMIX),
)

cptt.build((512, len(cptt.input_encoder.cat_attrs), 32), (512, len(cptt.input_encoder.num_attrs), 32))

loss = cptt.default_loss
metrics = cptt.metrics

optimizer = cptt.optimizer
callbacks = cptt.callbacks

cptt.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cptt.summary()

In [ ]:
history = cptt.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_servicenow_tab_train,
  validation_data=ds_servicenow_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cptt, history)

In [ ]:
cptt.evaluate(ds_servicenow_tab_train)

In [ ]:
cptt.evaluate(ds_servicenow_tab_test)

In [ ]:
model = cptt.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cptt.name}.keras"))

#model = keras.models.load_model("/content/Models/bpic13_tab_embedding.keras", custom_objects={"log1p": keras.ops.log1p})
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_servicenow_tab_train))
test_embeddings = np.squeeze(model.predict(ds_servicenow_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_servicenow_train,
  df_servicenow_test,
  cat_attrs=[
      EVENTLOG_ACTIVITY,
      EVENTLOG_RESOURCE,
      EVENTLOG_GROUP,
      'caller_id',
      'sys_updated_by',
      'contact_type',
      'location',
      'category',
      'subcategory',
      'u_symptom',
      'case:opened_by',
      'case:sys_created_by'
    ],
  num_attrs=[
      'priority',
      'knowledge',
      'reassignment_count',
      'sys_mod_count',
      'reopen_count',
      'u_priority_confirmation',
      'case:notify',
      "time:timestamp:elapsedcycle:seconds",
      "time:timestamp:elapsedprev:seconds",
      "time:timestamp:month",
      "time:timestamp:dayofyear",
      "time:timestamp:day",
      "time:timestamp:weekday",
      "time:timestamp:hour"
    ],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=10000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, num_samples=10000, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_servicenow_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_servicenow_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/servicenow_nomix_tab_embedding_test_dataset.zip -d /content/Data/Input/servicenow_nomix_tab_embedding_test_dataset
!unzip /content/Data/Input/servicenow_nomix_tab_embedding_train_dataset.zip -d /content/Data/Input/servicenow_nomix_tab_embedding_train_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: Randomly Initialized TabTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cptt = ContrastiveProcessTabTransformer(
    name="servicenow_random_tab_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      embed_dim=64,
      num_encoding_type='log_normal'
    ),
    embedding_model=TabTransformer(
        encoder_num_heads=4,
        encoder_dropout=0.1,
        ffn_dims=[256, 128],
    ),
    augmentation_1=TabularAugmentationLayer(**MEDIUM_TAB_AUGMENTATION),
    augmentation_2=TabularAugmentationLayer(**STRONG_TAB_AUGMENTATION),
)

cptt.build((256, len(cptt.input_encoder.cat_attrs), 32), (256, len(cptt.input_encoder.num_attrs), 32))

loss = cptt.default_loss
metrics = cptt.metrics

optimizer = cptt.optimizer
callbacks = cptt.callbacks

cptt.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)
cptt.trainable = False
cptt.summary()

In [ ]:
cptt.evaluate(ds_servicenow_tab_train)

In [ ]:
cptt.evaluate(ds_servicenow_tab_test)

In [ ]:
model = cptt.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cptt.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_servicenow_tab_train))
test_embeddings = np.squeeze(model.predict(ds_servicenow_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_servicenow_train,
  df_servicenow_test,
  cat_attrs=[
      EVENTLOG_ACTIVITY,
      EVENTLOG_RESOURCE,
      EVENTLOG_GROUP,
      'caller_id',
      'sys_updated_by',
      'contact_type',
      'location',
      'category',
      'subcategory',
      'u_symptom',
      'case:opened_by',
      'case:sys_created_by'
  ],
  num_attrs=[
      'priority',
      'knowledge',
      'reassignment_count',
      'sys_mod_count',
      'reopen_count',
      'u_priority_confirmation',
      'case:notify',
      "time:timestamp:elapsedcycle:seconds",
      "time:timestamp:elapsedprev:seconds",
      "time:timestamp:month",
      "time:timestamp:dayofyear",
      "time:timestamp:day",
      "time:timestamp:weekday",
      "time:timestamp:hour"
    ],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy,  num_samples=20000, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_servicenow_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_servicenow_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/servicenow_random_tab_embedding_test_dataset.zip -d /content/Data/Input/servicenow_random_tab_embedding_test_dataset
!unzip /content/Data/Input/servicenow_random_tab_embedding_train_dataset.zip -d /content/Data/Input/servicenow_random_tab_embedding_train_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

# Dataset: Dataset belonging to the help desk log of an Italian Company

## Input Preparation

In [ ]:
df_italy = pd.read_feather(os.path.join(INPUT_DATA_DIR, "finale_labeled.feather"))
df_italy

In [ ]:
df_italy_train = pd.read_feather(os.path.join(INPUT_DATA_DIR, "finale_train.feather"))
df_italy_train

In [ ]:
df_italy_test = pd.read_feather(os.path.join(INPUT_DATA_DIR, "finale_test.feather"))
df_italy_test

In [ ]:
PROJECT_NAME = "italy"
MAX_SEQ_LEN = 7
TRAIN_STEPS_EPOCH = len(df_italy_train) // DEFAULT_BATCH_SIZE

TIME_ATTRS = {
    'month': "time_timestamp_month",
    'hour': "time_timestamp_hour",
    'day': "time_timestamp_day",
    'weekday': "time_timestamp_weekday",
    'yearday': "time_timestamp_dayofyear",
}

ACTIVITY_VOCAB = np.array(df_italy[EVENTLOG_ACTIVITY].tolist() + [TOKEN_NA, TOKEN_EOC])
NUM_ACTIVITIES = np.unique(ACTIVITY_VOCAB).shape[0]

INTERVAL_SINCE_LAST_EVENT_VOCAB = df_italy_train["time:timestamp:elapsedprev:seconds"].to_numpy()
INTERVAL_SINCE_START_VOCAB = df_italy_train["time:timestamp:elapsedcycle:seconds"].to_numpy()
DAY_OF_WEEK_VOCAB = df_italy_train["time:timestamp:weekday:raw"].to_numpy()
HOUR_OF_DAY_VOCAB = df_italy_train["time:timestamp:hour:raw"].to_numpy()

TIME_ATTRS_VOCAB = {
  # Already scaled
  "time_timestamp_month": None,
  "time_timestamp_hour": None,
  "time_timestamp_day": None,
  "time_timestamp_weekday": None,
  "time_timestamp_dayofyear": None,
  # Need Scaling
  "time_timestamp_elapsedprev": INTERVAL_SINCE_LAST_EVENT_VOCAB,
  "time_timestamp_elapsedcycle": INTERVAL_SINCE_START_VOCAB,
}

STATIC_CATEGORICAL_ATTRS = {
    "case_responsible_section": df_italy["case:responsible_section"].unique().to_numpy(dtype=str),
    "case_support_section": df_italy["case:support_section"].unique().to_numpy(dtype=str),
}
STATIC_NUMERICAL_ATTRS = {}

DYNAMIC_CATEGORICAL_ATTRS = {
    "org_resource": df_italy["org:resource"].unique().to_numpy(dtype=str),
    "org_group": df_italy["org:group"].unique().to_numpy(dtype=str),
    "customer": df_italy["customer"].unique().to_numpy(dtype=str),
    "product": df_italy["product"].unique().to_numpy(dtype=str),
    "service_type": df_italy["service_type"].unique().to_numpy(dtype=str),
}
DYNAMIC_NUMERICAL_ATTRS = {
    "seriousness_2": None,
    "service_level": None,
}

DYNAMIC_RAW_ATTRS = {
    "org_resource_graph": -99,
    "org_group_graph": -99
}

In [ ]:
ds_italy_train = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "finale_train_dataset"),
    compression='GZIP'
)
ds_italy_train = ds_window_dynamic_attrs(ds_italy_train, MAX_SEQ_LEN)

print(len(ds_italy_train))

ds_italy_train = ds_cache_and_batch(ds_italy_train, shuffle=False)
ds_italy_train

In [ ]:
!unzip "/content/Data/Input/finale_tabular_test_dataset.zip" -d "/content/Data/Input/finale_tabular_test_dataset"
!unzip "/content/Data/Input/finale_tabular_train_dataset.zip" -d "/content/Data/Input/finale_tabular_train_dataset"

In [ ]:
ds_italy_tab_train = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "finale_tabular_train_dataset"),
    compression='GZIP'
)

print(len(ds_italy_tab_train))
ds_italy_tab_train = ds_remove_outputs(ds_italy_tab_train)
ds_italy_tab_train = ds_cache_and_batch(ds_italy_tab_train, batch_size=256, shuffle=False)
ds_italy_tab_train

In [ ]:
ds_italy_test = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "finale_test_dataset"),
    compression='GZIP'
)
ds_italy_test = ds_window_dynamic_attrs(ds_italy_test, MAX_SEQ_LEN)
print(len(ds_italy_test))

ds_italy_test = ds_cache_and_batch(ds_italy_test, shuffle=False)
ds_italy_test

In [ ]:
ds_italy_tab_test = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "finale_tabular_test_dataset"),
    compression='GZIP'
)

print(len(ds_italy_tab_test))
ds_italy_tab_test = ds_remove_outputs(ds_italy_tab_test)
ds_italy_tab_test = ds_cache_and_batch(ds_italy_tab_test, batch_size=256, shuffle=False)
ds_italy_tab_test

## LSTM

### Multitask ProcessLSTM

In [ ]:
model_name = f"lstm_{PROJECT_NAME}_multi"

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseProcessRNN(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    interval_since_last_event_vocab=INTERVAL_SINCE_LAST_EVENT_VOCAB,
    hour_of_day_vocab=HOUR_OF_DAY_VOCAB,
    day_of_week_vocab=DAY_OF_WEEK_VOCAB,
    output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'units': NUM_ACTIVITIES, 'activation': 'softmax'}, DEFAULT_NEXT_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}, DEFAULT_REMAINING_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}},
    name=model_name,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_italy_train,
  validation_data=ds_italy_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_italy_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_italy_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)
model_regression_report(model, ds_italy_test, DEFAULT_NEXT_TIME_OUTPUT, archive=True)
model_classification_report(model, ds_italy_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

In [ ]:
logs = os.path.join(OUTPUT_LOG_DATA_DIR, model_name)
%tensorboard --logdir "$logs"

## DA-LSTM

### Remaining Time DA-LSTM

In [ ]:
model_name = f"dalstm_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}"

ds_italy_dalstm_train = ds_to_single_target(ds_sequentialize_static_attrs(ds_italy_train, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_REMAINING_TIME_OUTPUT)
ds_italy_dalstm_train = ds_cache_and_batch(ds_italy_dalstm_train, shuffle=False)

ds_italy_dalstm_test = ds_to_single_target(ds_sequentialize_static_attrs(ds_italy_test, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_REMAINING_TIME_OUTPUT)
ds_italy_dalstm_test = ds_cache_and_batch(ds_italy_dalstm_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseDARNN(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    categorical_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS,
    numerical_attrs=DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS | {"time_timestamp_elapsedcycle": None, "time_timestamp_elapsedprev": None, "time_timestamp_hour": None, "time_timestamp_weekday": None},
    output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}},
    name=model_name,
    rnn_dropout=0.1,
    ff_dropout=0.15,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_italy_dalstm_train,
  validation_data=ds_italy_dalstm_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_italy_dalstm_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_italy_dalstm_test, DEFAULT_REMAINING_TIME_OUTPUT)

### Next Activity DA-LSTM

In [ ]:
model_name = f"dalstm_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}"

ds_italy_dalstm_train = ds_to_single_target(ds_sequentialize_static_attrs(ds_italy_train, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_italy_dalstm_train = ds_cache_and_batch(ds_italy_dalstm_train, shuffle=False)

ds_italy_dalstm_test = ds_to_single_target(ds_sequentialize_static_attrs(ds_italy_test, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_italy_dalstm_test = ds_cache_and_batch(ds_italy_dalstm_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseDARNN(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    categorical_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS,
    numerical_attrs=DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS | {"time_timestamp_elapsedcycle": None, "time_timestamp_elapsedprev": None, "time_timestamp_hour": None, "time_timestamp_weekday": None},
    output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'units': NUM_ACTIVITIES, 'activation': 'softmax'}},
    name=model_name,
    rnn_dropout=0.1,
    ff_dropout=0.15,
    rnn_units=[400, 400],
    ff_dims=[800, 400, 200, 100],
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_italy_dalstm_train,
  validation_data=ds_italy_dalstm_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_italy_dalstm_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_italy_dalstm_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Transformer

### Remaining Time Process Transformer

In [ ]:
model_name = f"processtransformer_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}"

ds_italy_transformer_train = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_italy_train, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_REMAINING_TIME_OUTPUT)
ds_italy_transformer_train = ds_cache_and_batch(ds_italy_transformer_train, shuffle=False)

ds_italy_transformer_test = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_italy_test, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_REMAINING_TIME_OUTPUT)
ds_italy_transformer_test = ds_cache_and_batch(ds_italy_transformer_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseProcessTransformer(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    interval_since_last_event_vocab=INTERVAL_SINCE_LAST_EVENT_VOCAB,
    hour_of_day_vocab=HOUR_OF_DAY_VOCAB,
    day_of_week_vocab=DAY_OF_WEEK_VOCAB,
    output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}},
    name=model_name,
    activity_encoding_type='embedding'
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_italy_train,
  validation_data=ds_italy_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_italy_transformer_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_italy_transformer_test, DEFAULT_REMAINING_TIME_OUTPUT)

In [ ]:
logs = os.path.join(OUTPUT_LOG_DATA_DIR, model_name)
%tensorboard --logdir "$logs"

### Next Activity Process Transformer

In [ ]:
model_name = f"processtransformer_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}"

ds_italy_transformer_train = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_italy_train, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_italy_transformer_train = ds_cache_and_batch(ds_italy_transformer_train, shuffle=False)

ds_italy_transformer_test = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_italy_test, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_italy_transformer_test = ds_cache_and_batch(ds_italy_transformer_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseProcessTransformer(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    interval_since_last_event_vocab=INTERVAL_SINCE_LAST_EVENT_VOCAB,
    hour_of_day_vocab=HOUR_OF_DAY_VOCAB,
    day_of_week_vocab=DAY_OF_WEEK_VOCAB,
    output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'units': NUM_ACTIVITIES, 'activation': 'softmax'}},
    name=model_name,
    activity_encoding_type='embedding'
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_italy_train,
  validation_data=ds_italy_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_italy_transformer_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_italy_transformer_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

In [ ]:
logs = os.path.join(OUTPUT_LOG_DATA_DIR, model_name)
%tensorboard --logdir "$logs"

## ProcessSetTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cpst = ContrastiveProcessSetTransformer(
    name="italy_set_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      num_encoding_type='log_normal',
      embed_dim=64,
    ),
    embedding_model=SetTransformer(
        num_inductions=4,
        num_seeds=2,
        encoder_num_heads=4,
        encoder_dropout=0.1,
        decoder_num_heads=4,
        decoder_dropout=0.1,
    ),
    augmentation_1=SetAugmentationLayer(**WEAK_SET_AUGMENTATION),
    augmentation_2=SetAugmentationLayer(**MEDIUM_SET_AUGMENTATION),

)

cpst.build((512,2,1))

loss = cpst.default_loss
metrics = cpst.metrics

#lr_schedule = keras.optimizers.schedules.CosineDecay(
#  initial_learning_rate=0.001,
#  decay_steps=10000
#)
optimizer = cpst.optimizer
callbacks = cpst.callbacks

cpst.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

cpst.summary()

In [ ]:
history = cpst.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_italy_tab_train,
  validation_data=ds_italy_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cpst, history)

In [ ]:
cpst.evaluate(ds_italy_tab_train)

In [ ]:
cpst.evaluate(ds_italy_tab_test)

In [ ]:
model = cpst.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cpst.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_italy_tab_train))
test_embeddings = np.squeeze(model.predict(ds_italy_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_italy_train,
  df_italy_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_RESOURCE, EVENTLOG_GROUP, 'customer', 'product', 'case:responsible_section', 'case:support_section'],
  num_attrs=['seriousness_2', 'service_level', "time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_italy_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_italy_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/italy_set_embedding_train_dataset.zip -d /content/Data/Input/italy_set_embedding_train_dataset
!unzip /content/Data/Input/italy_set_embedding_test_dataset.zip -d /content/Data/Input/italy_set_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: ProcessSetTransformer without Mix

In [ ]:
keras.utils.clear_session(free_memory=True)

cpst = ContrastiveProcessSetTransformer(
    name="italy_nomix_set_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      num_encoding_type='log_normal',
      embed_dim=64,
    ),
    embedding_model=SetTransformer(
        num_inductions=4,
        num_seeds=2,
        encoder_num_heads=4,
        encoder_dropout=0.1,
        decoder_num_heads=4,
        decoder_dropout=0.1,
    ),
    augmentation_1=SetAugmentationLayer(**WEAK_SET_AUGMENTATION_NOMIX),
    augmentation_2=SetAugmentationLayer(**MEDIUM_SET_AUGMENTATION_NOMIX),
)

cpst.build((512,2,1))

loss = cpst.default_loss
metrics = cpst.metrics

optimizer = cpst.optimizer
callbacks = cpst.callbacks

cpst.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cpst.summary()

In [ ]:
history = cpst.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_italy_tab_train,
  validation_data=ds_italy_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cpst, history)

In [ ]:
cpst.evaluate(ds_italy_tab_train)

In [ ]:
cpst.evaluate(ds_italy_tab_test)

In [ ]:
model = cpst.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cpst.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_italy_tab_train))
test_embeddings = np.squeeze(model.predict(ds_italy_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_italy_train,
  df_italy_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_RESOURCE, EVENTLOG_GROUP, 'customer', 'product', 'case:responsible_section', 'case:support_section'],
  num_attrs=['seriousness_2', 'service_level', "time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_italy_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_italy_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/italy_nomix_set_embedding_test_dataset.zip -d /content/Data/Input/italy_nomix_set_embedding_test_dataset
!unzip /content/Data/Input/italy_nomix_set_embedding_train_dataset.zip -d /content/Data/Input/italy_nomix_set_embedding_train_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: Randomly Initialized SetTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cpst = ContrastiveProcessSetTransformer(
    name="italy_random_set_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      num_encoding_type='log_normal',
      embed_dim=64,
    ),
    embedding_model=SetTransformer(
        num_inductions=8,
        num_seeds=2,
        encoder_num_heads=4,
        encoder_dropout=0.1,
        decoder_num_heads=4,
        decoder_dropout=0.1,
    ),
    augmentation_1=SetAugmentationLayer(**MEDIUM_SET_AUGMENTATION),
    augmentation_2=SetAugmentationLayer(**STRONG_SET_AUGMENTATION),
)

cpst.build((256,2,1))

loss = cpst.default_loss
metrics = cpst.metrics

optimizer = cpst.optimizer
callbacks = cpst.callbacks

cpst.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)
cpst.trainable = False
cpst.summary()

In [ ]:
cpst.evaluate(ds_italy_tab_train)

In [ ]:
cpst.evaluate(ds_italy_tab_test)

In [ ]:
model = cpst.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cpst.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_italy_tab_train))
test_embeddings = np.squeeze(model.predict(ds_italy_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_italy_train,
  df_italy_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_RESOURCE, EVENTLOG_GROUP, 'customer', 'product', 'case:responsible_section', 'case:support_section'],
  num_attrs=['seriousness_2', 'service_level', "time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_italy_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_italy_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/italy_random_set_embedding_train_dataset.zip -d /content/Data/Input/italy_random_set_embedding_train_dataset
!unzip /content/Data/Input/italy_random_set_embedding_test_dataset.zip -d /content/Data/Input/italy_random_set_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## ProcessTabTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cptt = ContrastiveProcessTabTransformer(
    name="italy_tab_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      embed_dim=64,
      num_encoding_type='log_normal'
    ),
    embedding_model=TabTransformer(
        encoder_num_heads=4,
        encoder_dropout=0.1,
        ffn_dims=[256, 128],
    ),
    augmentation_1=TabularAugmentationLayer(**WEAK_TAB_AUGMENTATION),
    augmentation_2=TabularAugmentationLayer(**MEDIUM_TAB_AUGMENTATION),

)

cptt.build((512, len(cptt.input_encoder.cat_attrs), 32), (512, len(cptt.input_encoder.num_attrs), 32))

loss = cptt.default_loss
metrics = cptt.metrics

optimizer = cptt.optimizer
callbacks = cptt.callbacks

cptt.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cptt.summary()

In [ ]:
history = cptt.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_italy_tab_train,
  validation_data=ds_italy_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cptt, history)

In [ ]:
cptt.evaluate(ds_italy_tab_train)

In [ ]:
cptt.evaluate(ds_italy_tab_test)

In [ ]:
model = cptt.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cptt.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_italy_tab_train))
test_embeddings = np.squeeze(model.predict(ds_italy_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_italy_train,
  df_italy_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_RESOURCE, EVENTLOG_GROUP, 'customer', 'product', 'case:responsible_section', 'case:support_section'],
  num_attrs=['seriousness_2', 'service_level', "time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_italy_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_italy_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_test_dataset"))

In [ ]:
logs = os.path.join(OUTPUT_DATA_DIR, "italy_tab_embedding_tensorboard")
%tensorboard --logdir "$logs"

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/italy_tab_embedding_train_dataset.zip -d /content/Data/Input/italy_tab_embedding_train_dataset
!unzip /content/Data/Input/italy_tab_embedding_test_dataset.zip -d /content/Data/Input/italy_tab_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: ProcessTabTransformer without Mix

In [ ]:
keras.utils.clear_session(free_memory=True)

cptt = ContrastiveProcessTabTransformer(
    name="italy_nomix_tab_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      embed_dim=64,
      num_encoding_type='log_normal'
    ),
    embedding_model=TabTransformer(
        encoder_num_heads=4,
        encoder_dropout=0.1,
        ffn_dims=[256, 128],
    ),
    augmentation_1=TabularAugmentationLayer(**WEAK_TAB_AUGMENTATION_NOMIX),
    augmentation_2=TabularAugmentationLayer(**MEDIUM_TAB_AUGMENTATION_NOMIX),

)

cptt.build((512, len(cptt.input_encoder.cat_attrs), 32), (512, len(cptt.input_encoder.num_attrs), 32))

loss = cptt.default_loss
metrics = cptt.metrics

optimizer = cptt.optimizer
callbacks = cptt.callbacks

cptt.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cptt.summary()

In [ ]:
history = cptt.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_italy_tab_train,
  validation_data=ds_italy_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cptt, history)

In [ ]:
cptt.evaluate(ds_italy_tab_train)

In [ ]:
cptt.evaluate(ds_italy_tab_test)

In [ ]:
model = cptt.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cptt.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_italy_tab_train))
test_embeddings = np.squeeze(model.predict(ds_italy_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_italy_train,
  df_italy_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_RESOURCE, EVENTLOG_GROUP, 'customer', 'product', 'case:responsible_section', 'case:support_section'],
  num_attrs=['seriousness_2', 'service_level', "time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_italy_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_italy_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_test_dataset"))

In [ ]:
logs = os.path.join(OUTPUT_DATA_DIR, "italy_tab_embedding_tensorboard")
%tensorboard --logdir "$logs"

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/italy_nomix_tab_embedding_test_dataset.zip -d /content/Data/Input/italy_nomix_tab_embedding_test_dataset
!unzip /content/Data/Input/italy_nomix_tab_embedding_train_dataset.zip -d /content/Data/Input/italy_nomix_tab_embedding_train_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: Randomly Initialized TabTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cptt = ContrastiveProcessTabTransformer(
    name="italy_random_tab_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      embed_dim=64,
      num_encoding_type='log_normal'
    ),
    embedding_model=TabTransformer(
        encoder_num_heads=4,
        encoder_dropout=0.1,
        ffn_dims=[256, 128],
    ),
    augmentation_1=TabularAugmentationLayer(**MEDIUM_TAB_AUGMENTATION),
    augmentation_2=TabularAugmentationLayer(**STRONG_TAB_AUGMENTATION),
)

cptt.build((256, len(cptt.input_encoder.cat_attrs), 32), (256, len(cptt.input_encoder.num_attrs), 32))

loss = cptt.default_loss
metrics = cptt.metrics

optimizer = cptt.optimizer
callbacks = cptt.callbacks

cptt.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)
cptt.trainable = False
cptt.summary()

In [ ]:
cptt.evaluate(ds_italy_tab_train)

In [ ]:
cptt.evaluate(ds_italy_tab_test)

In [ ]:
model = cptt.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cptt.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_italy_tab_train))
test_embeddings = np.squeeze(model.predict(ds_italy_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_italy_train,
  df_italy_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_RESOURCE, EVENTLOG_GROUP, 'customer', 'product', 'case:responsible_section', 'case:support_section'],
  num_attrs=['seriousness_2', 'service_level', "time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy,  num_samples=20000, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_italy_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_italy_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/italy_random_tab_embedding_train_dataset.zip -d /content/Data/Input/italy_random_tab_embedding_train_dataset
!unzip /content/Data/Input/italy_random_tab_embedding_test_dataset.zip -d /content/Data/Input/italy_random_tab_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

# Dataset: BPIC 2014

## Input Preparation

In [ ]:
df_bpic14 = pd.read_feather(os.path.join(INPUT_DATA_DIR, "Detail_Incident_Activity_labeled.feather"))
df_bpic14

In [ ]:
df_bpic14_train = pd.read_feather(os.path.join(INPUT_DATA_DIR, "Detail_Incident_Activity_train.feather"))
df_bpic14_train

In [ ]:
df_bpic14_test = pd.read_feather(os.path.join(INPUT_DATA_DIR, "Detail_Incident_Activity_test.feather"))
df_bpic14_test

In [ ]:
MAX_SEQ_LEN = 27
PROJECT_NAME = "bpic14"
TRAIN_STEPS_EPOCH = len(df_bpic14_train) // DEFAULT_BATCH_SIZE


TIME_ATTRS = {
    'month': "time_timestamp_month",
    'hour': "time_timestamp_hour",
    'day': "time_timestamp_day",
    'weekday': "time_timestamp_weekday",
    'yearday': "time_timestamp_dayofyear",
}
ACTIVITY_VOCAB = np.array(df_bpic14[EVENTLOG_ACTIVITY].tolist() + [TOKEN_NA, TOKEN_EOC])
NUM_ACTIVITIES = np.unique(ACTIVITY_VOCAB).shape[0]

INTERVAL_SINCE_START_VOCAB = df_bpic14_train["time:timestamp:elapsedcycle:seconds"].to_numpy()
INTERVAL_SINCE_LAST_EVENT_VOCAB = df_bpic14_train["time:timestamp:elapsedprev:seconds"].to_numpy()
DAY_OF_WEEK_VOCAB = df_bpic14_train["time:timestamp:weekday:raw"].to_numpy()
HOUR_OF_DAY_VOCAB = df_bpic14_train["time:timestamp:hour:raw"].to_numpy()

TIME_ATTRS_VOCAB = {
  # Already scaled
  "time_timestamp_month": None,
  "time_timestamp_hour": None,
  "time_timestamp_day": None,
  "time_timestamp_weekday": None,
  "time_timestamp_dayofyear": None,
  # Need Scaling
  "time_timestamp_elapsedprev": INTERVAL_SINCE_LAST_EVENT_VOCAB,
  "time_timestamp_elapsedcycle": INTERVAL_SINCE_START_VOCAB,
}

STATIC_CATEGORICAL_ATTRS = {
    "case_km_number": df_bpic14["case:KM number"].unique().to_numpy(dtype=str),
    "case_incident_category": df_bpic14["case:incident_Category"].unique().to_numpy(dtype=str),
    "case_incident_ci_type_aff": df_bpic14["case:incident_CI Type (aff)"].unique().to_numpy(dtype=str),
    "case_incident_ci_subtype_aff": df_bpic14["case:incident_CI Subtype (aff)"].unique().to_numpy(dtype=str),
    "case_incident_service_component_aff": df_bpic14["case:incident_Service Component WBS (aff)"].unique().to_numpy(dtype=str),
    "case_incident_ci_name_cby": df_bpic14["case:incident_CI Name (CBy)"].unique().to_numpy(dtype=str),
    "case_incident_ci_type_cby": df_bpic14["case:incident_CI Type (CBy)"].unique().to_numpy(dtype=str),
}
STATIC_NUMERICAL_ATTRS = {
    "case_interaction_priority": None,
    "case_incident_priority": None,
}

DYNAMIC_CATEGORICAL_ATTRS = {
    "org_group": df_bpic14["org:group"].unique().to_numpy(dtype=str),
}

DYNAMIC_RAW_ATTRS = {
    "org_group_graph": -99
}

DYNAMIC_NUMERICAL_ATTRS = {}

In [ ]:
!unzip "/content/Data/Input/Detail_Incident_Activity_tabular_test_dataset.zip" -d "/content/Data/Input/Detail_Incident_Activity_tabular_test_dataset"
!unzip "/content/Data/Input/Detail_Incident_Activity_tabular_train_dataset.zip" -d "/content/Data/Input/Detail_Incident_Activity_tabular_train_dataset"

In [ ]:
ds_bpic14_train = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "Detail_Incident_Activity_train_dataset"),
    compression='GZIP'
)
ds_bpic14_train = ds_window_dynamic_attrs(ds_bpic14_train, MAX_SEQ_LEN)
print(len(ds_bpic14_train))

ds_bpic14_train = ds_cache_and_batch(ds_bpic14_train, shuffle=False)
ds_bpic14_train

In [ ]:
ds_bpic14_tab_train = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "Detail_Incident_Activity_tabular_train_dataset"),
    compression='GZIP'
)

print(len(ds_bpic14_tab_train))
ds_bpic14_tab_train = ds_remove_outputs(ds_bpic14_tab_train).batch(256).cache(os.path.join(INTERIM_DATA_DIR, 'bpic14_tabular_train_cache')).prefetch(tf.data.AUTOTUNE)
#ds_bpic14_tab_train = ds_cache_and_batch(ds_bpic14_tab_train, batch_size=256, shuffle=False)
ds_bpic14_tab_train

In [ ]:
ds_bpic14_test = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "Detail_Incident_Activity_test_dataset"),
    compression='GZIP'
)

ds_bpic14_test = ds_window_dynamic_attrs(ds_bpic14_test, MAX_SEQ_LEN)
print(len(ds_bpic14_test))

ds_bpic14_test = ds_cache_and_batch(ds_bpic14_test, shuffle=False)
ds_bpic14_test

In [ ]:
ds_bpic14_tab_test = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "Detail_Incident_Activity_tabular_test_dataset"),
    compression='GZIP'
)

print(len(ds_bpic14_tab_test))
ds_bpic14_tab_test = ds_remove_outputs(ds_bpic14_tab_test).batch(256).cache(os.path.join(INTERIM_DATA_DIR, 'bpic14_tabular_test_cache')).prefetch(tf.data.AUTOTUNE)
#ds_bpic14_tab_test = ds_cache_and_batch(ds_bpic14_tab_test, batch_size=256, shuffle=False)
ds_bpic14_tab_test

## ProcessLSTM

### Multitask ProcessLSTM

In [ ]:
model_name = f"lstm_{PROJECT_NAME}_multi"

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseProcessRNN(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    interval_since_last_event_vocab=INTERVAL_SINCE_LAST_EVENT_VOCAB,
    hour_of_day_vocab=HOUR_OF_DAY_VOCAB,
    day_of_week_vocab=DAY_OF_WEEK_VOCAB,
    output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'units': NUM_ACTIVITIES, 'activation': 'softmax'}, DEFAULT_NEXT_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}, DEFAULT_REMAINING_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}},
    name=model_name,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  #backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic14_train,
  validation_data=ds_bpic14_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_bpic14_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_bpic14_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)
model_regression_report(model, ds_bpic14_test, DEFAULT_NEXT_TIME_OUTPUT, archive=True)
model_classification_report(model, ds_bpic14_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

In [ ]:
logs = os.path.join(OUTPUT_LOG_DATA_DIR, model_name)
%tensorboard --logdir "$logs"

## DA-LSTM

### Remaining Time DA-LSTM

In [ ]:
model_name = f"dalstm_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}"

ds_bpic14_dalstm_train = ds_to_single_target(ds_sequentialize_static_attrs(ds_bpic14_train, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_REMAINING_TIME_OUTPUT)
ds_bpic14_dalstm_train = ds_cache_and_batch(ds_bpic14_dalstm_train, shuffle=False)

ds_bpic14_dalstm_test = ds_to_single_target(ds_sequentialize_static_attrs(ds_bpic14_test, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_REMAINING_TIME_OUTPUT)
ds_bpic14_dalstm_test = ds_cache_and_batch(ds_bpic14_dalstm_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseDARNN(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    categorical_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS,
    numerical_attrs=DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS | {"time_timestamp_elapsedcycle": None, "time_timestamp_elapsedprev": None, "time_timestamp_hour": None, "time_timestamp_weekday": None},
    output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}},
    name=model_name,
    rnn_dropout=0.1,
    ff_dropout=0.15,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic14_dalstm_train,
  validation_data=ds_bpic14_dalstm_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_bpic14_dalstm_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_bpic14_dalstm_test, DEFAULT_REMAINING_TIME_OUTPUT)

### Next Activity DA-LSTM

In [ ]:
model_name = f"dalstm_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}"

ds_bpic14_dalstm_train = ds_to_single_target(ds_sequentialize_static_attrs(ds_bpic14_train, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_bpic14_dalstm_train = ds_cache_and_batch(ds_bpic14_dalstm_train, shuffle=False)

ds_bpic14_dalstm_test = ds_to_single_target(ds_sequentialize_static_attrs(ds_bpic14_test, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_bpic14_dalstm_test = ds_cache_and_batch(ds_bpic14_dalstm_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseDARNN(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    categorical_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS,
    numerical_attrs=DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS | {"time_timestamp_elapsedcycle": None, "time_timestamp_elapsedprev": None, "time_timestamp_hour": None, "time_timestamp_weekday": None},
    output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'units': NUM_ACTIVITIES, 'activation': 'softmax'}},
    name=model_name,
    rnn_dropout=0.1,
    ff_dropout=0.15,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic14_dalstm_train,
  validation_data=ds_bpic14_dalstm_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_bpic14_dalstm_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_bpic14_dalstm_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## ProcessTransformer

### Remaining Time Prediction Process Transformer

In [ ]:
model_name = f"processtransformer_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}"

ds_bpic14_transformer_train = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_bpic14_train, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_REMAINING_TIME_OUTPUT)
ds_bpic14_transformer_train = ds_cache_and_batch(ds_bpic14_transformer_train, shuffle=False)

ds_bpic14_transformer_test = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_bpic14_test, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_REMAINING_TIME_OUTPUT)
ds_bpic14_transformer_test = ds_cache_and_batch(ds_bpic14_transformer_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseProcessTransformer(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    interval_since_last_event_vocab=INTERVAL_SINCE_LAST_EVENT_VOCAB,
    hour_of_day_vocab=HOUR_OF_DAY_VOCAB,
    day_of_week_vocab=DAY_OF_WEEK_VOCAB,
    output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}},
    name=model_name,
    activity_encoding_type='embedding'
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic14_transformer_train,
  validation_data=ds_bpic14_transformer_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_bpic14_transformer_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_bpic14_transformer_test, DEFAULT_REMAINING_TIME_OUTPUT)

In [ ]:
logs = os.path.join(OUTPUT_LOG_DATA_DIR, model_name)
%tensorboard --logdir "$logs"

### Next Activity Prediction Process Transformer

In [ ]:
model_name = f"processtransformer_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}"

ds_bpic14_transformer_train = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_bpic14_train, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_bpic14_transformer_train = ds_cache_and_batch(ds_bpic14_transformer_train, shuffle=False)

ds_bpic14_transformer_test = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_bpic14_test, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_bpic14_transformer_test = ds_cache_and_batch(ds_bpic14_transformer_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseProcessTransformer(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    interval_since_last_event_vocab=INTERVAL_SINCE_LAST_EVENT_VOCAB,
    hour_of_day_vocab=HOUR_OF_DAY_VOCAB,
    day_of_week_vocab=DAY_OF_WEEK_VOCAB,
    output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'units': NUM_ACTIVITIES, 'activation': 'softmax'}},
    name=model_name,
    activity_encoding_type='embedding'
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic14_transformer_train,
  validation_data=ds_bpic14_transformer_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_bpic14_transformer_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_bpic14_transformer_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

In [ ]:
logs = os.path.join(OUTPUT_LOG_DATA_DIR, model_name)
%tensorboard --logdir "$logs"

## ProcessSetTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cpst = ContrastiveProcessSetTransformer(
    name="bpic14_set_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      num_encoding_type='log_normal',
      embed_dim=64,
    ),
    embedding_model=SetTransformer(
        num_inductions=8,
        num_seeds=2,
        encoder_num_heads=4,
        encoder_dropout=0.1,
        decoder_num_heads=4,
        decoder_dropout=0.1,
    ),
    augmentation_1=SetAugmentationLayer(**MEDIUM_SET_AUGMENTATION),
    augmentation_2=SetAugmentationLayer(**STRONG_SET_AUGMENTATION),
)

cpst.build((256,2,1))

loss = cpst.default_loss
metrics = cpst.metrics

#lr_schedule = keras.optimizers.schedules.CosineDecay(
#  initial_learning_rate=0.001,
#  decay_steps=10000
#)
optimizer = cpst.optimizer
callbacks = cpst.callbacks

cpst.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cpst.summary()

In [ ]:
history = cpst.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic14_tab_train,
  validation_data=ds_bpic14_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cpst, history)

In [ ]:
cpst.evaluate(ds_bpic14_tab_train)

In [ ]:
cpst.evaluate(ds_bpic14_tab_test)

In [ ]:
model = cpst.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cpst.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_bpic14_tab_train))
test_embeddings = np.squeeze(model.predict(ds_bpic14_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_bpic14_train,
  df_bpic14_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_GROUP, 'case:KM number', 'case:incident_CI Name (aff)', 'case:incident_CI Type (aff)', 'case:incident_CI Subtype (aff)', 'case:incident_Service Component WBS (aff)', 'case:incident_Category', 'case:incident_KM number', 'case:incident_CI Name (CBy)', 'case:incident_CI Type (CBy)', 'case:incident_CI Subtype (CBy)'],
  num_attrs=['case:incident_Priority', "time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_bpic14_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_bpic14_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/bpic14_set_embedding_train_dataset.zip -d /content/Data/Input/bpic14_set_embedding_train_dataset
!unzip /content/Data/Input/bpic14_set_embedding_test_dataset.zip -d /content/Data/Input/bpic14_set_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: ProcessSetTransformer without Mix

In [ ]:
keras.utils.clear_session(free_memory=True)

cpst = ContrastiveProcessSetTransformer(
    name="bpic14_nomix_set_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      num_encoding_type='log_normal',
      embed_dim=64,
    ),
    embedding_model=SetTransformer(
        num_inductions=8,
        num_seeds=2,
        encoder_num_heads=4,
        encoder_dropout=0.1,
        decoder_num_heads=4,
        decoder_dropout=0.1,
    ),
    augmentation_1=SetAugmentationLayer(**MEDIUM_SET_AUGMENTATION_NOMIX),
    augmentation_2=SetAugmentationLayer(**STRONG_SET_AUGMENTATION_NOMIX),
)

cpst.build((256,2,1))

loss = cpst.default_loss
metrics = cpst.metrics

#lr_schedule = keras.optimizers.schedules.CosineDecay(
#  initial_learning_rate=0.001,
#  decay_steps=10000
#)
optimizer = cpst.optimizer
callbacks = cpst.callbacks

cpst.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cpst.summary()

In [ ]:
history = cpst.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic14_tab_train,
  validation_data=ds_bpic14_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cpst, history)

In [ ]:
cpst.evaluate(ds_bpic14_tab_train)

In [ ]:
cpst.evaluate(ds_bpic14_tab_test)

In [ ]:
model = cpst.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cpst.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_bpic14_tab_train))
test_embeddings = np.squeeze(model.predict(ds_bpic14_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_bpic14_train,
  df_bpic14_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_GROUP, 'case:KM number', 'case:incident_CI Name (aff)', 'case:incident_CI Type (aff)', 'case:incident_CI Subtype (aff)', 'case:incident_Service Component WBS (aff)', 'case:incident_Category', 'case:incident_KM number', 'case:incident_CI Name (CBy)', 'case:incident_CI Type (CBy)', 'case:incident_CI Subtype (CBy)'],
  num_attrs=['case:incident_Priority', "time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_bpic14_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_bpic14_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/bpic14_nomix_set_embedding_train_dataset.zip -d /content/Data/Input/bpic14_nomix_set_embedding_train_dataset
!unzip /content/Data/Input/bpic14_nomix_set_embedding_test_dataset.zip -d /content/Data/Input/bpic14_nomix_set_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False,
  run_eagerly=False,
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: Randomly Initialized SetTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cpst = ContrastiveProcessSetTransformer(
    name="bpic14_random_set_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      num_encoding_type='log_normal',
      embed_dim=64,
    ),
    embedding_model=SetTransformer(
        num_inductions=8,
        num_seeds=2,
        encoder_num_heads=4,
        encoder_dropout=0.1,
        decoder_num_heads=4,
        decoder_dropout=0.1,
    ),
    augmentation_1=SetAugmentationLayer(**MEDIUM_SET_AUGMENTATION),
    augmentation_2=SetAugmentationLayer(**STRONG_SET_AUGMENTATION),
)

cpst.build((256,2,1))

loss = cpst.default_loss
metrics = cpst.metrics

optimizer = cpst.optimizer
callbacks = cpst.callbacks

cpst.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)
cpst.trainable = False
cpst.summary()

In [ ]:
cpst.evaluate(ds_bpic14_tab_train)

In [ ]:
cpst.evaluate(ds_bpic14_tab_test)

In [ ]:
model = cpst.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cpst.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_bpic14_tab_train))
test_embeddings = np.squeeze(model.predict(ds_bpic14_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_bpic14_train,
  df_bpic14_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_GROUP, 'case:KM number', 'case:incident_CI Name (aff)', 'case:incident_CI Type (aff)', 'case:incident_CI Subtype (aff)', 'case:incident_Service Component WBS (aff)', 'case:incident_Category', 'case:incident_KM number', 'case:incident_CI Name (CBy)', 'case:incident_CI Type (CBy)', 'case:incident_CI Subtype (CBy)'],
  num_attrs=['case:incident_Priority', "time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_bpic14_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_bpic14_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip "/content/Data/Input/bpic14_random_set_embedding_test_dataset.zip" -d "/content/Data/Input/bpic14_random_set_embedding_test_dataset"
!unzip "/content/Data/Input/bpic14_random_set_embedding_train_dataset.zip" -d "/content/Data/Input/bpic14_random_set_embedding_train_dataset"

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## ProcessTabTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cptt = ContrastiveProcessTabTransformer(
    name="bpic14_tab_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      embed_dim=64,
      num_encoding_type='log_normal'
    ),
    embedding_model=TabTransformer(
        encoder_num_heads=4,
        encoder_dropout=0.1,
        ffn_dims=[256, 128],
    ),
    augmentation_1=TabularAugmentationLayer(**MEDIUM_TAB_AUGMENTATION),
    augmentation_2=TabularAugmentationLayer(**STRONG_TAB_AUGMENTATION),
)

cptt.build((256, len(cptt.input_encoder.cat_attrs), 32), (256, len(cptt.input_encoder.num_attrs), 32))

loss = cptt.default_loss
metrics = cptt.metrics

optimizer = cptt.optimizer
callbacks = cptt.callbacks

cptt.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cptt.summary()

In [ ]:
history = cptt.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic14_tab_train,
  validation_data=ds_bpic14_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cptt, history)

In [ ]:
cptt.evaluate(ds_bpic14_tab_train)

In [ ]:
cptt.evaluate(ds_bpic14_tab_test)

In [ ]:
model = cptt.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cptt.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_bpic14_tab_train))
test_embeddings = np.squeeze(model.predict(ds_bpic14_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_bpic14_train,
  df_bpic14_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_GROUP, 'case:KM number', 'case:incident_CI Name (aff)', 'case:incident_CI Type (aff)', 'case:incident_CI Subtype (aff)', 'case:incident_Service Component WBS (aff)', 'case:incident_Category', 'case:incident_KM number', 'case:incident_CI Name (CBy)', 'case:incident_CI Type (CBy)', 'case:incident_CI Subtype (CBy)'],
  num_attrs=['case:incident_Priority', "time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy,  num_samples=20000, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_bpic14_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_bpic14_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_test_dataset"))

In [ ]:
logs = os.path.join(OUTPUT_DATA_DIR, "bpic14_tab_embedding_tensorboard")
%tensorboard --logdir "$logs"

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/bpic14_tab_embedding_test_dataset.zip -d /content/Data/Input/bpic14_tab_embedding_test_dataset
!unzip /content/Data/Input/bpic14_tab_embedding_train_dataset.zip -d /content/Data/Input/bpic14_tab_embedding_train_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64],
    rnn_units_nonshared=[32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: ProcessTabTransformer without Mix

In [ ]:
keras.utils.clear_session(free_memory=True)

cptt = ContrastiveProcessTabTransformer(
    name="bpic14_nomix_tab_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      embed_dim=64,
      num_encoding_type='log_normal'
    ),
    embedding_model=TabTransformer(
        encoder_num_heads=4,
        encoder_dropout=0.1,
        ffn_dims=[256, 128],
    ),
    augmentation_1=TabularAugmentationLayer(**MEDIUM_TAB_AUGMENTATION_NOMIX),
    augmentation_2=TabularAugmentationLayer(**STRONG_TAB_AUGMENTATION_NOMIX),
)

cptt.build((256, len(cptt.input_encoder.cat_attrs), 32), (256, len(cptt.input_encoder.num_attrs), 32))

loss = cptt.default_loss
metrics = cptt.metrics

optimizer = cptt.optimizer
callbacks = cptt.callbacks

cptt.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cptt.summary()

In [ ]:
history = cptt.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic14_tab_train,
  validation_data=ds_bpic14_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cptt, history)

In [ ]:
cptt.evaluate(ds_bpic14_tab_train)

In [ ]:
cptt.evaluate(ds_bpic14_tab_test)

In [ ]:
model = cptt.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cptt.name}.keras"))
model.summary()

In [ ]:
#train_embeddings = np.squeeze(model.predict(ds_bpic14_tab_train))
#test_embeddings = np.squeeze(model.predict(ds_bpic14_tab_test))

#np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_bpic14_train,
  df_bpic14_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_GROUP, 'case:KM number', 'case:incident_CI Name (aff)', 'case:incident_CI Type (aff)', 'case:incident_CI Subtype (aff)', 'case:incident_Service Component WBS (aff)', 'case:incident_Category', 'case:incident_KM number', 'case:incident_CI Name (CBy)', 'case:incident_CI Type (CBy)', 'case:incident_CI Subtype (CBy)'],
  num_attrs=['case:incident_Priority', "time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_bpic14_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_bpic14_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_test_dataset"))

In [ ]:
logs = os.path.join(OUTPUT_DATA_DIR, "bpic14_tab_embedding_tensorboard")
%tensorboard --logdir "$logs"

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/bpic14_nomix_tab_embedding_test_dataset.zip -d /content/Data/Input/bpic14_nomix_tab_embedding_test_dataset
!unzip /content/Data/Input/bpic14_nomix_tab_embedding_train_dataset.zip -d /content/Data/Input/bpic14_nomix_tab_embedding_train_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64],
    rnn_units_nonshared=[32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: Randomly Initialized TabTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cptt = ContrastiveProcessTabTransformer(
    name="bpic14_random_tab_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      embed_dim=64,
      num_encoding_type='log_normal'
    ),
    embedding_model=TabTransformer(
        encoder_num_heads=4,
        encoder_dropout=0.1,
        ffn_dims=[256, 128],
    ),
    augmentation_1=TabularAugmentationLayer(**MEDIUM_TAB_AUGMENTATION),
    augmentation_2=TabularAugmentationLayer(**STRONG_TAB_AUGMENTATION),
)

cptt.build((256, len(cptt.input_encoder.cat_attrs), 32), (256, len(cptt.input_encoder.num_attrs), 32))

loss = cptt.default_loss
metrics = cptt.metrics

optimizer = cptt.optimizer
callbacks = cptt.callbacks

cptt.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)
cptt.trainable = False
cptt.summary()

In [ ]:
cptt.evaluate(ds_bpic14_tab_train)

In [ ]:
cptt.evaluate(ds_bpic14_tab_test)

In [ ]:
model = cptt.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cptt.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_bpic14_tab_train))
test_embeddings = np.squeeze(model.predict(ds_bpic14_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_bpic14_train,
  df_bpic14_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_GROUP, 'case:KM number', 'case:incident_CI Name (aff)', 'case:incident_CI Type (aff)', 'case:incident_CI Subtype (aff)', 'case:incident_Service Component WBS (aff)', 'case:incident_Category', 'case:incident_KM number', 'case:incident_CI Name (CBy)', 'case:incident_CI Type (CBy)', 'case:incident_CI Subtype (CBy)'],
  num_attrs=['case:incident_Priority', "time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy,  num_samples=20000, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_bpic14_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_bpic14_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/bpic14_random_tab_embedding_test_dataset.zip -d /content/Data/Input/bpic14_random_tab_embedding_test_dataset
!unzip /content/Data/Input/bpic14_random_tab_embedding_train_dataset.zip -d /content/Data/Input/bpic14_random_tab_embedding_train_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64],
    rnn_units_nonshared=[32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

# Dataset: Helpdesk

## Input Preparation

In [ ]:
df_helpdesk = pd.read_feather(os.path.join(INPUT_DATA_DIR, "helpdesk_labeled.feather"))
df_helpdesk

In [ ]:
df_helpdesk_train = pd.read_feather(os.path.join(INPUT_DATA_DIR, "helpdesk_train.feather"))
df_helpdesk_train

In [ ]:
df_helpdesk_test = pd.read_feather(os.path.join(INPUT_DATA_DIR, "helpdesk_test.feather"))
df_helpdesk_test

In [ ]:
MAX_SEQ_LEN = 6
PROJECT_NAME = "helpdesk"
TRAIN_STEPS_EPOCH = len(df_helpdesk_train) // DEFAULT_BATCH_SIZE

TIME_ATTRS = {
    'month': "time_timestamp_month",
    'hour': "time_timestamp_hour",
    'day': "time_timestamp_day",
    'weekday': "time_timestamp_weekday",
    'yearday': "time_timestamp_dayofyear",
}

ACTIVITY_VOCAB = np.array(df_helpdesk[EVENTLOG_ACTIVITY].to_list() + [TOKEN_NA, TOKEN_EOC])
NUM_ACTIVITIES = np.unique(ACTIVITY_VOCAB).shape[0]

INTERVAL_SINCE_START_VOCAB = df_helpdesk_train["time:timestamp:elapsedcycle:seconds"].to_numpy()
INTERVAL_SINCE_LAST_EVENT_VOCAB = df_helpdesk_train["time:timestamp:elapsedprev:seconds"].to_numpy()
DAY_OF_WEEK_VOCAB = df_helpdesk_train["time:timestamp:weekday:raw"].to_numpy()
HOUR_OF_DAY_VOCAB = df_helpdesk_train["time:timestamp:hour:raw"].to_numpy()

TIME_ATTRS_VOCAB = {
  # Already scaled
  "time_timestamp_month": None,
  "time_timestamp_hour": None,
  "time_timestamp_day": None,
  "time_timestamp_weekday": None,
  "time_timestamp_dayofyear": None,
  # Need Scaling
  "time_timestamp_elapsedprev": INTERVAL_SINCE_LAST_EVENT_VOCAB,
  "time_timestamp_elapsedcycle": INTERVAL_SINCE_START_VOCAB,
}

STATIC_CATEGORICAL_ATTRS = {}
STATIC_NUMERICAL_ATTRS = {}

DYNAMIC_CATEGORICAL_ATTRS = {}
DYNAMIC_NUMERICAL_ATTRS = {}

In [ ]:
!unzip "/content/Data/Input/helpdesk_tabular_train_dataset.zip" -d "/content/Data/Input/helpdesk_tabular_train_dataset"
!unzip "/content/Data/Input/helpdesk_tabular_test_dataset.zip" -d "/content/Data/Input/helpdesk_tabular_test_dataset"

In [ ]:
ds_helpdesk_train = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "helpdesk_train_dataset"),
    compression='GZIP'
)
ds_helpdesk_train = ds_window_dynamic_attrs(ds_helpdesk_train, MAX_SEQ_LEN)
print(len(ds_helpdesk_train))

ds_helpdesk_train = ds_cache_and_batch(ds_helpdesk_train, shuffle=False)
ds_helpdesk_train

In [ ]:
ds_helpdesk_tab_train = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "helpdesk_tabular_train_dataset"),
    compression='GZIP'
)

print(len(ds_helpdesk_tab_train))
ds_helpdesk_tab_train = ds_remove_outputs(ds_helpdesk_tab_train).batch(256)
#ds_helpdesk_tab_train = ds_cache_and_batch(ds_helpdesk_tab_train, batch_size=256, shuffle=False)
ds_helpdesk_tab_train

In [ ]:
ds_helpdesk_test = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "helpdesk_test_dataset"),
    compression='GZIP'
)
ds_helpdesk_test = ds_window_dynamic_attrs(ds_helpdesk_test, MAX_SEQ_LEN)
print(len(ds_helpdesk_test))

ds_helpdesk_test = ds_cache_and_batch(ds_helpdesk_test, shuffle=False)
ds_helpdesk_test

In [ ]:
ds_helpdesk_tab_test = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "helpdesk_tabular_test_dataset"),
    compression='GZIP'
)

print(len(ds_helpdesk_tab_test))
ds_helpdesk_tab_test = ds_remove_outputs(ds_helpdesk_tab_test).batch(256)
#ds_helpdesk_tab_test = ds_cache_and_batch(ds_helpdesk_tab_test, batch_size=256, shuffle=False)
ds_helpdesk_tab_test

## ProcessLSTM

### Multitask LSTM

In [ ]:
model_name = f"lstm_{PROJECT_NAME}_multi"

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseProcessRNN(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    interval_since_last_event_vocab=INTERVAL_SINCE_LAST_EVENT_VOCAB,
    hour_of_day_vocab=HOUR_OF_DAY_VOCAB,
    day_of_week_vocab=DAY_OF_WEEK_VOCAB,
    output_dict={"next_activity": {'units': NUM_ACTIVITIES, 'activation': 'softmax'}, "next_time": {'units': 1, 'activation': 'linear'}, "remaining_time": {'units': 1, 'activation': 'linear'}},
    name=model_name,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_helpdesk_train,
  validation_data=ds_helpdesk_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_helpdesk_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_helpdesk_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)
model_regression_report(model, ds_helpdesk_test, DEFAULT_NEXT_TIME_OUTPUT, archive=True)
model_classification_report(model, ds_helpdesk_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

In [ ]:
logs = os.path.join(OUTPUT_LOG_DATA_DIR, model_name)
%tensorboard --logdir "$logs"

### Remaining Time DA-LSTM

In [ ]:
model_name = f"dalstm_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}"

ds_helpdesk_dalstm_train = ds_to_single_target(ds_sequentialize_static_attrs(ds_helpdesk_train, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_REMAINING_TIME_OUTPUT)
ds_helpdesk_dalstm_train = ds_cache_and_batch(ds_helpdesk_dalstm_train, shuffle=False)

ds_helpdesk_dalstm_test = ds_to_single_target(ds_sequentialize_static_attrs(ds_helpdesk_test, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_REMAINING_TIME_OUTPUT)
ds_helpdesk_dalstm_test = ds_cache_and_batch(ds_helpdesk_dalstm_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseDARNN(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    numerical_attrs={"time_timestamp_elapsedcycle": None, "time_timestamp_elapsedprev": None, "time_timestamp_hour": None, "time_timestamp_weekday": None},
    output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}},
    name=model_name,
    rnn_dropout=0.1,
    ff_dropout=0.15,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_helpdesk_dalstm_train,
  validation_data=ds_helpdesk_dalstm_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_helpdesk_dalstm_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_helpdesk_dalstm_test, DEFAULT_REMAINING_TIME_OUTPUT)

### Next Activity DA-LSTM

In [ ]:
model_name = f"dalstm_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}"

ds_helpdesk_dalstm_train = ds_to_single_target(ds_sequentialize_static_attrs(ds_helpdesk_train, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_helpdesk_dalstm_train = ds_cache_and_batch(ds_helpdesk_dalstm_train, shuffle=False)

ds_helpdesk_dalstm_test = ds_to_single_target(ds_sequentialize_static_attrs(ds_helpdesk_test, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_helpdesk_dalstm_test = ds_cache_and_batch(ds_helpdesk_dalstm_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseDARNN(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    numerical_attrs={"time_timestamp_elapsedcycle": None, "time_timestamp_elapsedprev": None, "time_timestamp_hour": None, "time_timestamp_weekday": None},
    output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'units': NUM_ACTIVITIES, 'activation': 'softmax'}},
    name=model_name,
    rnn_dropout=0.1,
    ff_dropout=0.15,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_helpdesk_dalstm_train,
  validation_data=ds_helpdesk_dalstm_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_helpdesk_dalstm_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_helpdesk_dalstm_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## ProcessTransformer

### Remaining Time Process Transformer

In [ ]:
model_name = f"processtransformer_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}"

ds_helpdesk_transformer_train = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_helpdesk_train, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_REMAINING_TIME_OUTPUT)
ds_helpdesk_transformer_train = ds_cache_and_batch(ds_helpdesk_transformer_train, shuffle=False)

ds_helpdesk_transformer_test = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_helpdesk_test, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_REMAINING_TIME_OUTPUT)
ds_helpdesk_transformer_test = ds_cache_and_batch(ds_helpdesk_transformer_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)


model = BaseProcessTransformer(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    interval_since_last_event_vocab=INTERVAL_SINCE_LAST_EVENT_VOCAB,
    hour_of_day_vocab=HOUR_OF_DAY_VOCAB,
    day_of_week_vocab=DAY_OF_WEEK_VOCAB,
    output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}},
    name=model_name,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_helpdesk_transformer_train,
  validation_data=ds_helpdesk_transformer_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_helpdesk_transformer_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_helpdesk_transformer_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

In [ ]:
logs = os.path.join(OUTPUT_LOG_DATA_DIR, model_name)
%tensorboard --logdir "$logs"

### Next Activity Process Transformer

In [ ]:
model_name = f"processtransformer_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}"

ds_helpdesk_transformer_train = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_helpdesk_train, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_helpdesk_transformer_train = ds_cache_and_batch(ds_helpdesk_transformer_train, shuffle=False)

ds_helpdesk_transformer_test = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_helpdesk_test, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_helpdesk_transformer_test = ds_cache_and_batch(ds_helpdesk_transformer_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseProcessTransformer(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    interval_since_last_event_vocab=INTERVAL_SINCE_LAST_EVENT_VOCAB,
    hour_of_day_vocab=HOUR_OF_DAY_VOCAB,
    day_of_week_vocab=DAY_OF_WEEK_VOCAB,
    output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'units': NUM_ACTIVITIES, 'activation': 'softmax'}},
    name=model_name,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_helpdesk_transformer_train,
  validation_data=ds_helpdesk_transformer_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_helpdesk_transformer_test)
model_visualize_history(model, history)

In [ ]:
x = np.load("/content/Data/Output/lstm_helpdesk_multi_next_activity.npz")
x

In [ ]:
model_classification_report(model, ds_helpdesk_transformer_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

In [ ]:
logs = os.path.join(OUTPUT_LOG_DATA_DIR, model_name)
%tensorboard --logdir "$logs"

## ProcessSetTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cpst = ContrastiveProcessSetTransformer(
    name="helpdesk_set_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[32, 16],
    projection_dim=8,
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs={ 'activity': ACTIVITY_VOCAB },
      embed_dim=8,
      num_encoding_type='log_normal'
    ),
    embedding_model=SetTransformer(
        num_inductions=4,
        num_seeds=1,
        encoder_num_heads=2,
        encoder_dropout=0.1,
        decoder_num_heads=4,
        decoder_dropout=0.1,
    ),
    augmentation_1=SetAugmentationLayer(**WEAK_SET_AUGMENTATION),
    augmentation_2=SetAugmentationLayer(**MEDIUM_SET_AUGMENTATION),

)

cpst.build((256,1,1))

loss = cpst.default_loss
metrics = cpst.metrics

#lr_schedule = keras.optimizers.schedules.CosineDecay(
#  initial_learning_rate=0.001,
#  decay_steps=10000
#)
optimizer = cpst.optimizer
callbacks = cpst.callbacks

cpst.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cpst.summary()

In [ ]:
history = cpst.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_helpdesk_tab_train,
  validation_data=ds_helpdesk_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cpst, history)

In [ ]:
cpst.evaluate(ds_helpdesk_tab_train)

In [ ]:
cpst.evaluate(ds_helpdesk_tab_test)

In [ ]:
model = cpst.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cpst.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_helpdesk_tab_train))
test_embeddings = np.squeeze(model.predict(ds_helpdesk_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_helpdesk_train,
  df_helpdesk_test,
  cat_attrs=[EVENTLOG_ACTIVITY],
  num_attrs=["time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_helpdesk_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_helpdesk_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 64

In [ ]:
!unzip /content/Data/Input/helpdesk_set_embedding_train_dataset.zip -d /content/Data/Input/helpdesk_set_embedding_train_dataset
!unzip /content/Data/Input/helpdesk_set_embedding_test_dataset.zip -d /content/Data/Input/helpdesk_set_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[8, 8],
    rnn_dropout=0.2,
    ff_dims=[16, 8, 4],
    ff_dropout=0.2,
    name=f"lstmset_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[8, 8],
    rnn_dropout=0.2,
    ff_dims=[16, 8, 4],
    ff_dropout=0.2,
    name=f"lstmset_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[2],
  enc_ff_dims_shared=[32],
  enc_heads_nonshared=[2],
  enc_ff_dims_nonshared=[32],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[16, 8, 4],
  ff_dropout=0.2,
  name=f"transformerset_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[2],
  enc_ff_dims_shared=[32],
  enc_heads_nonshared=[2],
  enc_ff_dims_nonshared=[32],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[16, 8, 4],
  ff_dropout=0.2,
  name=f"transformerset_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: ProcessSetTransformer without mix

In [ ]:
keras.utils.clear_session(free_memory=True)

cpst = ContrastiveProcessSetTransformer(
    name="helpdesk_nomix_set_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[32, 16],
    projection_dim=8,
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs={ 'activity': ACTIVITY_VOCAB },
      embed_dim=8,
      num_encoding_type='log_normal'
    ),
    embedding_model=SetTransformer(
        num_inductions=4,
        num_seeds=1,
        encoder_num_heads=2,
        encoder_dropout=0.1,
        decoder_num_heads=4,
        decoder_dropout=0.1,
    ),
    augmentation_1=SetAugmentationLayer(**WEAK_SET_AUGMENTATION_NOMIX),
    augmentation_2=SetAugmentationLayer(**MEDIUM_SET_AUGMENTATION_NOMIX),

)

cpst.build((256,1,1))

loss = cpst.default_loss
metrics = cpst.metrics

#lr_schedule = keras.optimizers.schedules.CosineDecay(
#  initial_learning_rate=0.001,
#  decay_steps=10000
#)
optimizer = cpst.optimizer
callbacks = cpst.callbacks

cpst.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cpst.summary()

In [ ]:
history = cpst.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_helpdesk_tab_train,
  validation_data=ds_helpdesk_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cpst, history)

In [ ]:
cpst.evaluate(ds_helpdesk_tab_train)

In [ ]:
cpst.evaluate(ds_helpdesk_tab_test)

In [ ]:
model = cpst.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cpst.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_helpdesk_tab_train))
test_embeddings = np.squeeze(model.predict(ds_helpdesk_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_helpdesk_train,
  df_helpdesk_test,
  cat_attrs=[EVENTLOG_ACTIVITY],
  num_attrs=["time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_helpdesk_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_helpdesk_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 64

In [ ]:
!unzip /content/Data/Input/helpdesk_nomix_set_embedding_train_dataset.zip -d /content/Data/Input/helpdesk_nomix_set_embedding_train_dataset
!unzip /content/Data/Input/helpdesk_nomix_set_embedding_test_dataset.zip -d /content/Data/Input/helpdesk_nomix_set_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[8, 8],
    rnn_dropout=0.2,
    ff_dims=[16, 8, 4],
    ff_dropout=0.2,
    name=f"lstmset_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[8, 8],
    rnn_dropout=0.2,
    ff_dims=[16, 8, 4],
    ff_dropout=0.2,
    name=f"lstmset_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[2],
  enc_ff_dims_shared=[32],
  enc_heads_nonshared=[2],
  enc_ff_dims_nonshared=[32],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[16, 8, 4],
  ff_dropout=0.2,
  name=f"transformerset_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[2],
  enc_ff_dims_shared=[32],
  enc_heads_nonshared=[2],
  enc_ff_dims_nonshared=[32],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[16, 8, 4],
  ff_dropout=0.2,
  name=f"transformerset_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: Randomly Initialized SetTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cpst = ContrastiveProcessSetTransformer(
    name="helpdesk_random_set_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      num_encoding_type='log_normal',
      embed_dim=64,
    ),
    embedding_model=SetTransformer(
        num_inductions=8,
        num_seeds=2,
        encoder_num_heads=4,
        encoder_dropout=0.1,
        decoder_num_heads=4,
        decoder_dropout=0.1,
    ),
    augmentation_1=SetAugmentationLayer(**MEDIUM_SET_AUGMENTATION),
    augmentation_2=SetAugmentationLayer(**STRONG_SET_AUGMENTATION),
)

cpst.build((256,2,1))

loss = cpst.default_loss
metrics = cpst.metrics

optimizer = cpst.optimizer
callbacks = cpst.callbacks

cpst.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)
cpst.trainable = False
cpst.summary()

In [ ]:
cpst.evaluate(ds_helpdesk_tab_train)

In [ ]:
cpst.evaluate(ds_helpdesk_tab_test)

In [ ]:
model = cpst.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cpst.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_helpdesk_tab_train))
test_embeddings = np.squeeze(model.predict(ds_helpdesk_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_helpdesk_train,
  df_helpdesk_test,
  cat_attrs=[EVENTLOG_ACTIVITY],
  num_attrs=["time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_helpdesk_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_helpdesk_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 64

In [ ]:
!unzip /content/Data/Input/helpdesk_random_set_embedding_train_dataset.zip -d /content/Data/Input/helpdesk_random_set_embedding_train_dataset
!unzip /content/Data/Input/helpdesk_random_set_embedding_test_dataset.zip -d /content/Data/Input/helpdesk_random_set_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[8, 8],
    rnn_dropout=0.2,
    ff_dims=[16, 8, 4],
    ff_dropout=0.2,
    name=f"lstmset_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[8, 8],
    rnn_dropout=0.2,
    ff_dims=[16, 8, 4],
    ff_dropout=0.2,
    name=f"lstmset_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[2],
  enc_ff_dims_shared=[32],
  enc_heads_nonshared=[2],
  enc_ff_dims_nonshared=[32],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[16, 8, 4],
  ff_dropout=0.2,
  name=f"transformerset_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[2],
  enc_ff_dims_shared=[32],
  enc_heads_nonshared=[2],
  enc_ff_dims_nonshared=[32],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[16, 8, 4],
  ff_dropout=0.2,
  name=f"transformerset_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## ProcessTabTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cptt = ContrastiveProcessTabTransformer(
    name="helpdesk_tab_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[32, 16],
    projection_dim=8,
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      embed_dim=8,
      num_encoding_type='log_normal'
    ),
    embedding_model=TabTransformer(
        encoder_num_heads=2,
        encoder_dropout=0.1,
        ffn_dims=[128, 64, 8],
    ),
    augmentation_1=TabularAugmentationLayer(**WEAK_TAB_AUGMENTATION),
    augmentation_2=TabularAugmentationLayer(**MEDIUM_TAB_AUGMENTATION),

)

cptt.build((256, len(cptt.input_encoder.cat_attrs), 32), (256, len(cptt.input_encoder.num_attrs), 32))

loss = cptt.default_loss
metrics = cptt.metrics

optimizer = cptt.optimizer
callbacks = cptt.callbacks

cptt.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cptt.summary()

In [ ]:
history = cptt.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_helpdesk_tab_train,
  validation_data=ds_helpdesk_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cptt, history)

In [ ]:
cptt.evaluate(ds_helpdesk_tab_train)

In [ ]:
cptt.evaluate(ds_helpdesk_tab_test)

In [ ]:
model = cptt.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cptt.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_helpdesk_tab_train))
test_embeddings = np.squeeze(model.predict(ds_helpdesk_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_helpdesk_train,
  df_helpdesk_test,
  cat_attrs=[EVENTLOG_ACTIVITY],
  num_attrs=["time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_helpdesk_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_helpdesk_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_test_dataset"))

In [ ]:
logs = os.path.join(OUTPUT_DATA_DIR, "helpdesk_tab_embedding_tensorboard")
%tensorboard --logdir "$logs"

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 64

In [ ]:
!unzip /content/Data/Input/helpdesk_tab_embedding_train_dataset.zip -d /content/Data/Input/helpdesk_tab_embedding_train_dataset
!unzip /content/Data/Input/helpdesk_tab_embedding_test_dataset.zip -d /content/Data/Input/helpdesk_tab_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[8, 8],
    rnn_dropout=0.2,
    ff_dims=[16, 8, 4],
    ff_dropout=0.2,
    name=f"lstmtab_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[8, 8],
    rnn_dropout=0.2,
    ff_dims=[16, 8, 4],
    ff_dropout=0.2,
    name=f"lstmtab_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[2],
  enc_ff_dims_shared=[32],
  enc_heads_nonshared=[2],
  enc_ff_dims_nonshared=[32],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[16, 8, 4],
  ff_dropout=0.2,
  name=f"transformertab_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[2],
  enc_ff_dims_shared=[32],
  enc_heads_nonshared=[2],
  enc_ff_dims_nonshared=[32],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[16, 8, 4],
  ff_dropout=0.2,
  name=f"transformertab_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: ProcessTabTransformer without Mix

In [ ]:
keras.utils.clear_session(free_memory=True)

cptt = ContrastiveProcessTabTransformer(
    name="helpdesk_nomix_tab_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[32, 16],
    projection_dim=8,
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      embed_dim=8,
      num_encoding_type='log_normal'
    ),
    embedding_model=TabTransformer(
        encoder_num_heads=2,
        encoder_dropout=0.1,
        ffn_dims=[128, 64, 8],
    ),
    augmentation_1=TabularAugmentationLayer(**WEAK_TAB_AUGMENTATION_NOMIX),
    augmentation_2=TabularAugmentationLayer(**MEDIUM_TAB_AUGMENTATION_NOMIX),
)

cptt.build((256, len(cptt.input_encoder.cat_attrs), 32), (256, len(cptt.input_encoder.num_attrs), 32))

loss = cptt.default_loss
metrics = cptt.metrics

optimizer = cptt.optimizer
callbacks = cptt.callbacks

cptt.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cptt.summary()

In [ ]:
history = cptt.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_helpdesk_tab_train,
  validation_data=ds_helpdesk_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cptt, history)

In [ ]:
cptt.evaluate(ds_helpdesk_tab_train)

In [ ]:
cptt.evaluate(ds_helpdesk_tab_test)

In [ ]:
model = cptt.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cptt.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_helpdesk_tab_train))
test_embeddings = np.squeeze(model.predict(ds_helpdesk_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_helpdesk_train,
  df_helpdesk_test,
  cat_attrs=[EVENTLOG_ACTIVITY],
  num_attrs=["time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_helpdesk_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_helpdesk_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_test_dataset"))

In [ ]:
logs = os.path.join(OUTPUT_DATA_DIR, "helpdesk_tab_embedding_tensorboard")
%tensorboard --logdir "$logs"

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 64

In [ ]:
!unzip /content/Data/Input/helpdesk_nomix_tab_embedding_train_dataset.zip -d /content/Data/Input/helpdesk_nomix_tab_embedding_train_dataset
!unzip /content/Data/Input/helpdesk_nomix_tab_embedding_test_dataset.zip -d /content/Data/Input/helpdesk_nomix_tab_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[8, 8],
    rnn_dropout=0.2,
    ff_dims=[16, 8, 4],
    ff_dropout=0.2,
    name=f"lstmtab_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[8, 8],
    rnn_dropout=0.2,
    ff_dims=[16, 8, 4],
    ff_dropout=0.2,
    name=f"lstmtab_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[2],
  enc_ff_dims_shared=[32],
  enc_heads_nonshared=[2],
  enc_ff_dims_nonshared=[32],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[16, 8, 4],
  ff_dropout=0.2,
  name=f"transformertab_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[2],
  enc_ff_dims_shared=[32],
  enc_heads_nonshared=[2],
  enc_ff_dims_nonshared=[32],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[16, 8, 4],
  ff_dropout=0.2,
  name=f"transformertab_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: Randomly Initialized TabTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cptt = ContrastiveProcessTabTransformer(
    name="helpdesk_random_tab_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      embed_dim=64,
      num_encoding_type='log_normal'
    ),
    embedding_model=TabTransformer(
        encoder_num_heads=4,
        encoder_dropout=0.1,
        ffn_dims=[256, 128],
    ),
    augmentation_1=TabularAugmentationLayer(**MEDIUM_TAB_AUGMENTATION),
    augmentation_2=TabularAugmentationLayer(**STRONG_TAB_AUGMENTATION),
)

cptt.build((256, len(cptt.input_encoder.cat_attrs), 32), (256, len(cptt.input_encoder.num_attrs), 32))

loss = cptt.default_loss
metrics = cptt.metrics

optimizer = cptt.optimizer
callbacks = cptt.callbacks

cptt.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)
cptt.trainable = False
cptt.summary()

In [ ]:
cptt.evaluate(ds_helpdesk_tab_train)

In [ ]:
cptt.evaluate(ds_helpdesk_tab_test)

In [ ]:
model = cptt.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cptt.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_helpdesk_tab_train))
test_embeddings = np.squeeze(model.predict(ds_helpdesk_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_helpdesk_train,
  df_helpdesk_test,
  cat_attrs=[EVENTLOG_ACTIVITY],
  num_attrs=["time:timestamp:elapsedcycle:seconds", "time:timestamp:elapsedprev:seconds", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_helpdesk_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_helpdesk_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 64

In [ ]:
!unzip /content/Data/Input/helpdesk_random_tab_embedding_train_dataset.zip -d /content/Data/Input/helpdesk_random_tab_embedding_train_dataset
!unzip /content/Data/Input/helpdesk_random_tab_embedding_test_dataset.zip -d /content/Data/Input/helpdesk_random_tab_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[8, 8],
    rnn_dropout=0.2,
    ff_dims=[16, 8, 4],
    ff_dropout=0.2,
    name=f"lstmtab_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[8, 8],
    rnn_dropout=0.2,
    ff_dims=[16, 8, 4],
    ff_dropout=0.2,
    name=f"lstmtab_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[2],
  enc_ff_dims_shared=[32],
  enc_heads_nonshared=[2],
  enc_ff_dims_nonshared=[32],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[16, 8, 4],
  ff_dropout=0.2,
  name=f"transformertab_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 64

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.shuffle(ds_train.cardinality()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[2],
  enc_ff_dims_shared=[32],
  enc_heads_nonshared=[2],
  enc_ff_dims_nonshared=[32],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[16, 8, 4],
  ff_dropout=0.2,
  name=f"transformertab_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.005, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

# Dataset: BPIC 2013

## Input Preparation

In [ ]:
df_bpic13 = pd.read_feather(os.path.join(INPUT_DATA_DIR, "BPI_Challenge_2013_incidents_labeled.feather"))
df_bpic13

In [ ]:
df_bpic13_train = pd.read_feather(os.path.join(INPUT_DATA_DIR, "BPI_Challenge_2013_incidents_train.feather"))
df_bpic13_train

In [ ]:
df_bpic13_test = pd.read_feather(os.path.join(INPUT_DATA_DIR, "BPI_Challenge_2013_incidents_test.feather"))
df_bpic13_test

In [ ]:
MAX_SEQ_LEN = 23
PROJECT_NAME = "bpic13"
TRAIN_STEPS_EPOCH = len(df_bpic13_train) // DEFAULT_BATCH_SIZE

TIME_ATTRS = {
    'month': "time_timestamp_month",
    'hour': "time_timestamp_hour",
    'day': "time_timestamp_day",
    'weekday': "time_timestamp_weekday",
    'yearday': "time_timestamp_dayofyear",
}
ACTIVITY_VOCAB = np.array(df_bpic13[EVENTLOG_ACTIVITY].to_list() + [TOKEN_NA, TOKEN_EOC])
NUM_ACTIVITIES = np.unique(ACTIVITY_VOCAB).shape[0]

INTERVAL_SINCE_START_VOCAB = df_bpic13_train["time:timestamp:elapsedcycle"].to_numpy()
INTERVAL_SINCE_LAST_EVENT_VOCAB = df_bpic13_train["time:timestamp:elapsedprev"].to_numpy()
DAY_OF_WEEK_VOCAB = df_bpic13_train["time:timestamp:weekday:raw"].to_numpy()
HOUR_OF_DAY_VOCAB = df_bpic13_train["time:timestamp:hour:raw"].to_numpy()

TIME_ATTRS_VOCAB = {
  # Already scaled
  "time_timestamp_month": None,
  "time_timestamp_hour": None,
  "time_timestamp_day": None,
  "time_timestamp_weekday": None,
  "time_timestamp_dayofyear": None,
  # Need Scaling
  "time_timestamp_elapsedprev": INTERVAL_SINCE_LAST_EVENT_VOCAB,
  "time_timestamp_elapsedcycle": INTERVAL_SINCE_START_VOCAB,
}

STATIC_CATEGORICAL_ATTRS = {
    "case_product": df_bpic13["case:Product"].unique().to_numpy(dtype=str),
    "case_country": df_bpic13["case:Country"].unique().to_numpy(dtype=str),
}
STATIC_NUMERICAL_ATTRS = {
    "case_latest_impact": None,
}

DYNAMIC_CATEGORICAL_ATTRS = {
  "status": df_bpic13["Status"].unique().to_numpy(dtype=str),
  "sub_status": df_bpic13["Sub Status"].unique().to_numpy(dtype=str),
  "org_role": df_bpic13["org:role"].unique().to_numpy(dtype=str),
  "org_line": df_bpic13["Involved Org line 3"].unique().to_numpy(dtype=str),
  "org_group": df_bpic13["org:group"].unique().to_numpy(dtype=str),
  "org_resource": df_bpic13["org:resource"].unique().to_numpy(dtype=str),
  "owner_country": df_bpic13["Owner Country"].unique().to_numpy(dtype=str),
}

DYNAMIC_RAW_ATTRS = {
  "org_group_graph": -99,
  "org_resource_graph": -99,
}

DYNAMIC_NUMERICAL_ATTRS = {}

In [ ]:
!unzip "/content/Data/Input/BPI_Challenge_2013_incidents_tabular_train_dataset.zip" -d "/content/Data/Input/BPI_Challenge_2013_incidents_tabular_train_dataset"
!unzip "/content/Data/Input/BPI_Challenge_2013_incidents_tabular_test_dataset.zip" -d "/content/Data/Input/BPI_Challenge_2013_incidents_tabular_test_dataset"

In [ ]:
ds_bpic13_train = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "BPI_Challenge_2013_incidents_train_dataset"),
    compression='GZIP'
)
ds_bpic13_train = ds_window_dynamic_attrs(ds_bpic13_train, MAX_SEQ_LEN)
print(len(ds_bpic13_train))

ds_bpic13_train = ds_cache_and_batch(ds_bpic13_train, shuffle=False)
ds_bpic13_train

In [ ]:
ds_bpic13_tab_train = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "BPI_Challenge_2013_incidents_tabular_train_dataset"),
    compression='GZIP'
)

print(len(ds_bpic13_tab_train))
ds_bpic13_tab_train = ds_remove_outputs(ds_bpic13_tab_train)
ds_bpic13_tab_train = ds_cache_and_batch(ds_bpic13_tab_train, batch_size=256, shuffle=False)
ds_bpic13_tab_train

In [ ]:
ds_bpic13_test = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "BPI_Challenge_2013_incidents_test_dataset"),
    compression='GZIP'
)

ds_bpic13_test = ds_window_dynamic_attrs(ds_bpic13_test, MAX_SEQ_LEN)
print(len(ds_bpic13_test))

ds_bpic13_test = ds_cache_and_batch(ds_bpic13_test, shuffle=False)
ds_bpic13_test

In [ ]:
ds_bpic13_tab_test = Dataset.load(
    os.path.join(INPUT_DATA_DIR, "BPI_Challenge_2013_incidents_tabular_test_dataset"),
    compression='GZIP'
)

print(len(ds_bpic13_tab_test))
ds_bpic13_tab_test = ds_remove_outputs(ds_bpic13_tab_test)
ds_bpic13_tab_test = ds_cache_and_batch(ds_bpic13_tab_test, batch_size=256, shuffle=False)
ds_bpic13_tab_test

## ProcessLSTM

### Multitask LSTM

In [ ]:
model_name = f"lstm_{PROJECT_NAME}_multi"

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseProcessRNN(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    interval_since_last_event_vocab=INTERVAL_SINCE_LAST_EVENT_VOCAB,
    hour_of_day_vocab=HOUR_OF_DAY_VOCAB,
    day_of_week_vocab=DAY_OF_WEEK_VOCAB,
    output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'units': NUM_ACTIVITIES, 'activation': 'softmax'}, DEFAULT_NEXT_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}, DEFAULT_REMAINING_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}},
    name=model_name,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic13_train,
  validation_data=ds_bpic13_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_bpic13_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_bpic13_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)
model_regression_report(model, ds_bpic13_test, DEFAULT_NEXT_TIME_OUTPUT, archive=True)
model_classification_report(model, ds_bpic13_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

In [ ]:
logs = os.path.join(OUTPUT_LOG_DATA_DIR, model_name)
%tensorboard --logdir "$logs"

### Remaining Time DA-LSTM

In [ ]:
model_name = f"dalstm_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}"

ds_bpic13_dalstm_train = ds_to_single_target(ds_sequentialize_static_attrs(ds_bpic13_train, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_REMAINING_TIME_OUTPUT)
ds_bpic13_dalstm_train = ds_cache_and_batch(ds_bpic13_dalstm_train, shuffle=False)

ds_bpic13_dalstm_test = ds_to_single_target(ds_sequentialize_static_attrs(ds_bpic13_test, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_REMAINING_TIME_OUTPUT)
ds_bpic13_dalstm_test = ds_cache_and_batch(ds_bpic13_dalstm_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseDARNN(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    categorical_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS,
    numerical_attrs=DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS | {"time_timestamp_elapsedcycle": None, "time_timestamp_elapsedprev": None, "time_timestamp_hour": None, "time_timestamp_weekday": None},
    output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}},
    name=model_name,
    rnn_dropout=0.1,
    ff_dropout=0.15,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic13_dalstm_train,
  validation_data=ds_bpic13_dalstm_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_bpic13_dalstm_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_bpic13_dalstm_test, DEFAULT_REMAINING_TIME_OUTPUT)

### Next Activity DA-LSTM

In [ ]:
model_name = f"dalstm_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}"

ds_bpic13_dalstm_train = ds_to_single_target(ds_sequentialize_static_attrs(ds_bpic13_train, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_bpic13_dalstm_train = ds_cache_and_batch(ds_bpic13_dalstm_train, shuffle=False)

ds_bpic13_dalstm_test = ds_to_single_target(ds_sequentialize_static_attrs(ds_bpic13_test, list(STATIC_CATEGORICAL_ATTRS.keys()) + list(STATIC_NUMERICAL_ATTRS.keys())), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_bpic13_dalstm_test = ds_cache_and_batch(ds_bpic13_dalstm_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseDARNN(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    categorical_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS,
    numerical_attrs=DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS | {"time_timestamp_elapsedcycle": None, "time_timestamp_elapsedprev": None, "time_timestamp_hour": None, "time_timestamp_weekday": None},
    output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'units': NUM_ACTIVITIES, 'activation': 'softmax'}},
    name=model_name,
    rnn_dropout=0.1,
    ff_dropout=0.15,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic13_dalstm_train,
  validation_data=ds_bpic13_dalstm_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_bpic13_dalstm_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_bpic13_dalstm_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## ProcessTransformer

### Remaining Time Process Transformer

In [ ]:
model_name = f"processtransformer_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}"

ds_bpic13_transformer_train = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_bpic13_train, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_REMAINING_TIME_OUTPUT)
ds_bpic13_transformer_train = ds_cache_and_batch(ds_bpic13_transformer_train, shuffle=False)

ds_bpic13_transformer_test = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_bpic13_test, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_REMAINING_TIME_OUTPUT)
ds_bpic13_transformer_test = ds_cache_and_batch(ds_bpic13_transformer_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)


model = BaseProcessTransformer(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    interval_since_last_event_vocab=INTERVAL_SINCE_LAST_EVENT_VOCAB,
    hour_of_day_vocab=HOUR_OF_DAY_VOCAB,
    day_of_week_vocab=DAY_OF_WEEK_VOCAB,
    output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'units': 1, 'activation': 'linear'}},
    name=model_name,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic13_transformer_train,
  validation_data=ds_bpic13_transformer_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_bpic13_transformer_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_bpic13_transformer_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

In [ ]:
logs = os.path.join(OUTPUT_LOG_DATA_DIR, model_name)
%tensorboard --logdir "$logs"

### Next Activity Process Transformer

In [ ]:
model_name = f"processtransformer_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}"

ds_bpic13_transformer_train = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_bpic13_train, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_bpic13_transformer_train = ds_cache_and_batch(ds_bpic13_transformer_train, shuffle=False)

ds_bpic13_transformer_test = ds_to_single_target(ds_desequentialize_dynamic_attrs(ds_bpic13_test, ["time_timestamp_elapsedprev", "time_timestamp_hour_raw", "time_timestamp_weekday_raw"]), DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_bpic13_transformer_test = ds_cache_and_batch(ds_bpic13_transformer_test, shuffle=False)

In [ ]:
keras.utils.clear_session(free_memory=True)

model = BaseProcessTransformer(
    max_case_len=MAX_SEQ_LEN,
    activity_vocab=ACTIVITY_VOCAB,
    interval_since_last_event_vocab=INTERVAL_SINCE_LAST_EVENT_VOCAB,
    hour_of_day_vocab=HOUR_OF_DAY_VOCAB,
    day_of_week_vocab=DAY_OF_WEEK_VOCAB,
    output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'units': NUM_ACTIVITIES, 'activation': 'softmax'}},
    name=model_name,
)

loss = model.default_loss
metrics = model.default_metrics
optimizer = model.default_optimizer
callbacks = model.default_callbacks(
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model_name}.csv"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model_name}_best.keras"),
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model_name}_backup"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, model_name),
  learning_rate_warmup=None,
  learning_rate_decay=None,
)
model = model.build_graph()

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  #run_eagerly=True,
)

model.summary()

In [ ]:
history = model.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic13_transformer_train,
  validation_data=ds_bpic13_transformer_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_save_files(model, ds_bpic13_transformer_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_bpic13_transformer_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

In [ ]:
logs = os.path.join(OUTPUT_LOG_DATA_DIR, model_name)
%tensorboard --logdir "$logs"

## ProcessSetTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cpst = ContrastiveProcessSetTransformer(
    name="bpic13_set_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      num_encoding_type='log_normal',
      embed_dim=64,
    ),
    embedding_model=SetTransformer(
        num_inductions=8,
        num_seeds=2,
        encoder_num_heads=4,
        encoder_dropout=0.1,
        decoder_num_heads=4,
        decoder_dropout=0.1,
    ),
    augmentation_1=SetAugmentationLayer(**MEDIUM_SET_AUGMENTATION),
    augmentation_2=SetAugmentationLayer(**STRONG_SET_AUGMENTATION),

)

cpst.build((512,2,1))

loss = cpst.default_loss
metrics = cpst.metrics

#lr_schedule = keras.optimizers.schedules.CosineDecay(
#  initial_learning_rate=0.001,
#  decay_steps=10000
#)
optimizer = cpst.optimizer
callbacks = cpst.callbacks

cpst.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cpst.summary()

In [ ]:
history = cpst.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic13_tab_train,
  validation_data=ds_bpic13_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cpst, history)

In [ ]:
cpst.evaluate(ds_bpic13_tab_train)

In [ ]:
cpst.evaluate(ds_bpic13_tab_test)

In [ ]:
model = cpst.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cpst.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_bpic13_tab_train))
test_embeddings = np.squeeze(model.predict(ds_bpic13_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_bpic13_train,
  df_bpic13_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_RESOURCE, EVENTLOG_GROUP, EVENTLOG_ROLE, 'Involved Org line 3', 'Status', 'Sub Status', 'Owner Country', 'case:Product', 'case:Country'],
  num_attrs=['case:SR Latest Impact', "time:timestamp:elapsedcycle", "time:timestamp:elapsedprev", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_bpic13_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_bpic13_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/bpic13_set_embedding_train_dataset.zip -d /content/Data/Input/bpic13_set_embedding_train_dataset
!unzip /content/Data/Input/bpic13_set_embedding_test_dataset.zip -d /content/Data/Input/bpic13_set_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64],
    rnn_units_nonshared=[32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: SetTransformer Without Mix

In [ ]:
keras.utils.clear_session(free_memory=True)

cpst = ContrastiveProcessSetTransformer(
    name="bpic13_nomix_set_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      num_encoding_type='log_normal',
      embed_dim=64,
    ),
    embedding_model=SetTransformer(
        num_inductions=8,
        num_seeds=2,
        encoder_num_heads=4,
        encoder_dropout=0.1,
        decoder_num_heads=4,
        decoder_dropout=0.1,
    ),
    augmentation_1=SetAugmentationLayer(**MEDIUM_SET_AUGMENTATION_NOMIX),
    augmentation_2=SetAugmentationLayer(**STRONG_SET_AUGMENTATION_NOMIX),

)

cpst.build((512,2,1))

loss = cpst.default_loss
metrics = cpst.metrics

#lr_schedule = keras.optimizers.schedules.CosineDecay(
#  initial_learning_rate=0.001,
#  decay_steps=10000
#)
optimizer = cpst.optimizer
callbacks = cpst.callbacks

cpst.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cpst.summary()

In [ ]:
history = cpst.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic13_tab_train,
  validation_data=ds_bpic13_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cpst, history)

In [ ]:
cpst.evaluate(ds_bpic13_tab_train)

In [ ]:
cpst.evaluate(ds_bpic13_tab_test)

In [ ]:
model = cpst.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cpst.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_bpic13_tab_train))
test_embeddings = np.squeeze(model.predict(ds_bpic13_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_bpic13_train,
  df_bpic13_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_RESOURCE, EVENTLOG_GROUP, EVENTLOG_ROLE, 'Involved Org line 3', 'Status', 'Sub Status', 'Owner Country', 'case:Product', 'case:Country'],
  num_attrs=['case:SR Latest Impact', "time:timestamp:elapsedcycle", "time:timestamp:elapsedprev", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_bpic13_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_bpic13_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/bpic13_nomix_set_embedding_train_dataset.zip -d /content/Data/Input/bpic13_nomix_set_embedding_train_dataset
!unzip /content/Data/Input/bpic13_nomix_set_embedding_test_dataset.zip -d /content/Data/Input/bpic13_nomix_set_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64],
    rnn_units_nonshared=[32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: Randomly Initialized SetTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cpst = ContrastiveProcessSetTransformer(
    name="bpic13_random_set_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      num_encoding_type='log_normal',
      embed_dim=64,
    ),
    embedding_model=SetTransformer(
        num_inductions=8,
        num_seeds=2,
        encoder_num_heads=4,
        encoder_dropout=0.1,
        decoder_num_heads=4,
        decoder_dropout=0.1,
    ),
    augmentation_1=SetAugmentationLayer(**MEDIUM_SET_AUGMENTATION),
    augmentation_2=SetAugmentationLayer(**STRONG_SET_AUGMENTATION),
)

cpst.build((256,2,1))

loss = cpst.default_loss
metrics = cpst.metrics

optimizer = cpst.optimizer
callbacks = cpst.callbacks

cpst.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)
cpst.trainable = False
cpst.summary()

In [ ]:
cpst.evaluate(ds_bpic13_tab_train)

In [ ]:
cpst.evaluate(ds_bpic13_tab_test)

In [ ]:
model = cpst.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cpst.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_bpic13_tab_train))
test_embeddings = np.squeeze(model.predict(ds_bpic13_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_bpic13_train,
  df_bpic13_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_RESOURCE, EVENTLOG_GROUP, EVENTLOG_ROLE, 'Involved Org line 3', 'Status', 'Sub Status', 'Owner Country', 'case:Product', 'case:Country'],
  num_attrs=['case:SR Latest Impact', "time:timestamp:elapsedcycle", "time:timestamp:elapsedprev", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_bpic13_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_bpic13_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cpst.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/bpic13_random_set_embedding_train_dataset.zip -d /content/Data/Input/bpic13_random_set_embedding_train_dataset
!unzip /content/Data/Input/bpic13_random_set_embedding_test_dataset.zip -d /content/Data/Input/bpic13_random_set_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64],
    rnn_units_nonshared=[32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64, 32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmset_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_set_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[256],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[256],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformerset_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## ProcessTabTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cptt = ContrastiveProcessTabTransformer(
    name="bpic13_tab_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      embed_dim=64,
      num_encoding_type='log_normal'
    ),
    embedding_model=TabTransformer(
        encoder_num_heads=4,
        encoder_dropout=0.1,
        ffn_dims=[512, 128],
    ),
    augmentation_1=TabularAugmentationLayer(**MEDIUM_TAB_AUGMENTATION),
    augmentation_2=TabularAugmentationLayer(**STRONG_TAB_AUGMENTATION),

)

cptt.build((512, len(cptt.input_encoder.cat_attrs), 32), (512, len(cptt.input_encoder.num_attrs), 32))

loss = cptt.default_loss
metrics = cptt.metrics

optimizer = cptt.optimizer
callbacks = cptt.callbacks

cptt.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cptt.summary()

In [ ]:
history = cptt.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic13_tab_train,
  validation_data=ds_bpic13_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cptt, history)

In [ ]:
cptt.evaluate(ds_bpic13_tab_train)

In [ ]:
cptt.evaluate(ds_bpic13_tab_test)

In [ ]:
model = cptt.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cptt.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_bpic13_tab_train))
test_embeddings = np.squeeze(model.predict(ds_bpic13_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_bpic13_train,
  df_bpic13_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_RESOURCE, EVENTLOG_GROUP, EVENTLOG_ROLE, 'Involved Org line 3', 'Status', 'Sub Status', 'Owner Country', 'case:Product', 'case:Country'],
  num_attrs=['case:SR Latest Impact', "time:timestamp:elapsedcycle", "time:timestamp:elapsedprev", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_bpic13_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_bpic13_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_test_dataset"))

In [ ]:
logs = os.path.join(OUTPUT_DATA_DIR, "italy_tab_embedding_tensorboard")
%tensorboard --logdir "$logs"

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/bpic13_tab_embedding_train_dataset.zip -d /content/Data/Input/bpic13_tab_embedding_train_dataset
!unzip /content/Data/Input/bpic13_tab_embedding_test_dataset.zip -d /content/Data/Input/bpic13_tab_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64],
    rnn_units_nonshared=[32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

#m2 = model

#model = m2.replace_prediction_head({DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}}, )
#model.freeze_base_model()

#optimizer = keras.optimizers.AdamW(learning_rate=0.000001, weight_decay=0.01, global_clipnorm=1.0)
optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64],
    rnn_units_nonshared=[32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.004)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.004)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
model

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: TabTransformer Without Mix

In [ ]:
keras.utils.clear_session(free_memory=True)

cptt = ContrastiveProcessTabTransformer(
    name="bpic13_nomix_tab_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      embed_dim=64,
      num_encoding_type='log_normal'
    ),
    embedding_model=TabTransformer(
        encoder_num_heads=4,
        encoder_dropout=0.1,
        ffn_dims=[512, 128],
    ),
    augmentation_1=TabularAugmentationLayer(**MEDIUM_TAB_AUGMENTATION_NOMIX),
    augmentation_2=TabularAugmentationLayer(**STRONG_TAB_AUGMENTATION_NOMIX),

)

cptt.build((512, len(cptt.input_encoder.cat_attrs), 32), (512, len(cptt.input_encoder.num_attrs), 32))

loss = cptt.default_loss
metrics = cptt.metrics

optimizer = cptt.optimizer
callbacks = cptt.callbacks

cptt.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)

cptt.summary()

In [ ]:
history = cptt.fit(
  epochs=DEFAULT_EPOCHS,
  x=ds_bpic13_tab_train,
  validation_data=ds_bpic13_tab_test,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model_visualize_history(cptt, history)

In [ ]:
cptt.evaluate(ds_bpic13_tab_train)

In [ ]:
cptt.evaluate(ds_bpic13_tab_test)

In [ ]:
model = cptt.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cptt.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_bpic13_tab_train))
test_embeddings = np.squeeze(model.predict(ds_bpic13_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_bpic13_train,
  df_bpic13_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_RESOURCE, EVENTLOG_GROUP, EVENTLOG_ROLE, 'Involved Org line 3', 'Status', 'Sub Status', 'Owner Country', 'case:Product', 'case:Country'],
  num_attrs=['case:SR Latest Impact', "time:timestamp:elapsedcycle", "time:timestamp:elapsedprev", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_bpic13_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_bpic13_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_test_dataset"))

In [ ]:
logs = os.path.join(OUTPUT_DATA_DIR, "italy_tab_embedding_tensorboard")
%tensorboard --logdir "$logs"

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/bpic13_nomix_tab_embedding_train_dataset.zip -d /content/Data/Input/bpic13_nomix_tab_embedding_train_dataset
!unzip /content/Data/Input/bpic13_nomix_tab_embedding_test_dataset.zip -d /content/Data/Input/bpic13_nomix_tab_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64],
    rnn_units_nonshared=[32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

#m2 = model

#model = m2.replace_prediction_head({DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}}, )
#model.freeze_base_model()

#optimizer = keras.optimizers.AdamW(learning_rate=0.000001, weight_decay=0.01, global_clipnorm=1.0)
optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64],
    rnn_units_nonshared=[32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_nomix_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.004)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_nomix_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_nomix_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.004)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

## Ablation: Randomly Initialized TabTransformer

In [ ]:
keras.utils.clear_session(free_memory=True)

cptt = ContrastiveProcessTabTransformer(
    name="bpic13_random_tab_embedding",
    projection_dropout=0.2,
    projection_hidden_dims=[256, 128],
    input_encoder=InputEncoder(
      num_attrs=TIME_ATTRS_VOCAB | DYNAMIC_NUMERICAL_ATTRS | STATIC_NUMERICAL_ATTRS,
      cat_attrs=DYNAMIC_CATEGORICAL_ATTRS | STATIC_CATEGORICAL_ATTRS | { 'activity': ACTIVITY_VOCAB },
      embed_dim=64,
      num_encoding_type='log_normal'
    ),
    embedding_model=TabTransformer(
        encoder_num_heads=4,
        encoder_dropout=0.1,
        ffn_dims=[256, 128],
    ),
    augmentation_1=TabularAugmentationLayer(**MEDIUM_TAB_AUGMENTATION),
    augmentation_2=TabularAugmentationLayer(**STRONG_TAB_AUGMENTATION),
)

cptt.build((256, len(cptt.input_encoder.cat_attrs), 32), (256, len(cptt.input_encoder.num_attrs), 32))

loss = cptt.default_loss
metrics = cptt.metrics

optimizer = cptt.optimizer
callbacks = cptt.callbacks

cptt.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  run_eagerly=False,
)
cptt.trainable = False
cptt.summary()

In [ ]:
cptt.evaluate(ds_bpic13_tab_train)

In [ ]:
cptt.evaluate(ds_bpic13_tab_test)

In [ ]:
model = cptt.build_embedding_model()
model.save(os.path.join(MODEL_DIR, f"{cptt.name}.keras"))
model.summary()

In [ ]:
train_embeddings = np.squeeze(model.predict(ds_bpic13_tab_train))
test_embeddings = np.squeeze(model.predict(ds_bpic13_tab_test))

np.savez_compressed(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"), train=train_embeddings, test=test_embeddings)

train_embeddings_dummy, test_embeddings_dummy = df_generate_simple_event_embeddings(
  df_bpic13_train,
  df_bpic13_test,
  cat_attrs=[EVENTLOG_ACTIVITY, EVENTLOG_RESOURCE, EVENTLOG_GROUP, EVENTLOG_ROLE, 'Involved Org line 3', 'Status', 'Sub Status', 'Owner Country', 'case:Product', 'case:Country'],
  num_attrs=['case:SR Latest Impact', "time:timestamp:elapsedcycle", "time:timestamp:elapsedprev", "time:timestamp:month", "time:timestamp:dayofyear", "time:timestamp:day", "time:timestamp:weekday", "time:timestamp:hour"],
)

print(f"{train_embeddings.shape} <- {train_embeddings_dummy.shape}")
print(f"{test_embeddings.shape} <- {test_embeddings_dummy.shape}")

In [ ]:
evaluate_embeddings(train_embeddings, original_embeddings=train_embeddings_dummy, num_samples=20000, return_df=True)

In [ ]:
evaluate_embeddings(test_embeddings, original_embeddings=test_embeddings_dummy, return_df=True)

In [ ]:
train_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['train']
test_embeddings = np.load(os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_embeddings.npz"))['test']

In [ ]:
ds_train = df_embedding_to_ds(df_bpic13_train, train_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_train, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_train_dataset"))

ds_test = df_embedding_to_ds(df_bpic13_test, test_embeddings, [EVENTLOG_LABEL_REM_TIME, EVENTLOG_LABEL_NEXT_TIME, EVENTLOG_LABEL_NEXT_ACT])
ds_write_files(ds_test, os.path.join(OUTPUT_DATA_DIR, f"{cptt.name}_test_dataset"))

### Remaining Time Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
!unzip /content/Data/Input/bpic13_random_tab_embedding_train_dataset.zip -d /content/Data/Input/bpic13_random_tab_embedding_train_dataset
!unzip /content/Data/Input/bpic13_random_tab_embedding_test_dataset.zip -d /content/Data/Input/bpic13_random_tab_embedding_test_dataset

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64],
    rnn_units_nonshared=[32],
    rnn_dropout=0.2,
    ff_dims=[64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

#m2 = model

#model = m2.replace_prediction_head({DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}}, )
#model.freeze_base_model()

#optimizer = keras.optimizers.AdamW(learning_rate=0.000001, weight_decay=0.01, global_clipnorm=1.0)
optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
    x=ds_train,
    validation_data=ds_test,
    epochs=DEFAULT_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding LSTM

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingLSTM(
    output_dict = {DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
    rnn_type='lstm',
    rnn_units_shared=[128, 64],
    rnn_units_nonshared=[32],
    rnn_dropout=0.2,
    ff_dims= [64, 32, 16],
    ff_dropout=0.2,
    name=f"lstmtab_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.008, clipvalue=3.0)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  #backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
)

model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

### Remaining Time Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_REMAINING_TIME_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_REMAINING_TIME_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_REMAINING_TIME_OUTPUT: {'activation': 'softplus', 'units': 1}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_random_{PROJECT_NAME}_{DEFAULT_REMAINING_TIME_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.004)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_regression_report(model, ds_test, DEFAULT_REMAINING_TIME_OUTPUT, archive=True)

### Next Activity Embedding Transformer

In [ ]:
BATCH_SIZE = 128

In [ ]:
ds_train = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_train_dataset'), compression='GZIP')
print(len(ds_train))

ds_train = ds_train.batch(BATCH_SIZE).shuffle(ds_train.cardinality()).prefetch(tf.data.AUTOTUNE)
ds_train = ds_to_single_target(ds_train, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_train

In [ ]:
ds_test = Dataset.load(os.path.join(INPUT_DATA_DIR, f'{PROJECT_NAME}_random_tab_embedding_test_dataset'), compression='GZIP')
print(len(ds_test))

ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_to_single_target(ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT)
ds_test

In [ ]:
keras.utils.clear_session(free_memory=True)

model = EmbeddingTransformer(
  output_dict={DEFAULT_NEXT_ACTIVITY_OUTPUT: {'activation': 'softmax', 'units': NUM_ACTIVITIES}},
  enc_heads_shared=[4],
  enc_ff_dims_shared=[512],
  enc_heads_nonshared=[4],
  enc_ff_dims_nonshared=[512],
  enc_dropout=0.2,
  pos_encoding_type='learned',
  deseq_type='maxpool',
  ff_dims=[256, 128, 64],
  ff_dropout=0.2,
  name=f"transformertab_random_{PROJECT_NAME}_{DEFAULT_NEXT_ACTIVITY_OUTPUT}",
)

optimizer = keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=0.004)
metrics = model.default_metrics
loss = model.default_loss
callbacks = model.default_callbacks(
  backup_dir=os.path.join(MODEL_BACKUP_DIR, f"{model.name}_backup"),
  checkpoint_file=os.path.join(MODEL_CHECKPOINT_DIR, f"{model.name}_best.keras"),
  log_file=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}.csv"),
  tensorboard_dir=os.path.join(OUTPUT_LOG_DATA_DIR, f"{model.name}_tensorboard"),
)

model.build((
    BATCH_SIZE,
    MAX_SEQ_LEN,
    ds_train.element_spec[0].shape[-1]
))

model.compile(
  optimizer=optimizer,
  loss=loss,
  metrics=metrics,
  jit_compile=False
)

#model.summary(expand_nested=True)

In [ ]:
history = model.fit(
  x=ds_train,
  validation_data=ds_test,
  epochs=DEFAULT_EPOCHS,
  callbacks=callbacks,
  verbose=1,
)

In [ ]:
model.evaluate(ds_test)

In [ ]:
model_save_files(model, ds_test)
model_visualize_history(model, history)

In [ ]:
model_classification_report(model, ds_test, DEFAULT_NEXT_ACTIVITY_OUTPUT, archive=True)

# Data Export

In [ ]:
output_file = f"results_{datetime.datetime.now().strftime('%Y-%m-%d_%H.%M.%S%z')}.zip"

!zip -r "$output_file" "$OUTPUT_DATA_DIR"

## A: Export to Google Drive

In [ ]:
drive.mount("/content/drive")

Path(GDRIVE_OUTPUT_DIR).mkdir(exist_ok=True)

!cp "$output_file" "$GDRIVE_OUTPUT_DIR"

drive.flush_and_unmount()

## B: Download to Local Machine

In [ ]:
files.download(output_file)